# AR-EAURP - Review 2**Adversarial-Resilient and Energy-Harvesting Aware EAURP**Three implementations and an honest comparison:| | What it is | Source ||---|---|---|| **A** | DRL-EAURP, re-implemented from the paper's equations (1)-(38) | `Lekha.pdf` || **B** | The senior's delivered code, logic verbatim | `senior_code.ipynb` || **C** | AR-EAURP - GAN trust defence + LSTM energy forecasting + CMDP | `Date_ 24_07_26.docx` |Run order: **Runtime > Run all**. Section 0 puts the project into the Colabsession, then Track 1 and Track 2 run in turn.> **How the code gets here:** this notebook carries the whole project inside it. Section 0.1 is 40 folded `%%writefile` cells that create `src/`, `experiments/` and `tests/` in the session. They render as an empty-looking gap, which is exactly why `src/` goes missing if you skip them. Nothing is downloaded.>> Prefer **`REVIEW2_FROM_GITHUB.ipynb`** if you would rather it clone the repository - that one is a tenth the size and always matches the latest commit.---### Read this before reading any number belowThe senior's code is not a network simulator; it is a metric generator.`src` and `dst` are drawn at random every round and never connected by a path,and delivery is a single coin flip against a *global average*:```pythonsuccess_prob = 0.45 + 0.25*avg_trust + 0.2*avg_energy + 0.1*avg_mobilitysuccess = random.random() < success_prob      # <- this is the entire "routing"```Because nothing forwards anything, a gray-hole that drops 15% of the packets itrelays **cannot be expressed in that model at all** - and the whole advancementis about gray-holes. So the evaluation runs on two tracks:* **Track 1 (as-is)** - each implementation in its native form. Preserves the  senior's published numbers.* **Track 2 (common harness)** - A, B and C as routing policies on one  mechanistic simulator where packets are walked hop by hop, with identical  seeds, topology, mobility, traffic and adversary placement. This is where the  fair comparison and the attack sweep live.

## 0. Setup

In [ ]:
# Dependency guard. Colab already ships everything needed, so this normally# installs nothing at all and just prints what it found.import importlib, subprocess, sysREQUIRED = ["numpy", "matplotlib", "networkx", "pandas", "torch"]missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]if missing:    print("installing:", missing)    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)else:    print("all dependencies already present - nothing to install")import numpy, matplotlib, networkxprint("numpy", numpy.__version__, "| matplotlib", matplotlib.__version__,      "| networkx", networkx.__version__)try:    import torch    print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())except Exception as exc:    print("torch unavailable ->", exc)    print("The GAN/LSTM/DQN will fall back to their documented non-learned")    print("baselines and every result will say so explicitly.")

In [ ]:
# Optional: mount Google Drive so results survive a disconnected session.# Every sweep writes its CSV the moment it finishes and is skipped on re-run,# so a dropped session costs only the sweep that was in flight.import osRESULTS_DIR = "results"try:    from google.colab import drive    drive.mount("/content/drive")    RESULTS_DIR = "/content/drive/MyDrive/cn_project_lekha/results"    os.makedirs(RESULTS_DIR, exist_ok=True)    print("checkpointing results to", RESULTS_DIR)except Exception as exc:    print("Drive not mounted (", exc, ")")    print("-> results stay in this session at ./results and are zipped at the end")os.environ["AR_EAURP_RESULTS"] = RESULTS_DIR

In [ ]:
# Create the package directories before the %%writefile cells below run.import osfor path in ["src/common", "src/routing", "src/a_drl_eaurp", "src/b_senior",             "src/c_ar_eaurp", "experiments", "tests", "results/csv",             "results/figures"]:    os.makedirs(path, exist_ok=True)print("source tree ready")

### 0.1 Source modules**These cells are what create `src/`.** They are folded away because they duplicate the files in the repository, so the section looks like an empty gap - but if you skip them, nothing below can import and `src/` will not exist in the file browser. Run the notebook from the top, or Runtime > Run all.The notebook is generated from the source tree by `tools/build_notebook.py`, so the two cannot drift.

In [ ]:
%%writefile src/__init__.py
"""AR-EAURP Review-2 project package (Colab-targeted)."""


In [ ]:
%%writefile src/common/__init__.py
"""Shared mechanistic MANET simulation harness (Track 2)."""


In [ ]:
%%writefile src/common/config.py
"""Global configuration for the AR-EAURP Review-2 simulations.

Every constant here is traceable to one of the two source papers:

  base.pdf  -- EAURP, A. Chandra & A.S.N. Chakravarthy, Sustainable Computing 2025
  Lekha.pdf -- DRL-EAURP, Lekha S., VIT Vellore

Section / equation references are given inline so the review panel can check
each value against the paper it came from.
"""

from dataclasses import dataclass

# --------------------------------------------------------------------------
# Network geometry -- base.pdf 3.2 / Lekha.pdf Sec. IV-A
# --------------------------------------------------------------------------
AREA_SIZE = 1000.0          # square deployment region, 1000 x 1000
COMM_RANGE = 150.0          # link exists iff d_ij <= R   (Lekha Eq. 4)

# --------------------------------------------------------------------------
# Energy model -- base.pdf 3.5 / Lekha.pdf Eq. (8)
# --------------------------------------------------------------------------
INITIAL_ENERGY = 100.0
ENERGY_TX = 0.5             # per 1024-byte transmission
ENERGY_RX = 0.2             # per 1024-byte reception
IDLE_DRAIN_MIN = 0.04       # senior proposed-model drain band
IDLE_DRAIN_MAX = 0.12
IDLE_DRAIN_MIN_BASE = 0.05  # senior EAURP-base drain band (Lekha Eq. 8)
IDLE_DRAIN_MAX_BASE = 0.15
ENERGY_THRESHOLD_FRAC = 0.2  # base.pdf 3.5: threshold = 0.2 * initial energy
DEAD_FRACTION_STOP = 0.8     # stop when 80% of nodes are dead (Lekha Sec. IV-A-3)

# --------------------------------------------------------------------------
# Trust model -- base.pdf 3.7 / Lekha.pdf Eq. (19)-(21)
# --------------------------------------------------------------------------
TRUST_THRESHOLD = 0.6        # PFR < 0.6 => suspicious (base.pdf 3.7)
TRUST_SMOOTHING = 0.7        # Lekha Eq. 20: T <- 0.7*T + 0.3*PFR
MIN_OBSERVATIONS = 5         # Lekha Eq. 21: classify only once R_i > 5
INITIAL_TRUST = 1.0

# Predictive trust weights -- Lekha Eq. (26)
PREDICTIVE_WEIGHTS = (0.5, 0.3, 0.2)
TRUST_HISTORY_LEN = 3

# --------------------------------------------------------------------------
# Mobility
# --------------------------------------------------------------------------
# Both papers sweep 10,000-40,000 "m/s", which is not physically meaningful for
# a MANET. We keep those values so our curves line up with the published ones,
# and additionally expose a realistic sweep that is reported as an annex.
SPEEDS_PAPER = [10000, 15000, 20000, 25000, 30000, 35000, 40000]
SPEEDS_REALISTIC = [1.0, 5.0, 10.0, 15.0, 20.0]
MAX_SPEED = 40000.0          # v_max used in the mobility factor (Lekha Eq. 22)
MOVE_SCALE = 0.001           # displacement = speed * MOVE_SCALE per round

NODE_COUNTS = [60, 80, 100, 120, 150, 200]
DEFAULT_NODES = 100
DEFAULT_SPEED = 20000

# --------------------------------------------------------------------------
# Traffic
# --------------------------------------------------------------------------
PACKET_SIZE_MIN = 512
PACKET_SIZE_MAX = 1024
PACKETS_PER_ROUND = 4
# 40 concurrent flows, not 10. Watchdog trust only accumulates on nodes that
# actually relay traffic, so path diversity is what determines detection
# coverage: raising this from 10 to 40 lifts black-hole detection recall from
# 0.23 to 0.47 with no other change.
N_FLOWS = 40                 # persistent CBR flows (Track 2 default)
LARGE_PACKET_THRESHOLD = 700  # grayhole selective-forwarding boundary (bytes)

# --------------------------------------------------------------------------
# Delay model -- Lekha.pdf Eq. (12): D = D_base + H * D_hop
# --------------------------------------------------------------------------
BASE_DELAY_MIN = 20.0
BASE_DELAY_MAX = 50.0
HOP_DELAY_MIN = 4.0
HOP_DELAY_MAX = 10.0

# --------------------------------------------------------------------------
# Link reliability (Track 2 mechanistic channel)
# --------------------------------------------------------------------------
# CALIBRATION NOTE -- state this at the review. Neither source paper contains a
# PHY or link model: both compute one global success probability per packet and
# flip a coin (Lekha Eq. 7/24/28/35). Any per-hop reliability we use is
# therefore our own modelling assumption, not something inherited from the
# papers. These values were chosen so that a clean 100-node network lands at
# PDR ~0.85-0.91 across the speed sweep, i.e. the same regime the papers report,
# leaving headroom for attacks to be visible. Every conclusion in the report
# rests on *relative* comparisons under identical channel settings.
LINK_BASE_RELIABILITY = 0.995  # per-hop success at zero range utilisation
LINK_RANGE_PENALTY = 0.02      # subtracted at the edge of COMM_RANGE
LINK_MOBILITY_PENALTY = 0.03   # subtracted at MAX_SPEED
LINK_MIN_RELIABILITY = 0.90

# MEASURED PROPERTY OF THE PAPERS' OWN TOPOLOGY -- at their stated parameters
# (N=100, AREA=1000x1000, R=150) only ~90.7% of node pairs are mutually
# reachable; the mean degree is 6.1 and the giant component holds ~95% of nodes.
# At N=60 reachability collapses to ~39%. A routing protocol cannot deliver
# across a partition, so a "no route" floor is inherent to the papers'
# configuration and is reported as its own loss category.

# --------------------------------------------------------------------------
# Control plane -- base.pdf 3.6
# --------------------------------------------------------------------------
PT_NID_PERIOD = 10            # rounds between PT_NID neighbour reports
PT_GID_PERIOD = 25            # rounds between PT_GID consensus rounds
GID_CONSENSUS_FRACTION = 0.5  # "majority of nodes detect a node as malicious"
GID_MIN_ACCUSERS = 2

# --------------------------------------------------------------------------
# Reinforcement learning -- Lekha.pdf Eq. (3), (33)-(37)
# --------------------------------------------------------------------------
RL_ALPHA = 0.1
RL_GAMMA = 0.9
RL_EPSILON = 0.1
REWARD_SUCCESS = 1.0
REWARD_FAILURE = -1.0

# --------------------------------------------------------------------------
# AR-EAURP (C) -- Date_24_07_26.docx
# --------------------------------------------------------------------------
GAN_FEATURE_DIM = 8
GAN_LATENT_DIM = 8
GAN_HIDDEN = 32
GAN_EPOCHS = 300
GAN_BATCH = 64
GAN_LR = 2e-4
GAN_LABEL_SMOOTH = 0.9
GAN_THRESHOLD_PERCENTILE = 95.0
CONTESTED_TRUST_WEIGHT = 0.5   # docx: "downgrades its trust weight by 50%"
CONTESTED_TTL = 50             # rounds a node stays contested
DETECT_PERIOD = 25             # rounds between anomaly-detection passes
FEATURE_WINDOW = 50            # observation window for the feature vector

LSTM_HIDDEN = 32
LSTM_INPUT_LEN = 20
LSTM_HORIZON = 10              # docx: "predict energy for the next 10 rounds"
LSTM_EPOCHS = 40
LSTM_LR = 1e-3

# Solar / kinetic harvesting
HARVEST_DAY_LENGTH = 400.0     # rounds per simulated day
HARVEST_PANEL_FRACTION = 0.7   # fraction of nodes carrying a panel
HARVEST_PEAK_MIN = 0.05
HARVEST_PEAK_MAX = 0.16
HARVEST_CLOUD_RHO = 0.92       # AR(1) cloud-occlusion persistence
HARVEST_CLOUD_SIGMA = 0.12
KINETIC_GAIN = 2.0e-7          # per unit speed
MAX_ENERGY_CAP = 120.0         # battery cannot exceed this

# CMDP -- docx step 4
CMDP_PDR_FLOOR = 0.90
CMDP_WINDOW = 100              # rounds in the running PDR window
CMDP_PENALTY = 5.0
CMDP_LAMBDA_LR = 0.05
CMDP_LAMBDA_MAX = 20.0

# DQN
DQN_HIDDEN = 64
DQN_BUFFER = 5000
DQN_BATCH = 64
DQN_LR = 1e-3
DQN_TARGET_SYNC = 200
DQN_EPS_START = 0.30
DQN_EPS_END = 0.05
DQN_EPS_DECAY = 800

# Attack sweep -- docx step 5
MALICIOUS_FRACTIONS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
ATTACKS = ["blackhole", "grayhole", "trust_poisoning"]
# Tuned so the *observed* PFR lands near 0.84 -- i.e. an overall drop rate of
# ~15%, exactly the "dropping just 15% of packets to stay above the 0.6 trust
# threshold" attack the advancement doc describes. With sizes ~ U[512,1024]
# about 63% of packets are "large", so 0.25 * 0.63 = 0.158 overall loss.
GRAYHOLE_LARGE_DROP_P = 0.25  # drop prob. for packets >= LARGE_PACKET_THRESHOLD
SLANDER_HIGH = 1.0            # trust a slanderer reports for its colluders
SLANDER_LOW = 0.15            # trust a slanderer reports for honest victims


@dataclass(frozen=True)
class Profile:
    """Runtime budget. QUICK is for smoke tests, FULL for the review numbers."""

    name: str
    rounds: int
    runs: int
    packets_per_round: int
    gan_epochs: int
    lstm_epochs: int


# ROUND-BUDGET NOTE -- worth explaining at the review.
#
# The papers charge energy only to an idle drain (Lekha Eq. 8) and never to
# transmission, so their networks survive ~1250 rounds. Once per-hop TX/RX cost
# is actually modelled, a 100-node network under load drains at ~0.17 J/round,
# and base.pdf's own rule -- exclude any node below 20% of initial energy --
# starts removing relays from round ~200 and has emptied the network by ~800.
#
# We therefore run the performance sweeps over a window in which the network is
# genuinely operational, and measure network lifetime in its own long-running
# experiment. Both numbers are reported; conflating them is what lets the
# papers claim a ~1230-round lifetime alongside a 0.96 delivery ratio.
LIFETIME_ROUNDS = 1500       # dedicated lifetime experiment (E4)

QUICK = Profile(
    name="quick",
    rounds=200,
    runs=2,
    packets_per_round=4,
    gan_epochs=60,
    lstm_epochs=10,
)

FULL = Profile(
    name="full",
    rounds=400,
    runs=5,
    packets_per_round=4,
    gan_epochs=300,
    lstm_epochs=40,
)

PROFILES = {"quick": QUICK, "full": FULL}


def get_profile(name):
    """Look up a runtime profile by name."""
    key = str(name).lower()
    if key not in PROFILES:
        raise ValueError(
            "unknown profile {0!r}; choose from {1}".format(name, sorted(PROFILES))
        )
    return PROFILES[key]


def energy_threshold():
    """base.pdf 3.5 -- nodes below 20% of initial energy are excluded."""
    return ENERGY_THRESHOLD_FRAC * INITIAL_ENERGY


In [ ]:
%%writefile src/common/seeding.py
"""Deterministic, stream-separated random number generation.

The whole point of this module: when we compare protocol A against B against C
we must be sure that any difference in the numbers comes from the *protocol*
and not from the protocols having consumed random numbers in a different order
and therefore having been handed a different world.

We solve that by giving each concern (topology, mobility, traffic, channel,
adversary, policy) its own independent generator, spawned from one master seed
via ``numpy.random.SeedSequence``. A protocol that draws more policy randomness
than another cannot shift the node layout, the mobility trace, the traffic
pattern or the adversary placement by even one bit.
"""

from dataclasses import dataclass

import numpy as np

# Order matters: spawned children are positional, so never reorder this list
# without regenerating every stored result.
STREAM_NAMES = (
    "topology",
    "mobility",
    "traffic",
    "channel",
    "adversary",
    "policy",
)


@dataclass
class SeedBundle:
    """One independent ``numpy`` generator per simulation concern."""

    master: int
    topology: np.random.Generator
    mobility: np.random.Generator
    traffic: np.random.Generator
    channel: np.random.Generator
    adversary: np.random.Generator
    policy: np.random.Generator

    def describe(self):
        return "SeedBundle(master={0})".format(self.master)


def make_seed_bundle(master_seed):
    """Build a :class:`SeedBundle` from a single integer master seed."""
    master_seed = int(master_seed)
    sequence = np.random.SeedSequence(master_seed)
    children = sequence.spawn(len(STREAM_NAMES))
    generators = {
        name: np.random.default_rng(child)
        for name, child in zip(STREAM_NAMES, children)
    }
    return SeedBundle(master=master_seed, **generators)


def run_seed(base_seed, run_index):
    """Derive a distinct master seed for run ``run_index`` of an experiment.

    Deliberately arithmetic rather than hashed so that the seed used for any
    reported number can be reconstructed by hand during the review.
    """
    return int(base_seed) + 1000 * int(run_index)


In [ ]:
%%writefile src/common/node.py
"""Node state for the mechanistic MANET harness.

Design note (performance): rather than one Python object per node, all node
state is held as parallel ``numpy`` arrays inside :class:`NodeState`. Mobility,
energy drain, neighbour discovery and trust decay are then single vectorised
operations instead of N-iteration Python loops. The per-hop forwarding loop is
inherently sequential and still uses plain indexing, but everything around it
is bulk. On a 2-vCPU Colab box this is the difference between a sweep that
finishes in minutes and one that does not finish at all.

The per-observer watchdog tables are N x N arrays: ``sent_to[i, j]`` is how
many packets node i handed to node j for forwarding, and ``fwd_seen[i, j]`` is
how many of those i observed j actually relay. The Packet Forwarding Ratio of
base.pdf 3.7 is then ``fwd_seen[i, j] / sent_to[i, j]``.

The large/small split of those counters is what makes selective forwarding
(gray-hole) visible at all -- a single scalar PFR cannot distinguish "drops 15%
of everything" from "drops 60% of the big packets and nothing else".
"""

import numpy as np

from . import config

# Role flags. Deliberately bit flags so behaviours compose: the docx's
# "trust poisoning" adversary is a gray-hole that also slanders.
ROLE_HONEST = 0
FLAG_BLACKHOLE = 1
FLAG_GRAYHOLE = 2
FLAG_SLANDERER = 4

ROLE_LABELS = {
    ROLE_HONEST: "honest",
    FLAG_BLACKHOLE: "blackhole",
    FLAG_GRAYHOLE: "grayhole",
    FLAG_SLANDERER: "slanderer",
    FLAG_GRAYHOLE | FLAG_SLANDERER: "grayhole+slanderer",
    FLAG_BLACKHOLE | FLAG_SLANDERER: "blackhole+slanderer",
}


def role_label(role):
    """Human-readable name for a role bitmask."""
    return ROLE_LABELS.get(int(role), "role<{0}>".format(int(role)))


class NodeState(object):
    """Vectorised state for every node in one simulation run."""

    def __init__(self, n_nodes, rng_topology, initial_energy=None):
        if n_nodes < 2:
            raise ValueError("need at least 2 nodes, got {0}".format(n_nodes))

        self.n = int(n_nodes)
        e0 = config.INITIAL_ENERGY if initial_energy is None else float(initial_energy)

        # --- position and mobility ---------------------------------------
        self.x = rng_topology.uniform(0.0, config.AREA_SIZE, self.n)
        self.y = rng_topology.uniform(0.0, config.AREA_SIZE, self.n)
        self.dest_x = rng_topology.uniform(0.0, config.AREA_SIZE, self.n)
        self.dest_y = rng_topology.uniform(0.0, config.AREA_SIZE, self.n)
        self.speed = np.full(self.n, float(config.DEFAULT_SPEED))

        # --- energy -------------------------------------------------------
        self.initial_energy = np.full(self.n, e0)
        self.energy = np.full(self.n, e0)
        self.alive = np.ones(self.n, dtype=bool)
        self.death_round = np.full(self.n, -1, dtype=np.int64)

        # Energy harvesting (only used by the AR-EAURP solar model).
        self.panel_gain = np.zeros(self.n)
        self.cloud = np.ones(self.n)
        self.harvest_last = np.zeros(self.n)

        # --- roles --------------------------------------------------------
        self.role = np.zeros(self.n, dtype=np.int64)

        # --- watchdog observation tables (observer i, subject j) ----------
        shape = (self.n, self.n)
        self.sent_to = np.zeros(shape)
        self.fwd_seen = np.zeros(shape)
        self.sent_large = np.zeros(shape)
        self.fwd_seen_large = np.zeros(shape)
        self.sent_small = np.zeros(shape)
        self.fwd_seen_small = np.zeros(shape)
        self.lat_sum = np.zeros(shape)
        self.lat_sq = np.zeros(shape)
        self.lat_n = np.zeros(shape)

        # i's trust in j (base.pdf 3.7, Lekha Eq. 20).
        self.trust = np.full(shape, config.INITIAL_TRUST)
        # Sliding trust history for the predictive model (Lekha Eq. 25-27).
        self.trust_history = np.full(
            (config.TRUST_HISTORY_LEN, self.n, self.n), config.INITIAL_TRUST
        )

        # --- control plane state (base.pdf 3.6-3.7) -----------------------
        # accusations[i, j] == 1 -> i has reported j as suspicious via PT_GID
        self.accusations = np.zeros(shape, dtype=np.int8)
        self.revoked = np.zeros(self.n, dtype=bool)
        self.revocation_round = np.full(self.n, -1, dtype=np.int64)

        # --- AR-EAURP anomaly state ---------------------------------------
        self.contested = np.zeros(self.n, dtype=bool)
        self.contested_until = np.zeros(self.n, dtype=np.int64)
        self.anomaly_score = np.zeros(self.n)

        # --- neighbour adjacency, filled by topology.rebuild --------------
        self.adjacency = np.zeros(shape, dtype=bool)
        self.neighbours = [np.zeros(0, dtype=np.int64) for _ in range(self.n)]

    # -- role helpers ------------------------------------------------------

    def is_blackhole(self, i):
        return bool(self.role[i] & FLAG_BLACKHOLE)

    def is_grayhole(self, i):
        return bool(self.role[i] & FLAG_GRAYHOLE)

    def is_slanderer(self, i):
        return bool(self.role[i] & FLAG_SLANDERER)

    def malicious_mask(self):
        """Boolean mask of nodes with any adversarial flag set."""
        return self.role != ROLE_HONEST

    # -- energy helpers ----------------------------------------------------

    def energy_threshold(self):
        """base.pdf 3.5 -- per-node 20%-of-initial forwarding threshold."""
        return config.ENERGY_THRESHOLD_FRAC * self.initial_energy

    def eligible_mask(self):
        """Nodes that may carry traffic: alive, not revoked, above threshold."""
        return self.alive & (~self.revoked) & (self.energy > self.energy_threshold())

    def normalised_energy(self):
        """E_i / E_init, clipped to [0, 1] (Lekha Eq. 23/31)."""
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where(
                self.initial_energy > 0.0, self.energy / self.initial_energy, 0.0
            )
        return np.clip(ratio, 0.0, 1.0)

    def mobility_factor(self):
        """M_i = 1 / (1 + v_i / v_max)  (Lekha Eq. 22/32)."""
        return 1.0 / (1.0 + self.speed / config.MAX_SPEED)

    # -- trust helpers -----------------------------------------------------

    def observed_pfr(self):
        """PFR[i, j] = fwd_seen[i, j] / sent_to[i, j], 1.0 where unobserved.

        Unobserved pairs default to 1.0 (fully trusted) to match base.pdf,
        which only ever penalises a node once it has actually been watched.
        """
        with np.errstate(divide="ignore", invalid="ignore"):
            pfr = np.where(self.sent_to > 0.0, self.fwd_seen / self.sent_to, 1.0)
        return np.clip(pfr, 0.0, 1.0)

    def network_trust(self):
        """Per-node trust as seen by the network: mean over active observers."""
        observed = self.sent_to > 0.0
        counts = observed.sum(axis=0)
        totals = np.where(observed, self.trust, 0.0).sum(axis=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            mean_trust = np.where(counts > 0, totals / np.maximum(counts, 1), config.INITIAL_TRUST)
        return np.clip(mean_trust, 0.0, 1.0)

    def summary(self):
        """Compact dict used for logging and sanity checks."""
        return {
            "n": self.n,
            "alive": int(self.alive.sum()),
            "revoked": int(self.revoked.sum()),
            "contested": int(self.contested.sum()),
            "malicious": int(self.malicious_mask().sum()),
            "mean_energy": float(self.energy[self.alive].mean()) if self.alive.any() else 0.0,
        }


In [ ]:
%%writefile src/common/mobility.py
"""Destination-driven mobility -- Lekha.pdf Eq. (5)-(6).

Each node walks toward a randomly assigned destination; on arrival a fresh
destination is drawn. Displacement per round is ``speed * MOVE_SCALE``, which
reproduces the senior's ``node.x += (dx/dist) * node.speed * 0.001``.
"""

import numpy as np

from . import config

ARRIVAL_RADIUS = 1.0


def assign_speeds(nodes, speed, rng_mobility=None, jitter=0.0):
    """Set every node's speed, optionally with multiplicative jitter."""
    base = np.full(nodes.n, float(speed))
    if jitter > 0.0 and rng_mobility is not None:
        base = base * rng_mobility.uniform(1.0 - jitter, 1.0 + jitter, nodes.n)
    nodes.speed = np.clip(base, 0.0, None)
    return nodes.speed


def step(nodes, rng_mobility):
    """Advance every node one round toward its destination (Eq. 5-6)."""
    dx = nodes.dest_x - nodes.x
    dy = nodes.dest_y - nodes.y
    dist = np.sqrt(dx * dx + dy * dy)

    # Nodes that have arrived pick a new destination and stay put this round,
    # matching the senior's `continue` branch.
    arrived = dist < ARRIVAL_RADIUS
    n_arrived = int(arrived.sum())
    if n_arrived:
        nodes.dest_x[arrived] = rng_mobility.uniform(0.0, config.AREA_SIZE, n_arrived)
        nodes.dest_y[arrived] = rng_mobility.uniform(0.0, config.AREA_SIZE, n_arrived)

    moving = ~arrived
    if moving.any():
        safe = np.where(dist > 0.0, dist, 1.0)
        stride = nodes.speed * config.MOVE_SCALE
        nodes.x[moving] += (dx[moving] / safe[moving]) * stride[moving]
        nodes.y[moving] += (dy[moving] / safe[moving]) * stride[moving]

    # Keep the swarm inside the deployment square. The senior's code lets nodes
    # drift out of the 1000x1000 area at high speed; clamping keeps the node
    # density (and therefore the connectivity) consistent across speeds, which
    # is what makes the speed sweep interpretable.
    np.clip(nodes.x, 0.0, config.AREA_SIZE, out=nodes.x)
    np.clip(nodes.y, 0.0, config.AREA_SIZE, out=nodes.y)


In [ ]:
%%writefile src/common/topology.py
"""Neighbour discovery -- Lekha.pdf Eq. (4).

A link exists between i and j iff the Euclidean distance between them is at
most ``COMM_RANGE``. Distances are computed as one vectorised N x N operation;
for N <= 200 this is a few hundred microseconds, versus tens of milliseconds
for the nested Python loop the senior's notebook uses.
"""

import numpy as np

from . import config


def pairwise_distances(nodes):
    """Full N x N Euclidean distance matrix."""
    points = np.stack((nodes.x, nodes.y), axis=1)
    diff = points[:, None, :] - points[None, :, :]
    return np.sqrt((diff * diff).sum(axis=-1))


def rebuild(nodes, comm_range=None):
    """Recompute adjacency and per-node neighbour lists.

    Dead and revoked nodes are removed from the topology: a flat battery cannot
    relay, and a revoked node is excluded from all network activity
    (base.pdf 3.7).
    """
    radius = config.COMM_RANGE if comm_range is None else float(comm_range)
    distances = pairwise_distances(nodes)

    adjacency = distances <= radius
    np.fill_diagonal(adjacency, False)

    usable = nodes.alive & (~nodes.revoked)
    adjacency &= usable[:, None]
    adjacency &= usable[None, :]

    nodes.adjacency = adjacency
    nodes.neighbours = [np.flatnonzero(row) for row in adjacency]
    return distances


def link_exists(nodes, i, j):
    """True when i and j are currently one hop apart."""
    return bool(nodes.adjacency[i, j])


def path_is_valid(nodes, path):
    """True when every hop of ``path`` still exists and every node is usable."""
    if not path or len(path) < 2:
        return False
    for node_id in path:
        if not nodes.alive[node_id] or nodes.revoked[node_id]:
            return False
    for a, b in zip(path[:-1], path[1:]):
        if not nodes.adjacency[a, b]:
            return False
    return True


def connected_pairs(nodes, rng, count):
    """Draw ``count`` (src, dst) pairs that are distinct and currently usable.

    Falls back to any two distinct eligible nodes; reachability is left to the
    routing layer so that "no route found" remains a real, measurable outcome.
    """
    eligible = np.flatnonzero(nodes.eligible_mask())
    if eligible.size < 2:
        return []
    pairs = []
    for _ in range(int(count)):
        src, dst = rng.choice(eligible, size=2, replace=False)
        pairs.append((int(src), int(dst)))
    return pairs


In [ ]:
%%writefile src/common/channel.py
"""Per-hop link model.

Neither source paper models a link: both compute one global success
probability per *packet* and flip a coin. Here reliability is a property of the
individual hop, degrading with the fraction of the communication range used
and with node speed. That is the minimum needed for a routing decision to have
consequences -- a longer path really is riskier than a short one.

Delay follows Lekha.pdf Eq. (12): ``D = D_base + H * D_hop``, accumulated hop
by hop rather than sampled once from a hop count drawn at random.
"""

import numpy as np

from . import config


def link_reliability(nodes, i, j, distance=None):
    """Probability that a single transmission from i to j is received."""
    if distance is None:
        dx = nodes.x[i] - nodes.x[j]
        dy = nodes.y[i] - nodes.y[j]
        distance = float(np.sqrt(dx * dx + dy * dy))

    range_use = min(1.0, distance / config.COMM_RANGE)
    speed_use = min(1.0, max(nodes.speed[i], nodes.speed[j]) / config.MAX_SPEED)

    reliability = (
        config.LINK_BASE_RELIABILITY
        - config.LINK_RANGE_PENALTY * range_use
        - config.LINK_MOBILITY_PENALTY * speed_use
    )
    return float(np.clip(reliability, config.LINK_MIN_RELIABILITY, 1.0))


def transmit(nodes, i, j, rng_channel, distance=None):
    """Attempt one hop. Returns True when the frame is received by j."""
    return bool(rng_channel.random() < link_reliability(nodes, i, j, distance))


def hop_delay(rng_channel):
    """Per-hop delay contribution in milliseconds (Lekha Eq. 12)."""
    return float(rng_channel.uniform(config.HOP_DELAY_MIN, config.HOP_DELAY_MAX))


def base_delay(rng_channel):
    """Fixed per-packet processing delay in milliseconds (Lekha Eq. 12)."""
    return float(rng_channel.uniform(config.BASE_DELAY_MIN, config.BASE_DELAY_MAX))


def spend_hop_energy(nodes, sender, receiver, size_bytes, energy_model):
    """Charge a transmission to the sender and a reception to the receiver."""
    tx_cost = energy_model.transmit_cost(size_bytes)
    rx_cost = energy_model.receive_cost(size_bytes)
    nodes.energy[sender] -= tx_cost
    nodes.energy[receiver] -= rx_cost
    return tx_cost + rx_cost


In [ ]:
%%writefile src/common/energy.py
"""Energy models.

Two models are provided:

``LinearDepletion``
    Lekha.pdf Eq. (8): ``E(t+1) = E(t) - dE``, ``dE ~ U[0.05, 0.15]``. This is
    what both papers use and what the senior's notebook implements. Note that
    it is monotonically decreasing and completely independent of what the node
    actually does, which is one of the limitations the advancement targets.

``SolarHarvesting``
    The AR-EAURP replacement (docx step 3): ``E(t+1) = E(t) - dE + H(t)`` with
    a diurnal solar term modulated by an AR(1) cloud process, plus a small
    kinetic term proportional to node speed. Only a fraction of nodes carry a
    panel, so the DRL agent has something non-trivial to learn: route through
    the nodes that are *predicted* to be in surplus.
"""

import numpy as np

from . import config


def _kill_newly_dead(nodes, round_idx):
    """Mark nodes whose battery just ran out."""
    newly_dead = nodes.alive & (nodes.energy <= 0.0)
    if newly_dead.any():
        nodes.energy[newly_dead] = 0.0
        nodes.alive[newly_dead] = False
        nodes.death_round[newly_dead] = int(round_idx)
    return int(newly_dead.sum())


class EnergyModel(object):
    """Interface shared by every energy model."""

    name = "base"

    def reset(self, nodes, rng_channel):
        """Hook for per-run initialisation."""
        return None

    def step(self, nodes, round_idx, rng_channel):
        raise NotImplementedError

    def transmit_cost(self, size_bytes):
        return config.ENERGY_TX * (float(size_bytes) / 1024.0)

    def receive_cost(self, size_bytes):
        return config.ENERGY_RX * (float(size_bytes) / 1024.0)


class LinearDepletion(EnergyModel):
    """Lekha.pdf Eq. (8) -- constant random drain, no harvesting."""

    name = "linear"

    def __init__(self, drain_min=None, drain_max=None):
        self.drain_min = config.IDLE_DRAIN_MIN if drain_min is None else float(drain_min)
        self.drain_max = config.IDLE_DRAIN_MAX if drain_max is None else float(drain_max)
        if self.drain_max < self.drain_min:
            raise ValueError("drain_max must be >= drain_min")

    def step(self, nodes, round_idx, rng_channel):
        drain = rng_channel.uniform(self.drain_min, self.drain_max, nodes.n)
        nodes.energy[nodes.alive] -= drain[nodes.alive]
        nodes.harvest_last[:] = 0.0
        return _kill_newly_dead(nodes, round_idx)


class SolarHarvesting(EnergyModel):
    """AR-EAURP energy model with diurnal solar plus kinetic harvesting."""

    name = "solar"

    def __init__(self, drain_min=None, drain_max=None, day_length=None):
        self.drain_min = config.IDLE_DRAIN_MIN if drain_min is None else float(drain_min)
        self.drain_max = config.IDLE_DRAIN_MAX if drain_max is None else float(drain_max)
        self.day_length = (
            config.HARVEST_DAY_LENGTH if day_length is None else float(day_length)
        )

    def reset(self, nodes, rng_channel):
        """Assign heterogeneous panels; some nodes deliberately have none."""
        has_panel = rng_channel.random(nodes.n) < config.HARVEST_PANEL_FRACTION
        gains = rng_channel.uniform(
            config.HARVEST_PEAK_MIN, config.HARVEST_PEAK_MAX, nodes.n
        )
        nodes.panel_gain = np.where(has_panel, gains, 0.0)
        nodes.cloud = np.ones(nodes.n)
        nodes.harvest_last = np.zeros(nodes.n)
        return nodes.panel_gain

    def irradiance(self, round_idx):
        """Diurnal term in [0, 1]; zero for the half of the day that is night."""
        phase = 2.0 * np.pi * (float(round_idx) / self.day_length)
        return max(0.0, float(np.sin(phase)))

    def harvest(self, nodes, round_idx, rng_channel):
        """Energy harvested by each node this round."""
        # AR(1) cloud occlusion, shared shape across nodes but independent noise.
        noise = rng_channel.normal(0.0, config.HARVEST_CLOUD_SIGMA, nodes.n)
        nodes.cloud = config.HARVEST_CLOUD_RHO * nodes.cloud + (
            1.0 - config.HARVEST_CLOUD_RHO
        ) * 1.0 + noise * (1.0 - config.HARVEST_CLOUD_RHO)
        np.clip(nodes.cloud, 0.0, 1.5, out=nodes.cloud)

        solar = nodes.panel_gain * self.irradiance(round_idx) * nodes.cloud
        kinetic = config.KINETIC_GAIN * nodes.speed
        return np.clip(solar + kinetic, 0.0, None)

    def step(self, nodes, round_idx, rng_channel):
        drain = rng_channel.uniform(self.drain_min, self.drain_max, nodes.n)
        gain = self.harvest(nodes, round_idx, rng_channel)
        nodes.harvest_last = gain

        delta = gain - drain
        nodes.energy[nodes.alive] += delta[nodes.alive]
        np.clip(nodes.energy, None, config.MAX_ENERGY_CAP, out=nodes.energy)
        return _kill_newly_dead(nodes, round_idx)


def make_energy_model(name, **kwargs):
    """Factory used by the experiment scripts and the notebook."""
    key = str(name).lower()
    if key in ("linear", "lekha", "eq8"):
        return LinearDepletion(**kwargs)
    if key in ("solar", "harvest", "harvesting"):
        return SolarHarvesting(**kwargs)
    raise ValueError("unknown energy model {0!r}".format(name))


In [ ]:
%%writefile src/common/traffic.py
"""Traffic generation.

Two modes:

``flows``   (default for Track 2)
    A fixed set of persistent CBR source/destination pairs. Realistic, and it
    lets the routing layer cache routes instead of rediscovering a path for
    every packet -- which is what keeps the sweep inside the Colab time budget.

``random``  (paper-faithful)
    A fresh (src, dst) pair drawn uniformly every packet, reproducing the
    senior's ``src = random.randint(...)`` behaviour.
"""

from dataclasses import dataclass

import numpy as np

from . import config


@dataclass
class Packet:
    """One data packet in flight."""

    pid: int
    src: int
    dst: int
    size: int
    created_round: int

    @property
    def is_large(self):
        """Gray-hole selective forwarding keys off this boundary."""
        return self.size >= config.LARGE_PACKET_THRESHOLD


class TrafficGenerator(object):
    """Produces the packets offered to the routing layer each round."""

    def __init__(self, nodes, rng_traffic, mode="flows", n_flows=None,
                 packets_per_round=None):
        self.nodes = nodes
        self.rng = rng_traffic
        self.mode = str(mode).lower()
        if self.mode not in ("flows", "random"):
            raise ValueError("traffic mode must be 'flows' or 'random'")

        self.n_flows = config.N_FLOWS if n_flows is None else int(n_flows)
        self.packets_per_round = (
            config.PACKETS_PER_ROUND if packets_per_round is None
            else int(packets_per_round)
        )
        self._next_pid = 0
        self.flows = self._make_flows() if self.mode == "flows" else []

    def _reachable_pool(self):
        """Node ids in the largest connected component of the initial topology.

        Real CBR flows are set up between hosts that can currently reach each
        other, so we draw endpoints from the giant component rather than
        uniformly. Without this, at the papers' own parameters roughly one flow
        in ten would sit permanently across a network partition and contribute
        nothing but "no route" losses for the whole run -- which adds variance
        without telling us anything about the protocol. Genuine route breaks
        still happen constantly once the nodes start moving.
        """
        adjacency = self.nodes.adjacency
        if adjacency is None or not adjacency.any():
            return np.arange(self.nodes.n)

        unseen = set(range(self.nodes.n))
        best = []
        while unseen:
            root = unseen.pop()
            component = [root]
            frontier = [root]
            while frontier:
                current = frontier.pop()
                for neighbour in np.flatnonzero(adjacency[current]):
                    neighbour = int(neighbour)
                    if neighbour in unseen:
                        unseen.discard(neighbour)
                        component.append(neighbour)
                        frontier.append(neighbour)
            if len(component) > len(best):
                best = component
        return np.asarray(sorted(best)) if len(best) >= 2 else np.arange(self.nodes.n)

    def _make_flows(self):
        """Pick persistent source/destination pairs, avoiding self-loops."""
        flows = []
        pool = self._reachable_pool()
        if pool.size < 2:
            return flows

        wanted = min(self.n_flows, max(1, int(pool.size) // 2))
        attempts = 0
        while len(flows) < wanted and attempts < 50 * wanted:
            attempts += 1
            src = int(self.rng.choice(pool))
            dst = int(self.rng.choice(pool))
            if src != dst and (src, dst) not in flows:
                flows.append((src, dst))
        return flows

    def _new_pid(self):
        self._next_pid += 1
        return self._next_pid

    def _size(self):
        return int(
            self.rng.integers(config.PACKET_SIZE_MIN, config.PACKET_SIZE_MAX + 1)
        )

    def generate(self, round_idx):
        """Return the packets offered this round.

        Flows whose endpoints are dead or revoked are skipped rather than
        rerouted, so losing an endpoint shows up as reduced offered load rather
        than as a spurious delivery failure.
        """
        packets = []
        usable = self.nodes.eligible_mask()

        if self.mode == "flows":
            if not self.flows:
                return packets
            picks = self.rng.integers(0, len(self.flows), self.packets_per_round)
            for index in picks:
                src, dst = self.flows[int(index)]
                if not (usable[src] and usable[dst]):
                    continue
                packets.append(
                    Packet(self._new_pid(), src, dst, self._size(), int(round_idx))
                )
            return packets

        eligible = np.flatnonzero(usable)
        if eligible.size < 2:
            return packets
        for _ in range(self.packets_per_round):
            src, dst = self.rng.choice(eligible, size=2, replace=False)
            packets.append(
                Packet(self._new_pid(), int(src), int(dst), self._size(), int(round_idx))
            )
        return packets


In [ ]:
%%writefile src/common/adversary.py
"""Threat model -- AR-EAURP advancement, docx step 1.

Three adversary behaviours, composable through the role bit flags in
``node.py``:

``blackhole``
    Answers every route request claiming an excellent route, then drops 100% of
    the data it is asked to relay. It keeps relaying *control* traffic so that
    it stays inside the topology and keeps attracting routes.

``grayhole`` (selective forwarding)
    Relays control traffic and small packets, and drops large data packets with
    probability ``GRAYHOLE_LARGE_DROP_P``. This is the attack the advancement
    doc is built around: with a 512-1024 byte size distribution and a 700-byte
    boundary, the observed Packet Forwarding Ratio lands near 0.8 -- comfortably
    above the 0.6 revocation threshold of base.pdf 3.7. A protocol that watches
    only a scalar PFR cannot see it at all.

``trust_poisoning``
    A gray-hole that also slanders: in its PT_GID reports it vouches for its
    fellow attackers (trust 1.0) and accuses the most trusted honest nodes
    (trust 0.15), attacking the consensus mechanism rather than the data plane.

Placement is drawn from the dedicated adversary stream, so the same nodes are
malicious no matter which protocol is under test.
"""

import numpy as np

from . import config
from .node import FLAG_BLACKHOLE, FLAG_GRAYHOLE, FLAG_SLANDERER, ROLE_HONEST

ATTACK_ROLES = {
    "none": ROLE_HONEST,
    "blackhole": FLAG_BLACKHOLE,
    "grayhole": FLAG_GRAYHOLE,
    "trust_poisoning": FLAG_GRAYHOLE | FLAG_SLANDERER,
}


def assign_roles(nodes, attack, malicious_fraction, rng_adversary):
    """Mark a fraction of the nodes as adversarial and return their indices."""
    nodes.role[:] = ROLE_HONEST
    key = str(attack).lower()
    if key not in ATTACK_ROLES:
        raise ValueError(
            "unknown attack {0!r}; choose from {1}".format(attack, sorted(ATTACK_ROLES))
        )

    role = ATTACK_ROLES[key]
    fraction = float(malicious_fraction)
    if role == ROLE_HONEST or fraction <= 0.0:
        return np.zeros(0, dtype=np.int64)

    count = int(round(fraction * nodes.n))
    count = max(0, min(count, nodes.n - 2))  # always leave an honest pair
    if count == 0:
        return np.zeros(0, dtype=np.int64)

    chosen = rng_adversary.choice(nodes.n, size=count, replace=False)
    nodes.role[chosen] = role
    return np.sort(chosen)


def forwards_data(nodes, node_id, packet, rng_adversary):
    """Does ``node_id`` actually relay this data packet?

    Returns True for honest nodes. This is the single place where adversarial
    packet-level behaviour is decided.
    """
    role = int(nodes.role[node_id])
    if role == ROLE_HONEST:
        return True

    if role & FLAG_BLACKHOLE:
        return False

    if role & FLAG_GRAYHOLE:
        if not packet.is_large:
            return True  # small packets pass, keeping the observed PFR high
        return bool(rng_adversary.random() >= config.GRAYHOLE_LARGE_DROP_P)

    # A pure slanderer attacks the control plane only; its data plane is honest.
    return True


def forwards_control(nodes, node_id):
    """Adversaries relay control traffic to stay reachable and attract routes."""
    return True


def advertises_false_route(nodes, node_id):
    """Black-holes claim a perfect route to everything (base.pdf threat model)."""
    return bool(int(nodes.role[node_id]) & FLAG_BLACKHOLE)


def reported_trust(nodes, observer, subject, honest_value):
    """Trust value ``observer`` publishes about ``subject`` in its PT_GID report.

    Honest nodes report what they measured. Slanderers invert the signal: they
    vouch for colluders and accuse whichever honest nodes are most trusted,
    which is precisely what makes the majority vote of base.pdf 3.7 exploitable.
    """
    if not (int(nodes.role[observer]) & FLAG_SLANDERER):
        return float(honest_value)

    if int(nodes.role[subject]) != ROLE_HONEST:
        return config.SLANDER_HIGH
    return config.SLANDER_LOW


def slander_targets(nodes, observer, k=None):
    """The honest neighbours a slanderer will accuse: the most trusted ones."""
    neighbours = nodes.neighbours[observer]
    if neighbours.size == 0:
        return np.zeros(0, dtype=np.int64)

    honest = neighbours[nodes.role[neighbours] == ROLE_HONEST]
    if honest.size == 0:
        return honest

    order = np.argsort(-nodes.trust[observer, honest])
    if k is None:
        k = max(1, honest.size // 2)
    return honest[order[: int(k)]]


def describe(attack, malicious_fraction):
    """Label used in CSV output and plot legends."""
    if str(attack).lower() == "none" or malicious_fraction <= 0.0:
        return "clean"
    return "{0}@{1:.0f}%".format(attack, 100.0 * float(malicious_fraction))


In [ ]:
%%writefile src/common/metrics.py
"""Metric accumulation for one simulation run.

Beyond the five metrics both papers report (delay, packet loss, throughput,
PDR, network lifetime) we also track:

* **why** each packet was lost -- no route, link failure, a dead relay, or an
  adversary dropping it. Without this breakdown "PDR went down" says nothing
  about whether the protocol or the attacker caused it.
* **detection quality** -- true/false positive rates against the known ground
  truth of which nodes are actually malicious. This is how the advancement (C)
  earns its keep, and neither paper measures it.
* **control overhead** -- PT_NID / PT_GID / PT_CREV and route-discovery traffic,
  so the cost of the trust machinery is visible rather than assumed free.
"""

import numpy as np

# Reasons a packet failed to arrive.
LOSS_NO_ROUTE = "no_route"
LOSS_LINK = "link_failure"
LOSS_DEAD_NODE = "dead_relay"
LOSS_MALICIOUS = "malicious_drop"
LOSS_REASONS = (LOSS_NO_ROUTE, LOSS_LINK, LOSS_DEAD_NODE, LOSS_MALICIOUS)


class RunMetrics(object):
    """Accumulates every statistic for a single run."""

    def __init__(self, n_nodes):
        self.n_nodes = int(n_nodes)

        self.packets_sent = 0
        self.packets_received = 0
        self.bytes_delivered = 0
        self.bytes_lost = 0
        self.total_delay = 0.0
        self.total_hops = 0

        self.losses = dict((reason, 0) for reason in LOSS_REASONS)

        self.control_packets = 0
        self.route_discoveries = 0
        self.route_failures = 0

        self.rounds_completed = 0
        self.first_death_round = None
        self.eighty_percent_dead_round = None

        self.energy_spent = 0.0
        self.energy_harvested = 0.0

        # CMDP bookkeeping (advancement, docx step 4).
        self.cmdp_checks = 0
        self.cmdp_violations = 0

        # Rolling window used by the CMDP constraint and by the DRL state.
        self._window = []
        self._window_size = 100

        self.detection = {
            "tp": 0, "fp": 0, "fn": 0, "tn": 0,
            "tpr": 0.0, "fpr": 0.0, "precision": 0.0, "f1": 0.0,
        }
        self.extra = {}

    # -- packet accounting -------------------------------------------------

    def record_sent(self, packet):
        self.packets_sent += 1

    def record_delivered(self, packet, delay_ms, hops):
        self.packets_received += 1
        self.bytes_delivered += int(packet.size)
        self.total_delay += float(delay_ms)
        self.total_hops += int(hops)
        self._push_window(1.0)

    def record_lost(self, packet, reason):
        if reason not in self.losses:
            self.losses[reason] = 0
        self.losses[reason] += 1
        self.bytes_lost += int(packet.size)
        self._push_window(0.0)

    def _push_window(self, outcome):
        self._window.append(float(outcome))
        if len(self._window) > self._window_size:
            self._window.pop(0)

    def window_pdr(self, default=1.0):
        """Delivery ratio over the recent window, used by the CMDP constraint."""
        if not self._window:
            return float(default)
        return float(sum(self._window) / len(self._window))

    def record_cmdp_check(self, satisfied):
        self.cmdp_checks += 1
        if not satisfied:
            self.cmdp_violations += 1

    # -- network accounting ------------------------------------------------

    def record_control(self, count=1):
        self.control_packets += int(count)

    def record_discovery(self, found):
        self.route_discoveries += 1
        if not found:
            self.route_failures += 1

    def record_round(self, round_idx, nodes):
        self.rounds_completed = int(round_idx) + 1
        dead = int((~nodes.alive).sum())
        if dead > 0 and self.first_death_round is None:
            self.first_death_round = int(round_idx)
        if (
            self.eighty_percent_dead_round is None
            and dead >= 0.8 * self.n_nodes
        ):
            self.eighty_percent_dead_round = int(round_idx)

    # -- detection ---------------------------------------------------------

    def score_detection(self, nodes, detected_mask=None):
        """Compare the protocol's verdicts with the ground-truth roles."""
        truth = nodes.malicious_mask()
        if detected_mask is None:
            detected_mask = nodes.revoked | nodes.contested
        detected = np.asarray(detected_mask, dtype=bool)

        tp = int((detected & truth).sum())
        fp = int((detected & ~truth).sum())
        fn = int((~detected & truth).sum())
        tn = int((~detected & ~truth).sum())

        tpr = tp / float(tp + fn) if (tp + fn) else 0.0
        fpr = fp / float(fp + tn) if (fp + tn) else 0.0
        precision = tp / float(tp + fp) if (tp + fp) else 0.0
        f1 = (
            2.0 * precision * tpr / (precision + tpr)
            if (precision + tpr) > 0.0
            else 0.0
        )

        self.detection = {
            "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "tpr": tpr, "fpr": fpr, "precision": precision, "f1": f1,
        }
        return self.detection

    # -- results -----------------------------------------------------------

    def finalise(self, nodes=None):
        """Collapse the accumulators into a flat dict ready for a CSV row."""
        rounds = max(1, self.rounds_completed)
        received = self.packets_received

        avg_delay = self.total_delay / received if received else 0.0
        pdr = received / float(self.packets_sent) if self.packets_sent else 0.0
        avg_hops = self.total_hops / float(received) if received else 0.0

        # Throughput uses the senior's convention -- delivered bits divided by
        # rounds, scaled by 1000 -- so our numbers sit on the same axis as the
        # published ones. It is bits per round / 1000, not true kbps; the
        # report says so explicitly.
        throughput = (self.bytes_delivered * 8.0) / (rounds * 1000.0)

        lifetime = (
            self.first_death_round if self.first_death_round is not None else rounds
        )

        # PDR restricted to packets for which a route existed at all. This
        # separates protocol quality from raw topology: at the papers' own
        # parameters ~9% of node pairs are not even connected, and no routing
        # protocol can deliver across a partition.
        routable = self.packets_sent - self.losses.get(LOSS_NO_ROUTE, 0)
        pdr_routable = received / float(routable) if routable > 0 else 0.0

        row = {
            "packets_sent": self.packets_sent,
            "packets_received": received,
            "packets_lost": self.packets_sent - received,
            "pdr": pdr,
            "pdr_routable": pdr_routable,
            "avg_delay_ms": avg_delay,
            "avg_hops": avg_hops,
            "throughput": throughput,
            "bytes_delivered": self.bytes_delivered,
            "packet_loss_bytes": self.bytes_lost,
            "network_lifetime": lifetime,
            "rounds_completed": rounds,
            "eighty_pct_dead_round": (
                self.eighty_percent_dead_round
                if self.eighty_percent_dead_round is not None
                else rounds
            ),
            "control_packets": self.control_packets,
            "route_discoveries": self.route_discoveries,
            "route_failures": self.route_failures,
            "energy_spent": self.energy_spent,
            "energy_harvested": self.energy_harvested,
            "cmdp_checks": self.cmdp_checks,
            "cmdp_violations": self.cmdp_violations,
            "cmdp_violation_rate": (
                self.cmdp_violations / float(self.cmdp_checks)
                if self.cmdp_checks else 0.0
            ),
        }

        for reason in LOSS_REASONS:
            row["loss_" + reason] = self.losses.get(reason, 0)

        for key, value in self.detection.items():
            row["det_" + key] = value

        if nodes is not None:
            alive = nodes.alive
            row["nodes_alive"] = int(alive.sum())
            row["nodes_revoked"] = int(nodes.revoked.sum())
            row["nodes_contested"] = int(nodes.contested.sum())
            row["mean_residual_energy"] = (
                float(nodes.energy[alive].mean()) if alive.any() else 0.0
            )
            row["total_energy_consumed"] = float(
                (nodes.initial_energy - nodes.energy).sum()
            )

        row.update(self.extra)
        return row


def aggregate(rows):
    """Average a list of per-run result dicts, adding ``*_std`` columns."""
    if not rows:
        return {}

    keys = sorted(rows[0].keys())
    out = {}
    for key in keys:
        values = [row.get(key) for row in rows]
        numeric = [v for v in values if isinstance(v, (int, float, np.integer, np.floating))]
        if len(numeric) == len(values) and numeric:
            array = np.asarray(numeric, dtype=float)
            out[key] = float(array.mean())
            out[key + "_std"] = float(array.std())
        else:
            out[key] = values[0]
    out["n_runs"] = len(rows)
    return out


In [ ]:
%%writefile src/routing/__init__.py
"""Routing substrate for the common harness (Track 2).

Implements the machinery specified in base.pdf -- the custom control packets,
the packet-forwarding-ratio trust model with majority-vote revocation, the
energy-gated AODV route discovery scored by ``R_Score = a*T + b*E``, and ECC
payload protection.

This is *supporting infrastructure*, not one of the three graded
implementations. It exists because Track 2 needs a routing layer that actually
forwards packets hop by hop, and because the slander attack in the advancement
is defined in terms of PT_GID consensus.
"""


In [ ]:
%%writefile src/routing/packets.py
"""Custom control packet types -- base.pdf 3.6.

The three hex codes are taken verbatim from the paper:

    PT_NID  0x81   Node Identification   -- node id, forwarding ratios, timestamp
    PT_GID  0x82   Group Identification  -- node ids with cumulative trust scores
    PT_CREV 0x83   Controlled Revocation -- id of a confirmed malicious node

Route discovery uses the standard AODV control set, kept separate so that the
protocol overhead of the trust machinery can be reported on its own.
"""

from dataclasses import dataclass, field

PT_NID = 0x81
PT_GID = 0x82
PT_CREV = 0x83

PT_RREQ = 0x01
PT_RREP = 0x02
PT_RERR = 0x03

PACKET_NAMES = {
    PT_NID: "PT_NID",
    PT_GID: "PT_GID",
    PT_CREV: "PT_CREV",
    PT_RREQ: "RREQ",
    PT_RREP: "RREP",
    PT_RERR: "RERR",
}

CONTROL_TYPES = (PT_NID, PT_GID, PT_CREV, PT_RREQ, PT_RREP, PT_RERR)


def packet_name(ptype):
    return PACKET_NAMES.get(int(ptype), "PT_{0:#04x}".format(int(ptype)))


@dataclass
class NidPacket:
    """base.pdf 3.6 -- periodic per-neighbour forwarding-ratio report."""

    ptype: int = PT_NID
    sender: int = -1
    timestamp: int = 0
    ratios: dict = field(default_factory=dict)   # neighbour id -> observed PFR


@dataclass
class GidPacket:
    """base.pdf 3.6 -- cumulative trust scores acting as a trust beacon."""

    ptype: int = PT_GID
    sender: int = -1
    timestamp: int = 0
    scores: dict = field(default_factory=dict)   # node id -> reported trust
    accusations: tuple = ()                      # node ids reported suspicious


@dataclass
class CrevPacket:
    """base.pdf 3.6 -- broadcast that a node has been revoked."""

    ptype: int = PT_CREV
    sender: int = -1
    timestamp: int = 0
    revoked: int = -1
    evidence: float = 0.0


@dataclass
class RouteRequest:
    """AODV RREQ carrying the accumulated route score (base.pdf 3.3)."""

    ptype: int = PT_RREQ
    rreq_id: int = 0
    source: int = -1
    destination: int = -1
    hops: tuple = ()
    score: float = 0.0


@dataclass
class RouteReply:
    """AODV RREP returned along the reverse path (base.pdf 3.4)."""

    ptype: int = PT_RREP
    source: int = -1
    destination: int = -1
    path: tuple = ()
    cumulative_score: float = 0.0


In [ ]:
%%writefile src/routing/ecc.py
"""ECC payload protection -- base.pdf 3.9.

The paper's construction is

    Encryption:  pick random r, C = M XOR (r * G)
    Decryption:  M = C XOR (r * P)     with P the receiver's public key

which is ElGamal-style elliptic-curve key agreement followed by an XOR with a
curve-point-derived keystream. We implement exactly that shape using real
elliptic-curve Diffie-Hellman (X25519) for the ``r * G`` / ``r * P`` agreement
and HKDF-SHA256 to stretch the agreed point into a keystream of the payload's
length.

Two engineering points worth stating at the review:

* The ECDH agreement is done **once per node pair and cached**, and only the
  keystream XOR runs per packet. That is how real protocols work, and it keeps
  the cost honest: the expensive asymmetric operation is amortised over a
  session rather than charged to every packet.
* base.pdf claims ECC adds security "without introducing any overhead". It is
  not free, so :func:`benchmark` measures it and the report quotes the number.

If ``cryptography`` is unavailable the module degrades to a clearly-labelled
hash-based stand-in so a simulation never fails for want of a crypto library.
"""

import hashlib
import os
import time

try:  # pragma: no cover - exercised by whichever environment runs this
    from cryptography.hazmat.primitives import hashes, serialization
    from cryptography.hazmat.primitives.asymmetric.x25519 import (
        X25519PrivateKey,
    )
    from cryptography.hazmat.primitives.kdf.hkdf import HKDF

    CRYPTO_AVAILABLE = True
except Exception:  # pragma: no cover
    CRYPTO_AVAILABLE = False

KEYSTREAM_INFO = b"EAURP-ECC-keystream"


def _hkdf(shared_secret, length, salt):
    if CRYPTO_AVAILABLE:
        return HKDF(
            algorithm=hashes.SHA256(),
            length=length,
            salt=salt,
            info=KEYSTREAM_INFO,
        ).derive(shared_secret)
    # Fallback: counter-mode SHA256. Labelled as such in `backend`.
    out = bytearray()
    counter = 0
    while len(out) < length:
        digest = hashlib.sha256(
            shared_secret + salt + KEYSTREAM_INFO + counter.to_bytes(4, "big")
        ).digest()
        out.extend(digest)
        counter += 1
    return bytes(out[:length])


class ECCKeyring(object):
    """Per-node ECC key pairs with cached pairwise shared secrets."""

    def __init__(self, n_nodes, enabled=True):
        self.n = int(n_nodes)
        self.enabled = bool(enabled)
        self.backend = "x25519" if CRYPTO_AVAILABLE else "sha256-fallback"

        self._private = {}
        self._public = {}
        self._shared = {}

        self.keygen_seconds = 0.0
        self.agreement_seconds = 0.0
        self.encrypt_seconds = 0.0
        self.encrypt_calls = 0
        self.agreements = 0

    # -- key management ----------------------------------------------------

    def _private_key(self, node_id):
        node_id = int(node_id)
        if node_id not in self._private:
            start = time.perf_counter()
            if CRYPTO_AVAILABLE:
                key = X25519PrivateKey.generate()
                self._private[node_id] = key
                self._public[node_id] = key.public_key()
            else:
                secret = os.urandom(32)
                self._private[node_id] = secret
                self._public[node_id] = hashlib.sha256(secret).digest()
            self.keygen_seconds += time.perf_counter() - start
        return self._private[node_id]

    def public_key_bytes(self, node_id):
        self._private_key(node_id)
        public = self._public[int(node_id)]
        if CRYPTO_AVAILABLE:
            return public.public_bytes(
                encoding=serialization.Encoding.Raw,
                format=serialization.PublicFormat.Raw,
            )
        return public

    def shared_secret(self, sender, receiver):
        """ECDH agreement between two nodes, computed once and cached."""
        key = (int(sender), int(receiver)) if sender <= receiver else (int(receiver), int(sender))
        if key in self._shared:
            return self._shared[key]

        start = time.perf_counter()
        private = self._private_key(key[0])
        self._private_key(key[1])
        if CRYPTO_AVAILABLE:
            secret = private.exchange(self._public[key[1]])
        else:
            secret = hashlib.sha256(private + self._public[key[1]]).digest()
        self.agreement_seconds += time.perf_counter() - start
        self.agreements += 1

        self._shared[key] = secret
        return secret

    # -- payload protection ------------------------------------------------

    def encrypt(self, sender, receiver, payload):
        """C = M XOR keystream(r * P). Returns the ciphertext."""
        if not self.enabled:
            return payload

        start = time.perf_counter()
        secret = self.shared_secret(sender, receiver)
        salt = os.urandom(8)
        keystream = _hkdf(secret, len(payload), salt)
        cipher = bytes(a ^ b for a, b in zip(payload, keystream))
        self.encrypt_seconds += time.perf_counter() - start
        self.encrypt_calls += 1
        return salt + cipher

    def decrypt(self, sender, receiver, blob):
        """M = C XOR keystream(r * G). Inverse of :meth:`encrypt`."""
        if not self.enabled:
            return blob
        salt, cipher = blob[:8], blob[8:]
        secret = self.shared_secret(sender, receiver)
        keystream = _hkdf(secret, len(cipher), salt)
        return bytes(a ^ b for a, b in zip(cipher, keystream))

    def cost_summary(self):
        """Numbers quoted in the overhead experiment (E6)."""
        per_packet = (
            self.encrypt_seconds / self.encrypt_calls if self.encrypt_calls else 0.0
        )
        per_agreement = (
            self.agreement_seconds / self.agreements if self.agreements else 0.0
        )
        return {
            "ecc_backend": self.backend,
            "ecc_keygen_seconds": self.keygen_seconds,
            "ecc_agreement_seconds": self.agreement_seconds,
            "ecc_agreements": self.agreements,
            "ecc_seconds_per_agreement": per_agreement,
            "ecc_encrypt_seconds": self.encrypt_seconds,
            "ecc_encrypt_calls": self.encrypt_calls,
            "ecc_seconds_per_packet": per_packet,
        }


def benchmark(n_pairs=32, payload_size=1024, repeats=200):
    """Standalone measurement of the real cost of the ECC layer."""
    keyring = ECCKeyring(max(2, n_pairs * 2), enabled=True)
    payload = os.urandom(int(payload_size))

    for index in range(int(n_pairs)):
        keyring.shared_secret(2 * index, 2 * index + 1)

    start = time.perf_counter()
    for index in range(int(repeats)):
        pair = index % max(1, int(n_pairs))
        keyring.encrypt(2 * pair, 2 * pair + 1, payload)
    elapsed = time.perf_counter() - start

    summary = keyring.cost_summary()
    summary["bench_payload_bytes"] = int(payload_size)
    summary["bench_repeats"] = int(repeats)
    summary["bench_total_seconds"] = elapsed
    summary["bench_seconds_per_packet"] = elapsed / max(1, int(repeats))
    return summary


def verify_roundtrip():
    """Sanity check used by the unit tests and by the notebook."""
    keyring = ECCKeyring(4, enabled=True)
    message = b"EAURP payload under test" * 8
    blob = keyring.encrypt(0, 1, message)
    return keyring.decrypt(0, 1, blob) == message


In [ ]:
%%writefile src/routing/aodv.py
"""Energy- and trust-gated route discovery -- base.pdf 3.3 to 3.5, 3.8.

The paper specifies discovery as an RREQ flood in which each node forwards only
when it is not revoked and holds more than 20% of its initial energy, with each
route accumulating ``R_Score = a*T + b*E``; the destination then picks the route
with the highest cumulative score and no revoked hop.

Two things are worth flagging to the review panel, because both are properties
of the *paper*, not of this implementation:

1. ``a = 1 - avg_energy/max_energy`` and ``b = avg_energy/max_energy`` invert
   the intuitive priority. When the network is energy-rich, ``b -> 1`` and the
   score is dominated by energy; trust only starts to matter once batteries are
   already low -- exactly the opposite of what a security-motivated protocol
   wants. We implement the coefficients as published and report the effect.

2. Selecting the *highest cumulative* score rewards longer routes, since every
   extra hop adds another non-negative term. Taken literally it prefers the
   longest path in the network. We therefore minimise accumulated
   ``(1 - R_Score_j) + hop_penalty``, which preserves the paper's intent -- prefer
   trusted, energy-rich relays -- without the pathology. The literal variant is
   available via ``cumulative_score`` for comparison.

Discovery is modelled by its outcome (a shortest-path search over the current
adjacency) rather than by simulating every individual RREQ frame. The gating
rules are identical, and it is orders of magnitude faster, which is what keeps
the sweeps inside the Colab time budget.
"""

import heapq

import numpy as np

from ..common import config

HOP_PENALTY = 0.35
BLACKHOLE_ADVERTISED_SCORE = 1.0

MODE_EAURP = "eaurp"
MODE_AODV = "aodv"
MODE_TRUST = "max_trust"
MODE_ENERGY = "max_energy"
MODE_DIVERSE = "diversified"
ROUTE_MODES = (MODE_EAURP, MODE_AODV, MODE_TRUST, MODE_ENERGY, MODE_DIVERSE)


def score_coefficients(nodes):
    """The (a, b) pair of base.pdf 3.3."""
    alive = nodes.alive
    if not alive.any():
        return 0.5, 0.5
    max_energy = float(nodes.initial_energy.max())
    if max_energy <= 0.0:
        return 0.5, 0.5
    b = float(np.clip(nodes.energy[alive].mean() / max_energy, 0.0, 1.0))
    return 1.0 - b, b


def route_scores(nodes, trust=None, energy=None, honour_blackhole=True):
    """Per-node ``R_Score`` in [0, 1] used to weight route discovery.

    ``trust`` and ``energy`` may be overridden so that a policy can substitute
    its own view -- the advancement, for instance, feeds in GAN-adjusted trust
    and LSTM-predicted energy surplus.
    """
    a, b = score_coefficients(nodes)
    trust_vec = nodes.network_trust() if trust is None else np.asarray(trust, dtype=float)
    energy_vec = (
        nodes.normalised_energy() if energy is None else np.asarray(energy, dtype=float)
    )
    scores = a * np.clip(trust_vec, 0.0, 1.0) + b * np.clip(energy_vec, 0.0, 1.0)

    if honour_blackhole:
        # A black-hole answers every RREQ claiming a perfect route. From the
        # network's point of view that is simply a maximal advertised score.
        from ..common.node import FLAG_BLACKHOLE

        liars = (nodes.role & FLAG_BLACKHOLE) != 0
        if liars.any():
            scores = np.where(liars, BLACKHOLE_ADVERTISED_SCORE, scores)

    return np.clip(scores, 0.0, 1.0)


def _edge_cost(scores, node_id, mode, hop_penalty):
    if mode == MODE_AODV:
        return 1.0  # plain hop-count AODV: no trust, no energy awareness
    return (1.0 - float(scores[node_id])) + hop_penalty


def find_route(nodes, src, dst, scores, mode=MODE_EAURP, hop_penalty=HOP_PENALTY,
               avoid=None, max_hops=None):
    """Dijkstra over the current topology honouring the base.pdf gating rules.

    Returns the path as a list ``[src, ..., dst]`` or ``None`` when no eligible
    route exists.
    """
    src = int(src)
    dst = int(dst)
    if src == dst:
        return None

    eligible = nodes.eligible_mask()
    if not (eligible[src] and eligible[dst]):
        return None

    avoid_set = set() if avoid is None else set(int(a) for a in avoid)
    cap = nodes.n if max_hops is None else int(max_hops)

    dist = {src: 0.0}
    previous = {}
    visited = set()
    queue = [(0.0, src)]

    while queue:
        cost, current = heapq.heappop(queue)
        if current in visited:
            continue
        visited.add(current)
        if current == dst:
            break

        for neighbour in nodes.neighbours[current]:
            neighbour = int(neighbour)
            if neighbour in visited:
                continue
            # base.pdf 3.3/3.5: only relay through nodes that are alive, not
            # revoked and above the 20%-of-initial-energy threshold. The
            # destination itself is always allowed.
            if neighbour != dst:
                if not eligible[neighbour] or neighbour in avoid_set:
                    continue
            step = _edge_cost(scores, neighbour, mode, hop_penalty)
            candidate = cost + step
            if candidate < dist.get(neighbour, float("inf")):
                dist[neighbour] = candidate
                previous[neighbour] = current
                heapq.heappush(queue, (candidate, neighbour))

    if dst not in previous and dst != src:
        return None

    path = [dst]
    while path[-1] != src:
        step = previous.get(path[-1])
        if step is None:
            return None
        path.append(step)
        if len(path) > cap:
            return None
    path.reverse()
    return path


def find_diverse_route(nodes, src, dst, scores, primary=None, **kwargs):
    """A second, node-disjoint-ish route used by the 'explore' action.

    Falls back to the primary route when no alternative exists, which is the
    honest outcome in a sparse topology rather than reporting a failure.
    """
    if primary is None:
        primary = find_route(nodes, src, dst, scores, **kwargs)
    if primary is None or len(primary) <= 2:
        return primary

    avoid = set(primary[1:-1])
    alternative = find_route(nodes, src, dst, scores, avoid=avoid, **kwargs)
    return alternative if alternative is not None else primary


def cumulative_score(scores, path):
    """The paper's literal cumulative R_Score for a discovered route."""
    if not path:
        return 0.0
    return float(sum(float(scores[node_id]) for node_id in path[1:]))


def mean_score(scores, path):
    """Per-hop mean score -- the length-normalised alternative."""
    if not path or len(path) < 2:
        return 0.0
    return cumulative_score(scores, path) / float(len(path) - 1)


class RouteCache(object):
    """Caches discovered routes and invalidates them when they break.

    base.pdf 3.8 -- a node checks the revocation list before forwarding, emits
    a route error when the path is broken, and rediscovers. Caching is also
    what makes the sweep affordable: rediscovering on every packet would mean a
    Dijkstra per packet instead of one per route break.
    """

    def __init__(self):
        self._routes = {}
        self.hits = 0
        self.misses = 0
        self.invalidations = 0

    def get(self, nodes, src, dst):
        key = (int(src), int(dst))
        path = self._routes.get(key)
        if path is None:
            self.misses += 1
            return None
        if not self._still_valid(nodes, path):
            del self._routes[key]
            self.invalidations += 1
            self.misses += 1
            return None
        self.hits += 1
        return path

    def put(self, src, dst, path):
        if path:
            self._routes[(int(src), int(dst))] = path

    def drop_node(self, node_id):
        """Remove every cached route traversing a newly revoked/dead node."""
        node_id = int(node_id)
        stale = [key for key, path in self._routes.items() if node_id in path]
        for key in stale:
            del self._routes[key]
        self.invalidations += len(stale)
        return len(stale)

    def clear(self):
        self._routes.clear()

    @staticmethod
    def _still_valid(nodes, path):
        for node_id in path:
            if not nodes.alive[node_id] or nodes.revoked[node_id]:
                return False
        for a, b in zip(path[:-1], path[1:]):
            if not nodes.adjacency[a, b]:
                return False
        return True


In [ ]:
%%writefile src/routing/trust.py
"""Trust evaluation, consensus and revocation -- base.pdf 3.6 to 3.8.

The chain specified by the paper is:

1. **Watchdog.** Node i hands a packet to neighbour j and listens for j to
   relay it. That gives the Packet Forwarding Ratio ``PFR = PF / PR``.
2. **PT_NID.** Neighbours periodically exchange those ratios.
3. **Trust update.** ``T <- 0.7*T + 0.3*PFR`` (Lekha Eq. 20; base.pdf uses the
   raw ratio, the moving average is the senior's refinement).
4. **PT_GID consensus.** "If many nodes complain about the misbehaviour of a
   single node, that node will be considered malicious."
5. **Energy excuse.** Before blaming a node, check its battery -- a node too
   flat to relay is not malicious. base.pdf 3.7 is explicit about this.
6. **PT_CREV.** Broadcast the revocation; every node drops routes through it.

Step 4 is precisely what the slander adversary attacks: the vote is only as
honest as the nodes casting it.
"""

import numpy as np

from ..common import adversary, config
from . import packets


def reset_observations(nodes):
    """Clear the watchdog tables (used between runs)."""
    for name in (
        "sent_to", "fwd_seen", "sent_large", "fwd_seen_large",
        "sent_small", "fwd_seen_small", "lat_sum", "lat_sq", "lat_n",
    ):
        getattr(nodes, name).fill(0.0)
    nodes.trust.fill(config.INITIAL_TRUST)
    nodes.trust_history.fill(config.INITIAL_TRUST)
    nodes.accusations.fill(0)


def observe_forward(nodes, observer, subject, packet, forwarded, latency=0.0):
    """Record that ``observer`` watched ``subject`` either relay or drop a packet."""
    observer = int(observer)
    subject = int(subject)

    nodes.sent_to[observer, subject] += 1.0
    if packet.is_large:
        nodes.sent_large[observer, subject] += 1.0
    else:
        nodes.sent_small[observer, subject] += 1.0

    if forwarded:
        nodes.fwd_seen[observer, subject] += 1.0
        if packet.is_large:
            nodes.fwd_seen_large[observer, subject] += 1.0
        else:
            nodes.fwd_seen_small[observer, subject] += 1.0
        nodes.lat_sum[observer, subject] += float(latency)
        nodes.lat_sq[observer, subject] += float(latency) * float(latency)
        nodes.lat_n[observer, subject] += 1.0


def refresh_trust(nodes, smoothing=None):
    """Moving-average trust update over every observed pair (Lekha Eq. 20)."""
    weight = config.TRUST_SMOOTHING if smoothing is None else float(smoothing)
    pfr = nodes.observed_pfr()
    observed = nodes.sent_to > 0.0
    updated = weight * nodes.trust + (1.0 - weight) * pfr
    nodes.trust = np.where(observed, updated, nodes.trust)
    np.clip(nodes.trust, 0.0, 1.0, out=nodes.trust)
    return nodes.trust


def push_trust_history(nodes):
    """Slide the trust history window (Lekha Eq. 27)."""
    nodes.trust_history[:-1] = nodes.trust_history[1:]
    nodes.trust_history[-1] = nodes.trust


def predicted_trust(nodes, weights=None):
    """Weighted trust forecast over the history window (Lekha Eq. 26)."""
    w = config.PREDICTIVE_WEIGHTS if weights is None else weights
    history = nodes.trust_history
    depth = min(len(w), history.shape[0])
    prediction = np.zeros_like(nodes.trust)
    # w[0] is the weight on the most recent value.
    for offset in range(depth):
        prediction += w[offset] * history[history.shape[0] - 1 - offset]
    return np.clip(prediction, 0.0, 1.0)


def pt_nid_round(nodes, run_metrics=None):
    """Periodic PT_NID exchange. Counts one control packet per active node."""
    active = int(nodes.eligible_mask().sum())
    if run_metrics is not None:
        run_metrics.record_control(active)
    return active


def _observer_counts(nodes):
    """How many nodes have actually watched each subject."""
    return (nodes.sent_to >= config.MIN_OBSERVATIONS).sum(axis=0)


def build_accusations(nodes, enable_slander=True):
    """Each observer decides whether to report each watched neighbour.

    Honest observers report what they measured. Slanderers publish inverted
    values, which is how the consensus of base.pdf 3.7 gets poisoned.
    """
    nodes.accusations.fill(0)
    watched = nodes.sent_to >= config.MIN_OBSERVATIONS
    observers = np.flatnonzero(nodes.alive & (~nodes.revoked))

    for observer in observers:
        subjects = np.flatnonzero(watched[observer])
        for subject in subjects:
            if subject == observer:
                continue
            measured = float(nodes.trust[observer, subject])
            if enable_slander:
                reported = adversary.reported_trust(
                    nodes, int(observer), int(subject), measured
                )
            else:
                reported = measured
            if reported < config.TRUST_THRESHOLD:
                nodes.accusations[observer, subject] = 1
    return nodes.accusations


def pt_gid_round(nodes, run_metrics=None, enable_slander=True,
                 enable_energy_excuse=True, route_cache=None, round_idx=0):
    """PT_GID consensus followed by PT_CREV revocation (base.pdf 3.6-3.7).

    Returns the list of node ids revoked during this round.
    """
    build_accusations(nodes, enable_slander=enable_slander)

    accusers = nodes.accusations.sum(axis=0)
    observers = _observer_counts(nodes)
    threshold = np.maximum(
        config.GID_MIN_ACCUSERS,
        np.ceil(config.GID_CONSENSUS_FRACTION * observers),
    )

    condemned = (accusers >= threshold) & (observers >= config.GID_MIN_ACCUSERS)
    condemned &= ~nodes.revoked
    condemned &= nodes.alive

    if enable_energy_excuse:
        # base.pdf 3.7 -- "before labeling it malicious, the energy level will
        # be checked. This helps prevent punishing an exemplary node just
        # because of its lower power."
        flat_battery = nodes.energy <= nodes.energy_threshold()
        condemned &= ~flat_battery

    revoked_now = np.flatnonzero(condemned)
    for node_id in revoked_now:
        nodes.revoked[node_id] = True
        nodes.revocation_round[node_id] = int(round_idx)
        if route_cache is not None:
            route_cache.drop_node(int(node_id))

    if run_metrics is not None:
        active = int(nodes.eligible_mask().sum())
        # One PT_GID beacon per active node, plus one PT_CREV broadcast per
        # revocation reaching every active node.
        run_metrics.record_control(active + active * len(revoked_now))

    return [int(node_id) for node_id in revoked_now]


def crev_packets(nodes, revoked_ids, round_idx):
    """Materialise the PT_CREV broadcasts, for logging and inspection."""
    return [
        packets.CrevPacket(
            sender=-1,
            timestamp=int(round_idx),
            revoked=int(node_id),
            evidence=float(nodes.accusations[:, node_id].sum()),
        )
        for node_id in revoked_ids
    ]


In [ ]:
%%writefile src/common/simulator.py
"""The mechanistic simulation loop shared by A, B and C (Track 2).

This is the piece the source material does not have. In the senior's notebook a
"transmission" is one coin flip against a global average, so there is no path,
no relay, and nothing for a routing policy to decide. Here every packet is
walked hop by hop: each relay can be dead, out of range, out of energy, or
adversarial, and the predecessor watches whether it actually forwarded. That is
what makes trust measurable, gray-hole attacks expressible, and a routing
decision consequential.

A protocol plugs in by subclassing :class:`RoutingPolicy`. The loop guarantees
that every policy sees byte-identical topology, mobility, traffic, channel and
adversary streams for a given seed, so any difference in the results is
attributable to the policy alone.

One round is:

    move -> rebuild topology -> harvest/drain energy -> offer packets
    -> (policy picks a route) -> walk the route hop by hop -> watchdog
    -> policy feedback -> periodic PT_NID / PT_GID / PT_CREV control plane
"""

from dataclasses import dataclass, field

import numpy as np

from . import adversary, channel, config, energy as energy_mod, metrics as metrics_mod
from . import mobility, node as node_mod, seeding, topology, traffic
from ..routing import aodv, ecc, trust as trust_mod


@dataclass
class SimParams:
    """Every knob for one simulation run."""

    n_nodes: int = config.DEFAULT_NODES
    speed: float = config.DEFAULT_SPEED
    rounds: int = 400
    seed: int = 12345
    attack: str = "none"
    malicious_fraction: float = 0.0
    energy_model: str = "linear"
    traffic_mode: str = "flows"
    packets_per_round: int = config.PACKETS_PER_ROUND
    use_ecc: bool = True
    enable_control_plane: bool = True
    enable_slander: bool = True
    enable_energy_excuse: bool = True
    stop_when_dead: bool = True
    extra: dict = field(default_factory=dict)


class Simulation(object):
    """Owns the world; a :class:`RoutingPolicy` decides how packets move."""

    def __init__(self, policy, params):
        self.policy = policy
        self.params = params

        self.seeds = seeding.make_seed_bundle(params.seed)
        self.nodes = node_mod.NodeState(params.n_nodes, self.seeds.topology)
        mobility.assign_speeds(self.nodes, params.speed)

        self.energy_model = energy_mod.make_energy_model(params.energy_model)
        self.energy_model.reset(self.nodes, self.seeds.channel)

        self.malicious_ids = adversary.assign_roles(
            self.nodes, params.attack, params.malicious_fraction, self.seeds.adversary
        )

        self.keyring = ecc.ECCKeyring(params.n_nodes, enabled=params.use_ecc)
        self.route_cache = aodv.RouteCache()
        self.metrics = metrics_mod.RunMetrics(params.n_nodes)
        self.metrics._window_size = config.CMDP_WINDOW

        self.traffic = traffic.TrafficGenerator(
            self.nodes,
            self.seeds.traffic,
            mode=params.traffic_mode,
            packets_per_round=params.packets_per_round,
        )

        topology.rebuild(self.nodes)
        self.scores = aodv.route_scores(self.nodes)
        self.round_idx = 0

    # -- helpers exposed to policies ---------------------------------------

    def refresh_scores(self, trust=None, energy=None):
        """Recompute per-node route scores, optionally from a policy's view."""
        self.scores = aodv.route_scores(self.nodes, trust=trust, energy=energy)
        return self.scores

    def discover(self, src, dst, mode=aodv.MODE_EAURP, use_cache=True):
        """Route discovery with caching and RERR-style invalidation."""
        if use_cache:
            cached = self.route_cache.get(self.nodes, src, dst)
            if cached is not None:
                return cached

        path = aodv.find_route(self.nodes, src, dst, self.scores, mode=mode)
        self.metrics.record_discovery(path is not None)
        if path is not None and use_cache:
            self.route_cache.put(src, dst, path)
        return path

    # -- the per-packet forwarding walk ------------------------------------

    def forward(self, packet, path):
        """Walk ``path`` hop by hop.

        Returns ``(delivered, delay_ms, hops_completed, reason)``. ``reason`` is
        ``None`` on success, otherwise one of the ``metrics.LOSS_*`` constants.
        """
        nodes = self.nodes
        rng_channel = self.seeds.channel
        rng_adv = self.seeds.adversary

        if self.params.use_ecc:
            # Payload confidentiality is end-to-end (base.pdf 3.9), so it is
            # applied once at the source, not at every hop.
            self.keyring.encrypt(packet.src, packet.dst, b"\x00" * min(packet.size, 256))

        delay = channel.base_delay(rng_channel)
        hops_done = 0

        for index in range(len(path) - 1):
            sender = int(path[index])
            receiver = int(path[index + 1])

            if not nodes.alive[receiver] or nodes.revoked[receiver]:
                return False, delay, hops_done, metrics_mod.LOSS_DEAD_NODE
            if not nodes.adjacency[sender, receiver]:
                return False, delay, hops_done, metrics_mod.LOSS_LINK

            spent = channel.spend_hop_energy(
                nodes, sender, receiver, packet.size, self.energy_model
            )
            self.metrics.energy_spent += spent

            if not channel.transmit(nodes, sender, receiver, rng_channel):
                return False, delay, hops_done, metrics_mod.LOSS_LINK

            hop_ms = channel.hop_delay(rng_channel)
            delay += hop_ms
            hops_done += 1

            if receiver == int(packet.dst):
                return True, delay, hops_done, None

            # The relay now decides whether to forward, and the node that just
            # handed it the packet watches (base.pdf 3.6 promiscuous monitoring).
            relays = adversary.forwards_data(nodes, receiver, packet, rng_adv)
            trust_mod.observe_forward(nodes, sender, receiver, packet, relays, hop_ms)
            if not relays:
                return False, delay, hops_done, metrics_mod.LOSS_MALICIOUS

        return False, delay, hops_done, metrics_mod.LOSS_LINK

    # -- the round loop ----------------------------------------------------

    def run(self):
        params = self.params
        nodes = self.nodes

        self.policy.reset(self)

        for round_idx in range(int(params.rounds)):
            self.round_idx = round_idx

            mobility.step(nodes, self.seeds.mobility)
            topology.rebuild(nodes)

            self.energy_model.step(nodes, round_idx, self.seeds.channel)
            # harvest_last is rewritten every step, so this is the per-round total.
            self.metrics.energy_harvested += float(nodes.harvest_last.sum())

            self.policy.on_round_start(self, round_idx)

            for packet in self.traffic.generate(round_idx):
                self.metrics.record_sent(packet)
                path = self.policy.select_route(self, packet)

                if not path or len(path) < 2:
                    self.metrics.record_lost(packet, metrics_mod.LOSS_NO_ROUTE)
                    self.policy.on_result(self, packet, False, path, 0,
                                          metrics_mod.LOSS_NO_ROUTE)
                    continue

                delivered, delay, hops, reason = self.forward(packet, path)
                if delivered:
                    self.metrics.record_delivered(packet, delay, hops)
                else:
                    self.metrics.record_lost(packet, reason)
                    # base.pdf 3.8 -- a broken path triggers a route error and
                    # rediscovery, so drop the stale cache entry.
                    self.route_cache.drop_node(int(path[min(hops + 1, len(path) - 1)]))
                self.policy.on_result(self, packet, delivered, path, hops, reason)

            if params.enable_control_plane:
                self._control_plane(round_idx)

            self.policy.on_round_end(self, round_idx)
            self.metrics.record_round(round_idx, nodes)

            if params.stop_when_dead:
                dead = int((~nodes.alive).sum())
                if dead >= config.DEAD_FRACTION_STOP * nodes.n:
                    break

        detected = self.policy.detected_mask(self)
        self.metrics.score_detection(nodes, detected)

        row = self.metrics.finalise(nodes)
        row.update(self.keyring.cost_summary())
        row.update(self.policy.extra_metrics(self))
        row.update(
            {
                "policy": self.policy.name,
                "n_nodes": params.n_nodes,
                "speed": params.speed,
                "attack": params.attack,
                "malicious_fraction": params.malicious_fraction,
                "energy_model": params.energy_model,
                "seed": params.seed,
                "malicious_nodes": int(self.malicious_ids.size),
                "route_cache_hits": self.route_cache.hits,
                "route_cache_misses": self.route_cache.misses,
            }
        )
        return row

    def _control_plane(self, round_idx):
        """PT_NID / PT_GID / PT_CREV, on the periods given in base.pdf 3.6."""
        if round_idx % config.PT_NID_PERIOD == 0:
            trust_mod.pt_nid_round(self.nodes, self.metrics)
            trust_mod.refresh_trust(self.nodes)
            trust_mod.push_trust_history(self.nodes)

        if round_idx % config.PT_GID_PERIOD == 0 and round_idx > 0:
            trust_mod.pt_gid_round(
                self.nodes,
                run_metrics=self.metrics,
                enable_slander=self.params.enable_slander,
                enable_energy_excuse=self.params.enable_energy_excuse,
                route_cache=self.route_cache,
                round_idx=round_idx,
            )


class RoutingPolicy(object):
    """Interface every protocol implements to plug into :class:`Simulation`."""

    name = "policy"

    def reset(self, sim):
        """Called once before the first round."""
        return None

    def on_round_start(self, sim, round_idx):
        """Called after the world has advanced, before traffic is offered."""
        return None

    def select_route(self, sim, packet):
        """Return the path this policy chooses, or ``None`` for no route."""
        return sim.discover(packet.src, packet.dst)

    def on_result(self, sim, packet, delivered, path, hops, reason):
        """Feedback hook -- where reinforcement learning updates happen."""
        return None

    def on_round_end(self, sim, round_idx):
        return None

    def detected_mask(self, sim):
        """Nodes this policy believes are malicious; ``None`` uses the default."""
        return None

    def extra_metrics(self, sim):
        """Policy-specific columns appended to the result row."""
        return {}


def run_simulation(policy, params):
    """Convenience wrapper: build a :class:`Simulation` and run it."""
    return Simulation(policy, params).run()


In [ ]:
%%writefile src/routing/eaurp.py
"""EAURP and plain-AODV routing policies -- base.pdf.

``AodvPolicy``
    The reference baseline both papers compare against: shortest hop count, no
    trust, no energy gate, no encryption. This replaces the senior's "Existing"
    curve, which is not a protocol at all but EAURP's own output multiplied by
    fixed constants (Lekha.pdf Eq. 14-18).

``EaurpPolicy``
    base.pdf as specified: routes scored by ``R_Score = a*T + b*E``, relays gated
    on the 20%-of-initial-energy threshold and the revocation list, trust from
    watchdog-observed packet forwarding ratios, majority-vote revocation over
    PT_GID, and ECC payload protection.

Both are ordinary :class:`~src.common.simulator.RoutingPolicy` implementations,
so they run on exactly the same world as A, B and C.
"""

import numpy as np

from ..common import config
from ..common.simulator import RoutingPolicy
from . import aodv


class AodvPolicy(RoutingPolicy):
    """Plain AODV: minimum hop count, security- and energy-blind."""

    name = "AODV"

    def select_route(self, sim, packet):
        return sim.discover(packet.src, packet.dst, mode=aodv.MODE_AODV)

    def detected_mask(self, sim):
        # Plain AODV has no detection mechanism at all, so it never accuses
        # anyone. Reporting an all-false mask keeps its TPR/FPR honest (0/0)
        # rather than inheriting the trust layer's verdicts.
        return np.zeros(sim.nodes.n, dtype=bool)


class EaurpPolicy(RoutingPolicy):
    """base.pdf EAURP: trust- and energy-aware route selection with revocation."""

    name = "EAURP"

    def __init__(self, rescore_period=10):
        self.rescore_period = int(rescore_period)

    def reset(self, sim):
        sim.refresh_scores()

    def on_round_start(self, sim, round_idx):
        # Route scores drift as energy drains and trust is updated; recomputing
        # every round would dominate the runtime for no measurable benefit.
        if round_idx % self.rescore_period == 0:
            sim.refresh_scores()

    def select_route(self, sim, packet):
        return sim.discover(packet.src, packet.dst, mode=aodv.MODE_EAURP)

    def detected_mask(self, sim):
        return sim.nodes.revoked.copy()

    def extra_metrics(self, sim):
        a, b = aodv.score_coefficients(sim.nodes)
        return {
            "score_weight_trust_a": a,
            "score_weight_energy_b": b,
            "mean_route_score": float(sim.scores.mean()),
        }


In [ ]:
%%writefile src/a_drl_eaurp/__init__.py
"""Implementation A -- DRL-EAURP, written from Lekha.pdf.

This is an independent re-implementation of the base paper from its published
equations (1)-(38). It was written from the paper alone; the senior's notebook
was consulted only afterwards, to compile the list of deviations reported in
``DEVIATIONS``.

The paper defines five models, and A implements all five:

    Existing      Eq. (14)-(18)   -- derived baseline, not a protocol
    EAURP base    Eq. (4)-(13)
    ATEAURP       Eq. (19)-(24)
    PSE-EAURP     Eq. (25)-(28)
    DRL-EAURP     Eq. (29)-(38)

Track 1 keeps the paper's own probabilistic structure. Track 2 mounts the same
DRL agent on the mechanistic harness (``policy_common``) so A can be compared
with B and C on a world where routes actually exist.
"""


In [ ]:
%%writefile src/a_drl_eaurp/paper_model.py
"""Track-1 scaffolding for DRL-EAURP, built from Lekha.pdf's equations.

Everything here traces to a numbered equation in the paper:

    Eq. (4)      link exists iff d_ij <= R
    Eq. (5)-(6)  destination-driven mobility
    Eq. (8)      energy depletion, dE in [0.05, 0.15]
    Eq. (9)-(13) PDR, throughput, average delay, per-packet delay, packet loss
    Eq. (19)     PFR_i = F_i / R_i
    Eq. (20)     T_i(t+1) = 0.7 T_i(t) + 0.3 PFR_i
    Eq. (21)     malicious iff T_i < 0.6 and R_i > 5
    Eq. (22)     M_i = 1 / (1 + v_i / v_max)
    Eq. (23)     E_norm = E_i / E_init

DOCUMENTED ASSUMPTIONS -- the paper does not specify these, so a choice had to
be made. Each is flagged here and repeated in the report, because they are
exactly the places where an independent reading can diverge from the delivered
code.

A1. **Who accrues forwarding credit.** Eq. (19) defines ``PFR = F_i / R_i`` but
    the paper never says which nodes increment F and R. It does model a packet
    as traversing ``H_k`` hops (Eq. 12), so we attribute the forward/receive
    counters to the ``H_k`` relay nodes actually carrying the packet. That makes
    trust a property of relaying behaviour, which is what Eq. (19) is clearly
    for. The delivered code instead credits the randomly chosen source and
    destination, which are not relays at all.

A2. **Energy drain band.** Eq. (8) gives one band, ``dE in [0.05, 0.15]``, and
    Sec. IV-D-6 says depletion is "continuously modeled as in" that equation for
    the DRL model too. A therefore uses [0.05, 0.15] for *every* variant. The
    delivered code silently uses [0.04, 0.12] for ATEAURP/PSE/DRL, which is
    where their entire network-lifetime advantage comes from.

A3. **Hop count.** Eq. (12) uses ``H_k`` without defining its distribution. We
    draw ``H ~ U{3..10}``, the only hop range consistent with the paper's
    reported delays.
"""

import math
import random

# Paper parameters -- Lekha.pdf Sec. IV-A
AREA_SIZE = 1000.0
COMM_RANGE = 150.0
INITIAL_ENERGY = 100.0
MIN_SPEED = 10000.0
MAX_SPEED = 40000.0
TRUST_THRESHOLD = 0.6          # Eq. (21)
MIN_RECEIVED_FOR_VERDICT = 5   # Eq. (21)

# Eq. (8): the single drain band the paper specifies. See assumption A2.
PAPER_DRAIN_MIN = 0.05
PAPER_DRAIN_MAX = 0.15

HOP_MIN, HOP_MAX = 3, 10       # assumption A3
PACKET_MIN, PACKET_MAX = 512, 1024
SIM_ROUNDS = 2000
DEAD_FRACTION = 0.8

# Deviations between the published paper and the delivered code, found by
# implementing A from the equations first and diffing afterwards.
DEVIATIONS = [
    {
        "topic": "Energy drain band",
        "paper": "Eq. (8): dE in [0.05, 0.15] for the energy model; Sec. IV-D-6 "
                 "states DRL-EAURP depletes energy the same way.",
        "code": "run_simulation uses U(0.05, 0.15) but run_proposed_simulation, "
                "run_predictive_simulation and run_drl_simulation all use "
                "U(0.04, 0.12).",
        "impact": "The reported ~1230-round lifetime of the proposed models "
                  "versus ~1000 for the base is produced entirely by this "
                  "undocumented constant change, not by any protocol behaviour.",
    },
    {
        "topic": "Trust counter attribution",
        "paper": "Eq. (19) PFR_i = F_i / R_i, i.e. what node i forwarded out of "
                 "what it received.",
        "code": "nodes[src].forwarded += 1 and nodes[dst].received += 1 -- the "
                "numerator and denominator are accumulated on two different, "
                "randomly chosen nodes that are not relays.",
        "impact": "PFR is not a forwarding ratio of anything. node.trust becomes "
                  "an unanchored drift, and node.malicious is never read.",
    },
    {
        "topic": "Existing baseline",
        "paper": "Eq. (14)-(18) openly define 'Existing' as EAURP output scaled "
                 "by fixed constants.",
        "code": "Same -- delay x1.2, loss x1.5, throughput x0.7, PDR x0.85, "
                "lifetime x0.98.",
        "impact": "Faithfully implemented, but it is a derived curve, not a "
                  "competing protocol. Every 'improvement over Existing' is "
                  "arithmetic, not a measurement.",
    },
    {
        "topic": "DRL action effect",
        "paper": "Eq. (36): P = P_base + 0.08 if A=0 else P_base + 0.02.",
        "code": "Same.",
        "impact": "Action 0 dominates action 1 unconditionally and independently "
                  "of state, so the optimal policy is constant. The Q-table "
                  "converges within a handful of updates and the state "
                  "<T,E,M> has no influence on the outcome.",
    },
    {
        "topic": "Trust threshold effect",
        "paper": "Eq. (21) classifies a node malicious when T_i < 0.6 and R_i > 5.",
        "code": "Sets node.malicious, but no routing, scoring or delivery "
                "decision ever reads that flag.",
        "impact": "The security mechanism has no effect on any reported metric.",
    },
]


class PaperNode(object):
    """A node exactly as the paper describes one."""

    def __init__(self, node_id, rng):
        self.id = node_id
        self.x = rng.uniform(0.0, AREA_SIZE)
        self.y = rng.uniform(0.0, AREA_SIZE)
        self.dest_x = rng.uniform(0.0, AREA_SIZE)
        self.dest_y = rng.uniform(0.0, AREA_SIZE)
        self.speed = rng.uniform(MIN_SPEED, MAX_SPEED)
        self.energy = INITIAL_ENERGY

        # Eq. (19)-(21)
        self.forwarded = 0
        self.received = 0
        self.trust = 1.0
        self.malicious = False

        # Eq. (25): trust history for the predictive model
        self.trust_history = [1.0, 1.0, 1.0]
        self.neighbors = []

    def mobility_factor(self):
        """Eq. (22)."""
        return 1.0 / (1.0 + self.speed / MAX_SPEED)

    def normalised_energy(self):
        """Eq. (23)."""
        return max(0.0, self.energy) / INITIAL_ENERGY


def create_network(n, rng):
    return [PaperNode(index, rng) for index in range(int(n))]


def update_neighbors(nodes):
    """Eq. (4): a link exists iff the Euclidean distance is at most R."""
    for node in nodes:
        node.neighbors = []
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            first, second = nodes[i], nodes[j]
            dx = first.x - second.x
            dy = first.y - second.y
            if math.sqrt(dx * dx + dy * dy) <= COMM_RANGE:
                first.neighbors.append(second.id)
                second.neighbors.append(first.id)


def move_network(nodes, rng):
    """Eq. (5)-(6): move each node toward its destination."""
    for node in nodes:
        dx = node.dest_x - node.x
        dy = node.dest_y - node.y
        distance = math.sqrt(dx * dx + dy * dy)
        if distance < 1.0:
            node.dest_x = rng.uniform(0.0, AREA_SIZE)
            node.dest_y = rng.uniform(0.0, AREA_SIZE)
            continue
        node.x += (dx / distance) * node.speed * 0.001
        node.y += (dy / distance) * node.speed * 0.001


def deplete_energy(nodes, rng, drain_min=PAPER_DRAIN_MIN, drain_max=PAPER_DRAIN_MAX):
    """Eq. (8). Returns the ids of nodes whose battery ran out this round."""
    newly_dead = []
    for node in nodes:
        if node.energy <= 0.0:
            continue
        node.energy -= rng.uniform(drain_min, drain_max)
        if node.energy <= 0.0:
            newly_dead.append(node.id)
    return newly_dead


def relay_nodes(nodes, source, destination, hops, rng):
    """Assumption A1 -- the ``H_k`` relays that actually carry a packet.

    The paper models a packet as crossing ``H_k`` hops but never names the
    intermediate nodes, so we sample them. This is what lets Eq. (19)'s
    forwarding ratio describe forwarding.
    """
    candidates = [node.id for node in nodes
                  if node.id not in (source, destination) and node.energy > 0.0]
    if not candidates:
        return []
    count = min(int(hops), len(candidates))
    return rng.sample(candidates, count)


def update_trust(node):
    """Eq. (19)-(21)."""
    if node.received == 0:
        return
    pfr = node.forwarded / float(node.received)
    node.trust = 0.7 * node.trust + 0.3 * pfr
    if node.trust < TRUST_THRESHOLD and node.received > MIN_RECEIVED_FOR_VERDICT:
        node.malicious = True


def push_trust_history(node):
    """Eq. (27): slide the three-deep trust history."""
    node.trust_history.append(node.trust)
    if len(node.trust_history) > 3:
        node.trust_history.pop(0)


def predict_trust(node):
    """Eq. (26): T_pred = 0.5 T_t + 0.3 T_{t-1} + 0.2 T_{t-2}."""
    history = node.trust_history
    return 0.5 * history[-1] + 0.3 * history[-2] + 0.2 * history[-3]


def network_state(nodes):
    """Eq. (30)-(32): the mean trust, energy and mobility of the network."""
    live = [node for node in nodes if node.energy > 0.0] or nodes
    count = float(len(live))
    trust = sum(node.trust for node in live) / count
    energy = sum(node.normalised_energy() for node in live) / count
    mobility = sum(node.mobility_factor() for node in live) / count
    return trust, energy, mobility


def run_paper_simulation(variant, num_nodes, speed, rounds=SIM_ROUNDS, seed=42,
                         drain_min=PAPER_DRAIN_MIN, drain_max=PAPER_DRAIN_MAX):
    """Run one variant of the paper's model.

    ``variant`` supplies the success probability (Eq. 7 / 24 / 28 / 35-36) and
    optionally reacts to the outcome (the DRL Q-update, Eq. 3 / 37).
    """
    rng = random.Random(seed)
    nodes = create_network(num_nodes, rng)
    for node in nodes:
        node.speed = float(speed)
    update_neighbors(nodes)

    variant.reset(nodes, rng)

    packets_sent = 0
    packets_received = 0
    packet_loss_bytes = 0
    total_delay = 0.0
    total_data = 0
    first_dead_round = None
    last_round = 0

    for round_idx in range(int(rounds)):
        last_round = round_idx
        move_network(nodes, rng)
        update_neighbors(nodes)

        source = rng.randrange(num_nodes)
        destination = rng.randrange(num_nodes)
        if source == destination:
            continue

        packets_sent += 1
        packet_size = rng.randint(PACKET_MIN, PACKET_MAX)
        hops = rng.randint(HOP_MIN, HOP_MAX)

        state = variant.observe(nodes, rng)
        probability = variant.success_probability(nodes, state, rng)
        success = rng.random() < probability

        if success:
            packets_received += 1
            # Eq. (12): D_k = D_base + H_k * D_hop
            delay = variant.delay(hops, rng)
            total_delay += delay
            total_data += packet_size
        else:
            packet_loss_bytes += packet_size

        # Assumption A1: credit the relays that carried the packet.
        relays = relay_nodes(nodes, source, destination, hops, rng)
        for relay_id in relays:
            node = nodes[relay_id]
            node.received += 1
            if success:
                node.forwarded += 1
            variant.update_node_trust(node)

        variant.feedback(nodes, state, success, rng)

        newly_dead = deplete_energy(nodes, rng, drain_min, drain_max)
        if newly_dead and first_dead_round is None:
            first_dead_round = round_idx

        dead = sum(1 for node in nodes if node.energy <= 0.0)
        if dead >= num_nodes * DEAD_FRACTION:
            break

    # Eq. (9)-(13)
    avg_delay = total_delay / packets_received if packets_received else 0.0
    simulation_time = max(last_round, 1)
    throughput = (total_data * 8.0) / (simulation_time * 1000.0)
    pdr = packets_received / float(packets_sent) if packets_sent else 0.0
    lifetime = first_dead_round if first_dead_round is not None else simulation_time

    flagged = sum(1 for node in nodes if node.malicious)
    return {
        "avg_delay_ms": avg_delay,
        "packet_loss_bytes": packet_loss_bytes,
        "throughput": throughput,
        "pdr": pdr,
        "network_lifetime": lifetime,
        "packets_sent": packets_sent,
        "packets_received": packets_received,
        "rounds_completed": simulation_time,
        "nodes_flagged_malicious": flagged,
    }


class PaperVariant(object):
    """Interface each of the paper's five models implements."""

    name = "variant"
    equations = ""

    def reset(self, nodes, rng):
        return None

    def observe(self, nodes, rng):
        """State handed to :meth:`success_probability` (Eq. 29 for the DRL model)."""
        return None

    def success_probability(self, nodes, state, rng):
        raise NotImplementedError

    def delay(self, hops, rng):
        """Eq. (12). The paper gives different bands per model."""
        return rng.uniform(20.0, 60.0) + hops * rng.uniform(5.0, 15.0)

    def update_node_trust(self, node):
        """Default: no trust model (the base EAURP model of Eq. 7)."""
        return None

    def feedback(self, nodes, state, success, rng):
        return None


In [ ]:
%%writefile src/a_drl_eaurp/variants.py
"""The five models of Lekha.pdf, each written from its own equations.

    ExistingModel   Eq. (14)-(18)
    EaurpBaseModel  Eq. (7)
    AteaurpModel    Eq. (19)-(24)
    PseEaurpModel   Eq. (25)-(28)
    DrlEaurpModel   Eq. (3), (29)-(38)

Every variant is a :class:`~src.a_drl_eaurp.paper_model.PaperVariant`, so they
all run through the same loop and differ only where the paper says they differ.
"""

import random

import numpy as np

from . import paper_model as pm

# Eq. (14)-(18): the constants the paper uses to derive its "Existing" curve.
EXISTING_SCALING = {
    "avg_delay_ms": 1.2,
    "packet_loss_bytes": 1.5,
    "throughput": 0.7,
    "pdr": 0.85,
    "network_lifetime": 0.98,
}


class EaurpBaseModel(pm.PaperVariant):
    """Eq. (7): P_success = max(0.4, 0.9 - v / 100000).

    Mobility is the only input. There is no trust term and no energy term, so
    this variant is the paper's own statement that its baseline ignores both.
    """

    name = "EAURP"
    equations = "Eq. (4)-(13)"

    def success_probability(self, nodes, state, rng):
        speed = nodes[0].speed if nodes else 0.0
        return max(0.4, 0.9 - (speed / 100000.0))

    def delay(self, hops, rng):
        return rng.uniform(20.0, 60.0) + hops * rng.uniform(5.0, 15.0)


class AteaurpModel(pm.PaperVariant):
    """Eq. (24): P = min(0.4 + 0.3 T + 0.2 E + 0.1 M, 0.95).

    Adds the adaptive trust update of Eq. (20) and the classification rule of
    Eq. (21).
    """

    name = "ATEAURP"
    equations = "Eq. (19)-(24)"

    def observe(self, nodes, rng):
        return pm.network_state(nodes)

    def success_probability(self, nodes, state, rng):
        trust, energy, mobility = state
        return min(0.4 + 0.3 * trust + 0.2 * energy + 0.1 * mobility, 0.95)

    def delay(self, hops, rng):
        return rng.uniform(20.0, 60.0) + hops * rng.uniform(4.0, 12.0)

    def update_node_trust(self, node):
        pm.update_trust(node)


class PseEaurpModel(AteaurpModel):
    """Eq. (28): P = min(0.4 + 0.35 T_pred + 0.15 E + 0.1 M, 0.97).

    Trust is replaced by the three-tap forecast of Eq. (26).
    """

    name = "PSE-EAURP"
    equations = "Eq. (25)-(28)"

    def observe(self, nodes, rng):
        live = [node for node in nodes if node.energy > 0.0] or nodes
        count = float(len(live))
        predicted = sum(pm.predict_trust(node) for node in live) / count
        _, energy, mobility = pm.network_state(nodes)
        return predicted, energy, mobility

    def success_probability(self, nodes, state, rng):
        predicted, energy, mobility = state
        return min(0.4 + 0.35 * predicted + 0.15 * energy + 0.1 * mobility, 0.97)

    def delay(self, hops, rng):
        return rng.uniform(20.0, 50.0) + hops * rng.uniform(4.0, 10.0)

    def update_node_trust(self, node):
        pm.update_trust(node)
        pm.push_trust_history(node)


class DrlEaurpModel(pm.PaperVariant):
    """Eq. (29)-(38): tabular Q-learning over the state <T, E, M>.

    Eq. (35): P_base = 0.45 + 0.25 T + 0.20 E + 0.10 M
    Eq. (36): P = P_base + 0.08 if A = 0 else P_base + 0.02, clipped to [0.4, 0.98]
    Eq. (37): r = +1 delivered, -1 lost
    Eq. (3):  Q(s,a) <- Q(s,a) + alpha [r + gamma max_a' Q(s',a') - Q(s,a)]
    Eq. (34): epsilon-greedy policy

    NOTE FOR THE REVIEW. Eq. (36) makes action 0 strictly better than action 1
    for every state, so the optimal policy is the constant "always exploit".
    The agent learns it within a few dozen updates, after which the state has no
    influence on anything. :meth:`policy_summary` reports how often the greedy
    action was 0, which makes that degeneracy measurable rather than asserted.
    """

    name = "DRL-EAURP"
    equations = "Eq. (3), (29)-(38)"

    def __init__(self, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.alpha = float(alpha)
        self.gamma = float(gamma)
        self.epsilon = float(epsilon)
        self.q_table = {}
        self.action_counts = [0, 0]
        self.updates = 0
        self._last = None

    # -- Q-table helpers ---------------------------------------------------

    def _q(self, state):
        if state not in self.q_table:
            self.q_table[state] = [0.0, 0.0]
        return self.q_table[state]

    def reset(self, nodes, rng):
        self.q_table = {}
        self.action_counts = [0, 0]
        self.updates = 0
        self._last = None

    def observe(self, nodes, rng):
        """Eq. (29)-(32), discretised to one decimal as the paper's code does."""
        trust, energy, mobility = pm.network_state(nodes)
        key = (round(trust, 1), round(energy, 1), round(mobility, 1))
        return key, trust, energy, mobility

    def success_probability(self, nodes, state, rng):
        key, trust, energy, mobility = state

        # Eq. (34): epsilon-greedy
        if rng.random() < self.epsilon:
            action = rng.randint(0, 1)
        else:
            values = self._q(key)
            action = 0 if values[0] >= values[1] else 1

        self.action_counts[action] += 1
        self._last = (key, action)

        # Eq. (35)-(36)
        base = 0.45 + 0.25 * trust + 0.20 * energy + 0.10 * mobility
        probability = base + (0.08 if action == 0 else 0.02)
        return min(max(probability, 0.4), 0.98)

    def delay(self, hops, rng):
        return rng.uniform(20.0, 50.0) + hops * rng.uniform(4.0, 10.0)

    def update_node_trust(self, node):
        pm.update_trust(node)

    def feedback(self, nodes, state, success, rng):
        """Eq. (37) reward, Eq. (3) Q-update."""
        if self._last is None:
            return
        key, action = self._last
        reward = 1.0 if success else -1.0

        next_key = self.observe(nodes, rng)[0] if nodes else key
        best_next = max(self._q(next_key))

        values = self._q(key)
        values[action] += self.alpha * (
            reward + self.gamma * best_next - values[action]
        )
        self.updates += 1

    def policy_summary(self):
        """Evidence for the degeneracy noted in the class docstring."""
        total = sum(self.action_counts) or 1
        greedy_zero = sum(
            1 for values in self.q_table.values() if values[0] >= values[1]
        )
        return {
            "q_states_visited": len(self.q_table),
            "q_updates": self.updates,
            "action0_share": self.action_counts[0] / float(total),
            "states_preferring_action0": greedy_zero,
            "states_total": len(self.q_table),
        }


class ExistingModel(object):
    """Eq. (14)-(18) -- NOT a protocol.

    The paper derives its "Existing" curve by scaling EAURP's own output by
    fixed constants. Implemented faithfully, and labelled in every output so it
    is never mistaken for a measurement.
    """

    name = "Existing"
    equations = "Eq. (14)-(18)"
    is_derived = True

    @staticmethod
    def derive(eaurp_result):
        derived = dict(eaurp_result)
        for key, factor in EXISTING_SCALING.items():
            if key in derived:
                derived[key] = derived[key] * factor
        derived["provenance"] = (
            "DERIVED from EAURP output via Lekha.pdf Eq. (14)-(18); "
            "not a simulated protocol"
        )
        return derived


VARIANTS = {
    "EAURP": EaurpBaseModel,
    "ATEAURP": AteaurpModel,
    "PSE-EAURP": PseEaurpModel,
    "DRL-EAURP": DrlEaurpModel,
}


def make_variant(name):
    key = str(name).upper().replace("_", "-")
    for candidate, cls in VARIANTS.items():
        if candidate.upper() == key:
            return cls()
    raise ValueError(
        "unknown variant {0!r}; choose from {1}".format(name, sorted(VARIANTS))
    )


In [ ]:
%%writefile src/a_drl_eaurp/run_track1.py
"""Track-1 driver for implementation A.

Reproduces the paper's speed and node-count sweeps for all five models, using
the paper's own equations and -- crucially -- the single energy drain band that
Eq. (8) actually specifies, for every variant. See assumption A2 in
``paper_model``: the delivered code quietly uses a gentler band for the three
proposed models, which is where their entire lifetime advantage comes from.
"""

import csv
import os

import numpy as np

from . import paper_model as pm
from .variants import DrlEaurpModel, ExistingModel, make_variant

SPEED_RANGE = [10000, 15000, 20000, 25000, 30000, 35000, 40000]
NODE_COUNTS = [60, 80, 100, 120, 150, 200]
MODEL_ORDER = ["EAURP", "ATEAURP", "PSE-EAURP", "DRL-EAURP"]


def run_speed_sweep(rounds=pm.SIM_ROUNDS, runs=3, speeds=None, n_nodes=100,
                    base_seed=42, drain_min=None, drain_max=None, verbose=True):
    """Every model across the speed range, averaged over ``runs`` seeds."""
    speeds = SPEED_RANGE if speeds is None else speeds
    drain_min = pm.PAPER_DRAIN_MIN if drain_min is None else drain_min
    drain_max = pm.PAPER_DRAIN_MAX if drain_max is None else drain_max

    results = {}
    policy_notes = {}

    for model_name in MODEL_ORDER:
        per_speed = []
        for speed in speeds:
            runs_out = []
            variant = make_variant(model_name)
            for run_index in range(int(runs)):
                out = pm.run_paper_simulation(
                    variant, n_nodes, speed, rounds=rounds,
                    seed=base_seed + 1000 * run_index,
                    drain_min=drain_min, drain_max=drain_max,
                )
                runs_out.append(out)
            averaged = {
                key: float(np.mean([row[key] for row in runs_out]))
                for key in runs_out[0]
            }
            averaged["speed"] = speed
            per_speed.append(averaged)
            if isinstance(variant, DrlEaurpModel):
                policy_notes[speed] = variant.policy_summary()
        results[model_name] = per_speed
        if verbose:
            pdrs = [row["pdr"] for row in per_speed]
            delays = [row["avg_delay_ms"] for row in per_speed]
            lifetimes = [row["network_lifetime"] for row in per_speed]
            print("  A/{0:11s} PDR {1:.3f}-{2:.3f}  delay {3:.1f}-{4:.1f} ms  "
                  "lifetime {5:.0f}".format(
                      model_name, min(pdrs), max(pdrs),
                      min(delays), max(delays), float(np.mean(lifetimes))))

    # Eq. (14)-(18): derive "Existing" from the EAURP curve.
    results["Existing"] = [ExistingModel.derive(row) for row in results["EAURP"]]
    if verbose:
        pdrs = [row["pdr"] for row in results["Existing"]]
        print("  A/{0:11s} PDR {1:.3f}-{2:.3f}   [DERIVED, Eq. 14-18]".format(
            "Existing", min(pdrs), max(pdrs)))

    return results, policy_notes


def run_density_sweep(rounds=pm.SIM_ROUNDS, runs=3, node_counts=None, speed=20000,
                      base_seed=42, verbose=True):
    """Delay, loss and reliability against node count (paper Figs. 4 and 6)."""
    node_counts = NODE_COUNTS if node_counts is None else node_counts
    results = {}
    for model_name in MODEL_ORDER:
        per_count = []
        for count in node_counts:
            variant = make_variant(model_name)
            runs_out = [
                pm.run_paper_simulation(
                    variant, count, speed, rounds=rounds,
                    seed=base_seed + 1000 * run_index,
                )
                for run_index in range(int(runs))
            ]
            averaged = {
                key: float(np.mean([row[key] for row in runs_out]))
                for key in runs_out[0]
            }
            averaged["n_nodes"] = count
            per_count.append(averaged)
        results[model_name] = per_count
        if verbose:
            print("  A/{0:11s} density sweep done".format(model_name))
    results["Existing"] = [ExistingModel.derive(row) for row in results["EAURP"]]
    return results


def drain_band_ablation(rounds=pm.SIM_ROUNDS, runs=3, speed=20000, n_nodes=100,
                        base_seed=42):
    """Isolate the effect of the undocumented drain-band change.

    Runs DRL-EAURP twice: once with the band Eq. (8) specifies, once with the
    band the delivered code actually uses. If the lifetime gap the paper
    attributes to its protocol is really just this constant, this shows it.
    """
    out = {}
    for label, (low, high) in (
        ("paper_Eq8_0.05-0.15", (0.05, 0.15)),
        ("code_0.04-0.12", (0.04, 0.12)),
    ):
        rows = [
            pm.run_paper_simulation(
                make_variant("DRL-EAURP"), n_nodes, speed, rounds=rounds,
                seed=base_seed + 1000 * index, drain_min=low, drain_max=high,
            )
            for index in range(int(runs))
        ]
        out[label] = {
            key: float(np.mean([row[key] for row in rows])) for key in rows[0]
        }
    lifetimes = [out[key]["network_lifetime"] for key in out]
    out["lifetime_delta"] = float(lifetimes[1] - lifetimes[0])
    return out


def to_rows(speed_results, density_results=None):
    """Flatten into CSV rows."""
    rows = []
    for model_name, entries in speed_results.items():
        for entry in entries:
            row = {
                "track": "track1_as_is",
                "implementation": "A_paper_reimplementation",
                "model": model_name,
                "sweep": "speed",
                "speed": entry.get("speed"),
                "n_nodes": 100,
            }
            for key in ("pdr", "avg_delay_ms", "throughput", "packet_loss_bytes",
                        "network_lifetime", "packets_sent", "packets_received",
                        "nodes_flagged_malicious"):
                row[key] = entry.get(key)
            row["derived"] = bool(model_name == "Existing")
            rows.append(row)

    if density_results:
        for model_name, entries in density_results.items():
            for entry in entries:
                row = {
                    "track": "track1_as_is",
                    "implementation": "A_paper_reimplementation",
                    "model": model_name,
                    "sweep": "density",
                    "speed": 20000,
                    "n_nodes": entry.get("n_nodes"),
                }
                for key in ("pdr", "avg_delay_ms", "throughput",
                            "packet_loss_bytes", "network_lifetime",
                            "packets_sent", "packets_received",
                            "nodes_flagged_malicious"):
                    row[key] = entry.get(key)
                row["derived"] = bool(model_name == "Existing")
                rows.append(row)
    return rows


def write_csv(rows, path):
    directory = os.path.dirname(os.path.abspath(path))
    if directory:
        os.makedirs(directory, exist_ok=True)
    fieldnames = sorted({key for row in rows for key in row})
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path


def main(out_dir="results/csv", rounds=pm.SIM_ROUNDS, runs=3, density=True):
    print("A (DRL-EAURP re-implemented from Lekha.pdf equations)")
    speed_results, policy_notes = run_speed_sweep(rounds=rounds, runs=runs)
    density_results = (
        run_density_sweep(rounds=rounds, runs=runs) if density else None
    )
    rows = to_rows(speed_results, density_results)
    path = write_csv(rows, os.path.join(out_dir, "a_paper_track1.csv"))
    print("  ->", path)

    if policy_notes:
        sample = policy_notes[sorted(policy_notes)[0]]
        print("  DRL policy check: {0} states visited, action-0 share "
              "{1:.3f}, {2}/{3} states prefer action 0".format(
                  sample["q_states_visited"], sample["action0_share"],
                  sample["states_preferring_action0"], sample["states_total"]))
    return speed_results, density_results, policy_notes


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/a_drl_eaurp/policy_common.py
"""Implementation A on the mechanistic harness (Track 2).

The agent is the paper's: tabular Q-learning over the state <T, E, M> of
Eq. (29)-(32), rounded to one decimal, an epsilon-greedy policy (Eq. 34), two
actions (Eq. 33), the +/-1 reward of Eq. (37) and the update of Eq. (3).

THE ONE NECESSARY TRANSLATION. Eq. (36) expresses an action as an additive
bonus to a global success probability -- ``+0.08`` for exploit, ``+0.02`` for
explore. That is only meaningful when there is no route. Here routes exist, so
the two actions map to what they are *named* after:

    A = 0  exploit  -> take the highest-trust route
    A = 1  explore  -> take a diversified (node-disjoint) route

This is the smallest change that lets the paper's agent act at all, and it is
strictly favourable to the paper: an agent choosing between real routes has
more to work with than one adding a constant to a coin flip.

TRUST ATTRIBUTION follows Eq. (19) as written -- a node's forwarding ratio is
what *it* relayed out of what *it* received, measured by the watchdog. This is
the point where A and B diverge on Track 2: the delivered code credits the
randomly chosen source and destination instead, which is reproduced faithfully
in ``src/b_senior/policy_common.py``.
"""

import numpy as np

from ..common import config
from ..common.simulator import RoutingPolicy
from ..routing import aodv

ACTION_EXPLOIT = 0
ACTION_EXPLORE = 1


class TabularDrlPolicy(RoutingPolicy):
    """Lekha.pdf's Q-learning agent driving real route selection."""

    name = "A:DRL-EAURP"
    trust_source = "watchdog_pfr"   # Eq. (19) as written

    def __init__(self, alpha=None, gamma=None, epsilon=None, rescore_period=10):
        self.alpha = config.RL_ALPHA if alpha is None else float(alpha)
        self.gamma = config.RL_GAMMA if gamma is None else float(gamma)
        self.epsilon = config.RL_EPSILON if epsilon is None else float(epsilon)
        self.rescore_period = int(rescore_period)

        self.q_table = {}
        self.action_counts = [0, 0]
        self.updates = 0
        self.total_reward = 0.0
        self._pending = None

    # -- state and policy --------------------------------------------------

    def _q(self, state):
        if state not in self.q_table:
            self.q_table[state] = [0.0, 0.0]
        return self.q_table[state]

    def state_of(self, sim):
        """Eq. (29)-(32), discretised to one decimal."""
        nodes = sim.nodes
        alive = nodes.alive
        if not alive.any():
            return (0.0, 0.0, 0.0)
        trust = float(nodes.network_trust()[alive].mean())
        energy = float(nodes.normalised_energy()[alive].mean())
        mobility = float(nodes.mobility_factor()[alive].mean())
        return (round(trust, 1), round(energy, 1), round(mobility, 1))

    def choose_action(self, sim, state):
        """Eq. (34): epsilon-greedy."""
        if sim.seeds.policy.random() < self.epsilon:
            return int(sim.seeds.policy.integers(0, 2))
        values = self._q(state)
        return ACTION_EXPLOIT if values[0] >= values[1] else ACTION_EXPLORE

    # -- harness hooks -----------------------------------------------------

    def reset(self, sim):
        self.q_table = {}
        self.action_counts = [0, 0]
        self.updates = 0
        self.total_reward = 0.0
        self._pending = None
        sim.refresh_scores(trust=self.trust_vector(sim))

    def trust_vector(self, sim):
        """Per-node trust the policy feeds into route scoring."""
        return sim.nodes.network_trust()

    def on_round_start(self, sim, round_idx):
        if round_idx % self.rescore_period == 0:
            sim.refresh_scores(trust=self.trust_vector(sim))

    def select_route(self, sim, packet):
        state = self.state_of(sim)
        action = self.choose_action(sim, state)
        self.action_counts[action] += 1

        if action == ACTION_EXPLOIT:
            path = sim.discover(packet.src, packet.dst, mode=aodv.MODE_TRUST)
        else:
            primary = sim.discover(packet.src, packet.dst, mode=aodv.MODE_EAURP)
            path = aodv.find_diverse_route(
                sim.nodes, packet.src, packet.dst, sim.scores, primary=primary
            )

        self._pending = (state, action)
        return path

    def on_result(self, sim, packet, delivered, path, hops, reason):
        """Eq. (37) reward, Eq. (3) update."""
        if self._pending is None:
            return
        state, action = self._pending
        reward = config.REWARD_SUCCESS if delivered else config.REWARD_FAILURE
        self.total_reward += reward

        next_state = self.state_of(sim)
        best_next = max(self._q(next_state))
        values = self._q(state)
        values[action] += self.alpha * (
            reward + self.gamma * best_next - values[action]
        )
        self.updates += 1
        self._pending = None

    def detected_mask(self, sim):
        return sim.nodes.revoked.copy()

    def extra_metrics(self, sim):
        total = sum(self.action_counts) or 1
        greedy_zero = sum(
            1 for values in self.q_table.values() if values[0] >= values[1]
        )
        return {
            "q_states": len(self.q_table),
            "q_updates": self.updates,
            "action0_share": self.action_counts[0] / float(total),
            "states_preferring_action0": greedy_zero,
            "mean_reward": self.total_reward / float(max(1, self.updates)),
            "trust_source": self.trust_source,
        }


In [ ]:
%%writefile src/b_senior/__init__.py
"""Implementation B -- the senior's delivered code.

``senior_asis``
    The notebook's logic, copied verbatim. No algorithmic change whatsoever:
    the only edits are a ``main()`` guard, a headless matplotlib backend and
    file output, so it can run unattended and write CSVs. Every function body,
    constant and random draw is the senior's.

``policy_common``
    The senior's tabular Q-learning agent lifted onto the mechanistic harness,
    so B can be compared with A and C on identical worlds (Track 2).
"""


In [ ]:
%%writefile src/b_senior/senior_asis.py
"""Implementation B -- senior_code.ipynb, logic copied verbatim.

TRANSCRIPTION POLICY
--------------------
Every function body below is character-for-character the senior's, in the order
the notebook executes them. ``random.seed(42)`` is set once, in the same place
(notebook cell 3), because their published numbers only reproduce when the
notebook is run top to bottom.

The only additions are: this docstring, a ``main()`` guard, an ``Agg``
matplotlib backend, CSV/PNG output, and the ``PROVENANCE`` table below. No
constant, formula or random draw has been altered.

WHAT THIS CODE ACTUALLY IS -- read before interpreting any of its numbers
------------------------------------------------------------------------
This is not a network simulator. It is a metric generator. ``src`` and ``dst``
are drawn at random each round and are never connected by a path; delivery is
one coin flip against a *global average*:

    success_prob = 0.45 + 0.25*avg_trust + 0.2*avg_energy + 0.1*avg_mobility
    success = random.random() < success_prob

Consequences, all verifiable in the code below:

1. No route exists, so trust never selects a path.
2. ``nodes[src].forwarded += 1`` and ``nodes[dst].received += 1`` accumulate the
   Packet Forwarding Ratio across *unrelated* nodes, so ``node.trust`` is a
   drift, not a measurement.
3. ``node.malicious`` is assigned by ``adaptive_trust_update`` and then never
   read by anything.
4. The Q-learning is degenerate: action 0 always adds +0.08 and action 1 always
   +0.02 regardless of state, so Q converges to "always exploit" almost
   immediately and ``get_state`` has no influence on behaviour.
5. The "Existing" baseline is EAURP's own output multiplied by constants
   (Lekha.pdf Eq. 14-18), not a protocol.
6. The network-lifetime difference between the base and proposed models comes
   entirely from the drain constants -- U(0.05,0.15) versus U(0.04,0.12).

None of that is a defect in the transcription; it is what the delivered code
does. It is reported here so the comparison in the report can be honest.
"""

import csv
import math
import os
import random
from collections import defaultdict

import numpy as np

# Which reported numbers are computed and which are literals. Quoted in the
# report so nobody has to take the distinction on trust.
PROVENANCE = {
    "cells_2_11": "simulated (coin-flip model, no routing)",
    "cell_12": "HARDCODED -- random.randint/uniform printed as packet trace",
    "cell_13": "HARDCODED -- random.uniform printed as measured metrics "
               "(PDR 89.04, delay 498.64 ms, throughput 17.46 kbps, "
               "lifetime 99.28%); these appear in Lekha.pdf Sec. IV-A-6 as "
               "'observed behaviour'",
    "existing_baseline": "DERIVED -- EAURP output x {1.2, 1.5, 0.7, 0.85, 0.98} "
                         "(Lekha.pdf Eq. 14-18), not a simulated protocol",
}

# ==========================================================================
# Cell 4 -- configuration (verbatim)
# ==========================================================================

# Network settings
AREA_SIZE = 1000
COMM_RANGE = 150

# Node settings
INITIAL_ENERGY = 100
ENERGY_TX = 0.5
ENERGY_RX = 0.2

# Trust parameters
TRUST_THRESHOLD = 0.6

# Speed range used in paper graphs
MIN_SPEED = 10000
MAX_SPEED = 40000

# Node counts used in graphs
NODE_COUNTS = [60, 80, 100, 120, 150, 200]

# Simulation rounds
SIM_ROUNDS = 2000


# ==========================================================================
# Cell 5 -- Node (verbatim)
# ==========================================================================

class Node:

    def __init__(self, node_id):

        self.id = node_id

        # Position
        self.x = random.uniform(0, AREA_SIZE)
        self.y = random.uniform(0, AREA_SIZE)

        # Mobility
        self.speed = random.uniform(MIN_SPEED, MAX_SPEED)

        # Destination for movement
        self.dest_x = random.uniform(0, AREA_SIZE)
        self.dest_y = random.uniform(0, AREA_SIZE)

        # Energy model
        self.energy = INITIAL_ENERGY

        # Trust model
        self.forwarded = 0
        self.received = 0
        self.trust = 1.0

        # Node status
        self.malicious = False

        # Neighbors
        self.neighbors = []

        # Routing table
        self.routes = {}


# ==========================================================================
# Cells 6-8 -- network construction and mobility (verbatim)
# ==========================================================================

def create_network(n):

    nodes = []

    for i in range(n):
        nodes.append(Node(i))

    return nodes


def update_neighbors(nodes):

    for node in nodes:
        node.neighbors = []

    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):

            n1 = nodes[i]
            n2 = nodes[j]

            dist = math.sqrt((n1.x - n2.x) ** 2 + (n1.y - n2.y) ** 2)

            if dist <= COMM_RANGE:

                n1.neighbors.append(n2.id)
                n2.neighbors.append(n1.id)


def move_network(nodes):

    for node in nodes:

        dx = node.dest_x - node.x
        dy = node.dest_y - node.y

        dist = math.sqrt(dx * dx + dy * dy)

        if dist < 1:
            node.dest_x = random.uniform(0, AREA_SIZE)
            node.dest_y = random.uniform(0, AREA_SIZE)
            continue

        node.x += (dx / dist) * node.speed * 0.001
        node.y += (dy / dist) * node.speed * 0.001


# ==========================================================================
# Cell 9 -- EAURP base model (verbatim)
# ==========================================================================

def run_simulation(num_nodes, speed):

    nodes = create_network(num_nodes)

    for n in nodes:
        n.speed = speed

    update_neighbors(nodes)

    packets_sent = 0
    packets_received = 0
    packet_loss_bytes = 0
    total_delay = 0
    total_data = 0

    first_dead_round = None

    for r in range(SIM_ROUNDS):

        move_network(nodes)
        update_neighbors(nodes)

        src = random.randint(0, num_nodes - 1)
        dst = random.randint(0, num_nodes - 1)

        if src == dst:
            continue

        packets_sent += 1

        packet_size = random.randint(512, 1024)  # bytes

        hops = random.randint(3, 10)

        # success probability decreases with speed
        success_prob = max(0.4, 0.9 - (speed / 100000))

        success = random.random() < success_prob

        if success:

            packets_received += 1

            # realistic delay model (milliseconds)
            base_delay = random.uniform(20, 60)
            hop_delay = hops * random.uniform(5, 15)

            delay = base_delay + hop_delay

            total_delay += delay

            total_data += packet_size

        else:

            packet_loss_bytes += packet_size

        # energy consumption
        for n in nodes:

            n.energy -= random.uniform(0.05, 0.15)

            if n.energy <= 0 and first_dead_round is None:
                first_dead_round = r

        dead = sum(1 for n in nodes if n.energy <= 0)

        if dead >= num_nodes * 0.8:
            break

    avg_delay = total_delay / packets_received if packets_received > 0 else 0

    simulation_time = max(r, 1)

    throughput = (total_data * 8) / (simulation_time * 1000)

    pdr = packets_received / packets_sent if packets_sent > 0 else 0

    lifetime = first_dead_round if first_dead_round else simulation_time

    return avg_delay, packet_loss_bytes, throughput, pdr, lifetime


# ==========================================================================
# Cell 17 -- ATEAURP (verbatim)
# ==========================================================================

def adaptive_trust_update(node):

    if node.received == 0:
        return

    pfr = node.forwarded / node.received

    # moving average trust update
    node.trust = 0.7 * node.trust + 0.3 * pfr

    if node.trust < TRUST_THRESHOLD and node.received > 5:
        node.malicious = True


def mobility_score(node):

    # mobility stability factor
    return 1 / (1 + node.speed / MAX_SPEED)


def run_proposed_simulation(num_nodes, speed):

    nodes = create_network(num_nodes)

    for n in nodes:
        n.speed = speed

    update_neighbors(nodes)

    packets_sent = 0
    packets_received = 0
    packet_loss_bytes = 0
    total_delay = 0
    total_data = 0

    first_dead_round = None

    for r in range(SIM_ROUNDS):

        move_network(nodes)
        update_neighbors(nodes)

        src = random.randint(0, num_nodes - 1)
        dst = random.randint(0, num_nodes - 1)

        if src == dst:
            continue

        packets_sent += 1

        packet_size = random.randint(512, 1024)

        hops = random.randint(3, 10)

        # cross layer metrics
        avg_trust = np.mean([n.trust for n in nodes])
        avg_energy = np.mean([n.energy for n in nodes]) / INITIAL_ENERGY
        avg_mobility = np.mean([mobility_score(n) for n in nodes])

        success_prob = 0.4 + 0.3 * avg_trust + 0.2 * avg_energy + 0.1 * avg_mobility
        success_prob = min(success_prob, 0.95)

        success = random.random() < success_prob

        if success:

            packets_received += 1

            base_delay = random.uniform(20, 60)
            hop_delay = hops * random.uniform(4, 12)

            delay = base_delay + hop_delay

            total_delay += delay

            total_data += packet_size

        else:

            packet_loss_bytes += packet_size

        # trust updates
        nodes[src].forwarded += 1
        nodes[dst].received += 1

        adaptive_trust_update(nodes[src])
        adaptive_trust_update(nodes[dst])

        # energy consumption
        for n in nodes:

            n.energy -= random.uniform(0.04, 0.12)

            if n.energy <= 0 and first_dead_round is None:
                first_dead_round = r

        dead = sum(1 for n in nodes if n.energy <= 0)

        if dead >= num_nodes * 0.8:
            break

    avg_delay = total_delay / packets_received if packets_received > 0 else 0

    simulation_time = max(r, 1)

    throughput = (total_data * 8) / (simulation_time * 1000)

    pdr = packets_received / packets_sent if packets_sent > 0 else 0

    lifetime = first_dead_round if first_dead_round else simulation_time

    return avg_delay, packet_loss_bytes, throughput, pdr, lifetime


# ==========================================================================
# Cells 22-25 -- PSE-EAURP (verbatim)
# ==========================================================================

def init_trust_history(nodes):
    for n in nodes:
        n.trust_history = [1.0, 1.0, 1.0]  # initial trust history


def predict_trust(node):

    t = node.trust_history

    # weighted prediction
    pred = 0.5 * t[-1] + 0.3 * t[-2] + 0.2 * t[-3]

    return pred


def update_trust_history(node):

    node.trust_history.append(node.trust)

    if len(node.trust_history) > 3:
        node.trust_history.pop(0)


def run_predictive_simulation(num_nodes, speed):

    nodes = create_network(num_nodes)

    init_trust_history(nodes)

    for n in nodes:
        n.speed = speed

    update_neighbors(nodes)

    packets_sent = 0
    packets_received = 0
    packet_loss_bytes = 0
    total_delay = 0
    total_data = 0

    first_dead_round = None

    for r in range(SIM_ROUNDS):

        move_network(nodes)
        update_neighbors(nodes)

        src = random.randint(0, num_nodes - 1)
        dst = random.randint(0, num_nodes - 1)

        if src == dst:
            continue

        packets_sent += 1

        packet_size = random.randint(512, 1024)
        hops = random.randint(3, 10)

        # NEW: predictive trust
        avg_pred_trust = np.mean([predict_trust(n) for n in nodes])
        avg_energy = np.mean([n.energy for n in nodes]) / INITIAL_ENERGY
        avg_mobility = np.mean([1 / (1 + n.speed / MAX_SPEED) for n in nodes])

        success_prob = 0.4 + 0.35 * avg_pred_trust + 0.15 * avg_energy + 0.1 * avg_mobility
        success_prob = min(success_prob, 0.97)

        success = random.random() < success_prob

        if success:

            packets_received += 1

            delay = random.uniform(20, 50) + hops * random.uniform(4, 10)

            total_delay += delay
            total_data += packet_size

        else:
            packet_loss_bytes += packet_size

        # trust update
        nodes[src].forwarded += 1
        nodes[dst].received += 1

        adaptive_trust_update(nodes[src])
        adaptive_trust_update(nodes[dst])

        update_trust_history(nodes[src])
        update_trust_history(nodes[dst])

        # energy
        for n in nodes:
            n.energy -= random.uniform(0.04, 0.12)

            if n.energy <= 0 and first_dead_round is None:
                first_dead_round = r

        dead = sum(1 for n in nodes if n.energy <= 0)

        if dead >= num_nodes * 0.8:
            break

    avg_delay = total_delay / packets_received if packets_received > 0 else 0

    simulation_time = max(r, 1)

    throughput = (total_data * 8) / (simulation_time * 1000)

    pdr = packets_received / packets_sent if packets_sent > 0 else 0

    lifetime = first_dead_round if first_dead_round else simulation_time

    return avg_delay, packet_loss_bytes, throughput, pdr, lifetime


# ==========================================================================
# Cell 31 -- DRL-EAURP (verbatim)
# ==========================================================================

# Q-learning parameters
Q_table = defaultdict(lambda: [0, 0])

alpha = 0.1
gamma = 0.9
epsilon = 0.1


def get_state(nodes):

    avg_trust = np.mean([n.trust for n in nodes])
    avg_energy = np.mean([n.energy for n in nodes]) / INITIAL_ENERGY
    avg_mobility = np.mean([1 / (1 + n.speed / MAX_SPEED) for n in nodes])

    state = (
        round(avg_trust, 1),
        round(avg_energy, 1),
        round(avg_mobility, 1)
    )

    return state


def choose_action(state):

    if random.random() < epsilon:
        return random.randint(0, 1)

    return np.argmax(Q_table[state])


def update_Q(state, action, reward, next_state):

    best_next = max(Q_table[next_state])

    Q_table[state][action] = Q_table[state][action] + alpha * (
        reward + gamma * best_next - Q_table[state][action]
    )


def run_drl_simulation(num_nodes, speed):

    nodes = create_network(num_nodes)

    for n in nodes:
        n.speed = speed

    update_neighbors(nodes)

    packets_sent = 0
    packets_received = 0
    packet_loss_bytes = 0
    total_delay = 0
    total_data = 0

    first_dead_round = None

    for r in range(SIM_ROUNDS):

        move_network(nodes)
        update_neighbors(nodes)

        src = random.randint(0, num_nodes - 1)
        dst = random.randint(0, num_nodes - 1)

        if src == dst:
            continue

        packets_sent += 1

        packet_size = random.randint(512, 1024)
        hops = random.randint(3, 10)

        # RL state
        state = get_state(nodes)

        action = choose_action(state)

        # Cross-layer routing metrics
        avg_trust = np.mean([n.trust for n in nodes])
        avg_energy = np.mean([n.energy for n in nodes]) / INITIAL_ENERGY
        avg_mobility = np.mean([1 / (1 + n.speed / MAX_SPEED) for n in nodes])

        # Base EAURP routing probability
        base_success_prob = 0.45 + 0.25 * avg_trust + 0.2 * avg_energy + 0.1 * avg_mobility

        # DRL routing adjustment
        if action == 0:
            success_prob = base_success_prob + 0.08   # trusted path
        else:
            success_prob = base_success_prob + 0.02   # exploration path

        success_prob = min(max(success_prob, 0.4), 0.98)

        success = random.random() < success_prob

        if success:

            packets_received += 1

            delay = random.uniform(20, 50) + hops * random.uniform(4, 10)

            total_delay += delay
            total_data += packet_size

            reward = 1

        else:

            packet_loss_bytes += packet_size

            reward = -1

        # Trust updates (EAURP logic)
        nodes[src].forwarded += 1
        nodes[dst].received += 1

        adaptive_trust_update(nodes[src])
        adaptive_trust_update(nodes[dst])

        # Next state for Q-learning
        next_state = get_state(nodes)

        update_Q(state, action, reward, next_state)

        # Energy consumption
        for n in nodes:

            n.energy -= random.uniform(0.04, 0.12)

            if n.energy <= 0 and first_dead_round is None:
                first_dead_round = r

        dead = sum(1 for n in nodes if n.energy <= 0)

        if dead >= num_nodes * 0.8:
            break

    avg_delay = total_delay / packets_received if packets_received > 0 else 0

    simulation_time = max(r, 1)

    throughput = (total_data * 8) / (simulation_time * 1000)

    pdr = packets_received / packets_sent if packets_sent > 0 else 0

    lifetime = first_dead_round if first_dead_round else simulation_time

    return avg_delay, packet_loss_bytes, throughput, pdr, lifetime


# ==========================================================================
# Cell 13 -- the hardcoded "metrics" block, reproduced so the report can show
# exactly what it does. NOT a measurement of anything.
# ==========================================================================

def hardcoded_metrics_block():
    """Cell 13 of the notebook, verbatim. Every value is ``random.uniform``."""
    send_packets = random.randint(225, 230)

    recv_packets = int(send_packets * random.uniform(0.86, 0.90))

    packet_loss_metric = send_packets - recv_packets

    local_pdr_metric = (recv_packets / send_packets) * 100

    routing_overhead = random.uniform(9.5, 10.5)

    local_avg_delay_metric = random.uniform(490, 520)

    local_throughput_metric = random.uniform(16.5, 18)

    local_network_lifetime_metric = random.uniform(99.1, 99.3)

    total_energy = random.uniform(92, 93)

    return {
        "send": send_packets,
        "recv": recv_packets,
        "PacketDeliveryRatio": local_pdr_metric,
        "Routingoverheads": routing_overhead,
        "AverageDelaynsec": local_avg_delay_metric,
        "Packetloss": packet_loss_metric,
        "Throughput_kbps": local_throughput_metric,
        "NetworkLifetime_percentage": local_network_lifetime_metric,
        "TotalEnergyConsumed": total_energy,
        "_provenance": "ALL VALUES ARE random.uniform LITERALS -- NOT MEASURED",
    }


# ==========================================================================
# Driver -- runs the notebook's experiment sequence and writes CSVs
# ==========================================================================

SPEED_RANGE = [10000, 15000, 20000, 25000, 30000, 35000, 40000]
RUNS = 5


def run_all(runs=RUNS, speed_range=None, node_counts=None, seed=42):
    """Reproduce every model in the notebook, in notebook order.

    ``random.seed`` is set once here, exactly as notebook cell 3 does, because
    the senior's published numbers depend on top-to-bottom execution order.
    """
    speed_range = SPEED_RANGE if speed_range is None else speed_range
    node_counts = NODE_COUNTS if node_counts is None else node_counts

    random.seed(seed)

    results = {"speed_range": list(speed_range), "node_counts": list(node_counts)}

    # ---- EAURP base, averaged over RUNS (notebook cell 10) ----
    delay_vs_speed, packetloss_vs_speed = [], []
    throughput_vs_speed, pdr_vs_speed, network_lifetime = [], [], []

    for speed in speed_range:
        delay_avg = loss_avg = thr_avg = pdr_avg = life_avg = 0
        for _ in range(runs):
            d, loss, thr, pdr, life = run_simulation(100, speed)
            delay_avg += d
            loss_avg += loss
            thr_avg += thr
            pdr_avg += pdr
            life_avg += life
        delay_vs_speed.append(delay_avg / runs)
        packetloss_vs_speed.append(loss_avg / runs)
        throughput_vs_speed.append(thr_avg / runs)
        pdr_vs_speed.append(pdr_avg / runs)
        network_lifetime.append(life_avg / runs)

    results["EAURP"] = {
        "delay": delay_vs_speed, "loss": packetloss_vs_speed,
        "throughput": throughput_vs_speed, "pdr": pdr_vs_speed,
        "lifetime": network_lifetime,
    }

    # ---- "Existing": EAURP scaled by constants (Lekha Eq. 14-18) ----
    existing = {"delay": [], "loss": [], "throughput": [], "pdr": [], "lifetime": []}
    for speed in speed_range:
        d, loss, thr, pdr, life = run_simulation(100, speed)
        existing["delay"].append(d * 1.2)
        existing["loss"].append(loss * 1.5)
        existing["throughput"].append(thr * 0.7)
        existing["pdr"].append(pdr * 0.85)
        existing["lifetime"].append(life * 0.98)
    existing["_provenance"] = PROVENANCE["existing_baseline"]
    results["Existing"] = existing

    # ---- node-count sweep (notebook cell 10) ----
    delay_vs_nodes, packetloss_vs_nodes, data_reliability = [], [], []
    for nodes_count in node_counts:
        delay_avg = loss_avg = pdr_avg = 0
        for _ in range(runs):
            d, loss, thr, pdr, life = run_simulation(nodes_count, 20000)
            delay_avg += d
            loss_avg += loss
            pdr_avg += pdr
        delay_vs_nodes.append(delay_avg / runs)
        packetloss_vs_nodes.append(loss_avg / runs)
        data_reliability.append(pdr_avg / runs)
    results["EAURP_vs_nodes"] = {
        "delay": delay_vs_nodes, "loss": packetloss_vs_nodes,
        "reliability": data_reliability,
    }

    # ---- ATEAURP / PSE-EAURP / DRL-EAURP (single run each, as in notebook) ----
    for label, runner in (
        ("ATEAURP", run_proposed_simulation),
        ("PSE-EAURP", run_predictive_simulation),
        ("DRL-EAURP", run_drl_simulation),
    ):
        bucket = {"delay": [], "loss": [], "throughput": [], "pdr": [], "lifetime": []}
        for speed in speed_range:
            d, loss, thr, pdr, life = runner(100, speed)
            bucket["delay"].append(d)
            bucket["loss"].append(loss)
            bucket["throughput"].append(thr)
            bucket["pdr"].append(pdr)
            bucket["lifetime"].append(life)
        results[label] = bucket

    results["hardcoded_cell_13"] = hardcoded_metrics_block()
    results["provenance"] = PROVENANCE
    return results


def to_rows(results):
    """Flatten :func:`run_all` output into CSV rows."""
    rows = []
    speeds = results["speed_range"]
    for model in ("Existing", "EAURP", "ATEAURP", "PSE-EAURP", "DRL-EAURP"):
        block = results.get(model)
        if not block:
            continue
        for index, speed in enumerate(speeds):
            rows.append({
                "track": "track1_as_is",
                "implementation": "B_senior_code",
                "model": model,
                "speed": speed,
                "avg_delay_ms": block["delay"][index],
                "packet_loss_bytes": block["loss"][index],
                "throughput": block["throughput"][index],
                "pdr": block["pdr"][index],
                "network_lifetime": block["lifetime"][index],
            })
    return rows


def write_csv(rows, path):
    directory = os.path.dirname(os.path.abspath(path))
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    return path


def main(out_dir="results/csv", runs=RUNS):
    results = run_all(runs=runs)
    rows = to_rows(results)
    path = write_csv(rows, os.path.join(out_dir, "b_senior_as_is.csv"))
    print("B (senior's code, as-is) ->", path)
    for model in ("Existing", "EAURP", "ATEAURP", "PSE-EAURP", "DRL-EAURP"):
        block = results[model]
        print("  {0:11s} PDR {1:.3f}-{2:.3f}  delay {3:.1f}-{4:.1f} ms  "
              "lifetime {5:.0f}".format(
                  model, min(block["pdr"]), max(block["pdr"]),
                  min(block["delay"]), max(block["delay"]),
                  np.mean(block["lifetime"])))
    return results


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/b_senior/policy_common.py
"""Implementation B on the mechanistic harness (Track 2).

Same Q-learning agent as A -- identical alpha, gamma, epsilon, identical
<T, E, M> state discretisation, identical two actions and +/-1 reward -- because
both come from the same equations.

The one difference is faithful to the delivered code, and it is the whole point
of running B on Track 2:

    A (paper, Eq. 19)  trust[i][j] = what j relayed out of what i gave it,
                       measured by the watchdog.

    B (delivered code) nodes[src].forwarded += 1
                       nodes[dst].received  += 1
                       ...where src and dst are drawn at random and are not
                       relays of anything.

On the coin-flip model that difference is invisible, because trust only feeds a
scalar in a probability formula. On a harness where trust actually selects
routes, it is the difference between routing on evidence and routing on noise.
Reproducing it here is what makes that visible.
"""

import numpy as np

from ..a_drl_eaurp.policy_common import TabularDrlPolicy


class SeniorDrlPolicy(TabularDrlPolicy):
    """The senior's DRL-EAURP, including its trust-attribution semantics."""

    name = "B:DRL-EAURP(senior)"
    trust_source = "endpoint_counters"   # nodes[src].forwarded / nodes[dst].received

    def __init__(self, **kwargs):
        super(SeniorDrlPolicy, self).__init__(**kwargs)
        # The code's own counters, kept separately from the harness watchdog so
        # that both views can be reported side by side.
        self._forwarded = None
        self._received = None
        self._trust = None

    def reset(self, sim):
        n = sim.nodes.n
        self._forwarded = np.zeros(n)
        self._received = np.zeros(n)
        self._trust = np.ones(n)
        super(SeniorDrlPolicy, self).reset(sim)

    def trust_vector(self, sim):
        """Trust as the delivered code computes it."""
        if self._trust is None:
            return np.ones(sim.nodes.n)
        return np.clip(self._trust, 0.0, 1.0)

    def state_of(self, sim):
        """Eq. (29)-(32) over the code's own trust values."""
        nodes = sim.nodes
        alive = nodes.alive
        if not alive.any() or self._trust is None:
            return (0.0, 0.0, 0.0)
        trust = float(np.clip(self._trust, 0.0, 1.0)[alive].mean())
        energy = float(nodes.normalised_energy()[alive].mean())
        mobility = float(nodes.mobility_factor()[alive].mean())
        return (round(trust, 1), round(energy, 1), round(mobility, 1))

    def on_result(self, sim, packet, delivered, path, hops, reason):
        # Verbatim from senior_code.ipynb:
        #     nodes[src].forwarded += 1
        #     nodes[dst].received  += 1
        #     adaptive_trust_update(nodes[src]); adaptive_trust_update(nodes[dst])
        self._forwarded[packet.src] += 1.0
        self._received[packet.dst] += 1.0
        for node_id in (packet.src, packet.dst):
            self._adaptive_trust_update(node_id)

        super(SeniorDrlPolicy, self).on_result(
            sim, packet, delivered, path, hops, reason
        )

    def _adaptive_trust_update(self, node_id):
        """``adaptive_trust_update`` from the notebook, unchanged."""
        received = self._received[node_id]
        if received == 0:
            return
        pfr = self._forwarded[node_id] / received
        self._trust[node_id] = 0.7 * self._trust[node_id] + 0.3 * pfr

    def detected_mask(self, sim):
        """The delivered code never acts on ``node.malicious``.

        ``adaptive_trust_update`` sets the flag and nothing reads it, so B
        detects nobody. Reporting an all-false mask states that honestly rather
        than borrowing the harness's revocation list.
        """
        return np.zeros(sim.nodes.n, dtype=bool)

    def extra_metrics(self, sim):
        row = super(SeniorDrlPolicy, self).extra_metrics(sim)
        if self._trust is not None:
            watchdog = sim.nodes.network_trust()
            row["code_trust_mean"] = float(np.clip(self._trust, 0.0, 1.0).mean())
            row["watchdog_trust_mean"] = float(watchdog.mean())
            # Correlation between the code's trust signal and a real
            # forwarding-behaviour measurement. Near zero means the signal
            # carries no information about who actually forwards.
            code = np.clip(self._trust, 0.0, 1.0)
            if code.std() > 1e-9 and watchdog.std() > 1e-9:
                row["trust_signal_correlation"] = float(
                    np.corrcoef(code, watchdog)[0, 1]
                )
            else:
                row["trust_signal_correlation"] = 0.0
        return row


In [ ]:
%%writefile src/c_ar_eaurp/__init__.py
"""Implementation C -- AR-EAURP, the advancement (Date_ 24_07_26.docx).

Adversarial-Resilient and Energy-Harvesting Aware EAURP, built on top of the
senior's DRL-EAURP. The five steps of the roadmap map onto five modules:

    1. Advanced threat model          -> ``threat_model`` (+ ``common.adversary``)
    2. GAN-based anomaly detection    -> ``gan_detector``
    3. LSTM predictive harvesting     -> ``lstm_energy``
    4. Constrained MDP for security   -> ``cmdp_agent``
    5. Simulation and attack injection-> ``protocol`` + ``experiments/e3_attack``

Every learned component has a documented, non-learned fallback that doubles as
the baseline it must beat: the GAN falls back to a percentile threshold, the
LSTM to persistence, and the DQN to the tabular Q-learner of the base paper.
Those fallbacks are what make claims like "the LSTM helps" checkable instead of
assumed, and they let the whole pipeline run even where torch is unavailable.
"""


In [ ]:
%%writefile src/c_ar_eaurp/threat_model.py
"""Behaviour features for anomaly detection -- AR-EAURP step 1/2.

The adversary behaviours themselves live in ``common.adversary`` (they have to,
because the harness applies them during forwarding). What lives here is the
observation side: turning the watchdog counters into the feature vectors the
GAN discriminator sees.

THE TWO FEATURES THAT MATTER -- these are the technical core of C.

``drop_rate_large`` vs ``drop_rate_small``
    A single scalar Packet Forwarding Ratio cannot tell "drops 15% of
    everything" apart from "drops 25% of the big packets and nothing else". The
    gray-hole of the threat model is the second thing, and measured on this
    harness its scalar PFR sits at 0.84 -- comfortably above the 0.6 revocation
    threshold of base.pdf, hence invisible. Split the same observations by
    packet size and the gap is 0.25, which is enormous. Selective forwarding is
    only stealthy if you refuse to look at what is being selected.

``report_disagreement``
    How far one observer's published opinion of a node sits from the median
    opinion of every other observer. An honest node reports what it measured, so
    its disagreement is small. A slanderer publishes 1.0 for its colluders and
    0.15 for the most trusted honest nodes, so its disagreement is large -- and
    it is large *whichever* direction it lies in, which is what makes the
    feature catch both vouching and accusing.

Everything is computed as a delta against the previous detection pass, so the
features describe recent behaviour rather than an ever-flattening lifetime
average.
"""

import numpy as np

from ..common import adversary, config

FEATURE_NAMES = (
    "pfr_short",
    "pfr_long",
    "pfr_delta",
    "drop_rate_large",
    "drop_rate_small",
    "fwd_latency_mean",
    "fwd_latency_std",
    "report_disagreement",
)
assert len(FEATURE_NAMES) == config.GAN_FEATURE_DIM


def _safe_ratio(numerator, denominator, default=1.0):
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(denominator > 0.0, numerator / np.maximum(denominator, 1e-9),
                        default)


class BehaviourFeatureExtractor(object):
    """Turns watchdog counters into per-(observer, subject) feature vectors."""

    def __init__(self, n_nodes, min_observations=None):
        self.n = int(n_nodes)
        self.min_observations = (
            config.MIN_OBSERVATIONS if min_observations is None
            else int(min_observations)
        )
        shape = (self.n, self.n)
        self._prev_sent = np.zeros(shape)
        self._prev_fwd = np.zeros(shape)
        self._prev_sent_large = np.zeros(shape)
        self._prev_fwd_large = np.zeros(shape)
        self._prev_sent_small = np.zeros(shape)
        self._prev_fwd_small = np.zeros(shape)

    def snapshot(self, nodes):
        """Record the current counters as the baseline for the next window."""
        self._prev_sent = nodes.sent_to.copy()
        self._prev_fwd = nodes.fwd_seen.copy()
        self._prev_sent_large = nodes.sent_large.copy()
        self._prev_fwd_large = nodes.fwd_seen_large.copy()
        self._prev_sent_small = nodes.sent_small.copy()
        self._prev_fwd_small = nodes.fwd_seen_small.copy()

    def report_disagreement(self, nodes):
        """|this observer's opinion - median opinion of the others|, per pair."""
        observed = nodes.sent_to >= self.min_observations
        reported = np.array(nodes.trust, dtype=float)

        # Apply each observer's *published* value, which for a slanderer is a lie.
        slanderers = np.flatnonzero(
            (nodes.role & 4) != 0  # FLAG_SLANDERER
        )
        for observer in slanderers:
            for subject in np.flatnonzero(observed[observer]):
                reported[observer, subject] = adversary.reported_trust(
                    nodes, int(observer), int(subject),
                    float(nodes.trust[observer, subject]),
                )

        disagreement = np.zeros_like(reported)
        for subject in range(self.n):
            observers = np.flatnonzero(observed[:, subject])
            if observers.size < 2:
                continue
            opinions = reported[observers, subject]
            median = float(np.median(opinions))
            disagreement[observers, subject] = np.abs(opinions - median)
        return disagreement

    def extract(self, nodes, update_snapshot=True):
        """Return ``(features, observer_ids, subject_ids)`` for watched pairs.

        ``features`` has shape ``(n_pairs, GAN_FEATURE_DIM)``.
        """
        d_sent = nodes.sent_to - self._prev_sent
        d_fwd = nodes.fwd_seen - self._prev_fwd
        d_sent_large = nodes.sent_large - self._prev_sent_large
        d_fwd_large = nodes.fwd_seen_large - self._prev_fwd_large
        d_sent_small = nodes.sent_small - self._prev_sent_small
        d_fwd_small = nodes.fwd_seen_small - self._prev_fwd_small

        pfr_short = _safe_ratio(d_fwd, d_sent, default=1.0)
        pfr_long = _safe_ratio(nodes.fwd_seen, nodes.sent_to, default=1.0)
        drop_large = 1.0 - _safe_ratio(d_fwd_large, d_sent_large, default=1.0)
        drop_small = 1.0 - _safe_ratio(d_fwd_small, d_sent_small, default=1.0)

        latency_mean = _safe_ratio(nodes.lat_sum, nodes.lat_n, default=0.0)
        second_moment = _safe_ratio(nodes.lat_sq, nodes.lat_n, default=0.0)
        latency_var = np.clip(second_moment - latency_mean ** 2, 0.0, None)
        latency_std = np.sqrt(latency_var)

        disagreement = self.report_disagreement(nodes)

        # A pair qualifies once it has enough total observations and has been
        # seen at least once in the current window.
        eligible = (nodes.sent_to >= self.min_observations) & (d_sent > 0.0)
        observers, subjects = np.nonzero(eligible)

        if observers.size == 0:
            if update_snapshot:
                self.snapshot(nodes)
            return (
                np.zeros((0, config.GAN_FEATURE_DIM)),
                observers.astype(np.int64),
                subjects.astype(np.int64),
            )

        features = np.column_stack([
            pfr_short[observers, subjects],
            pfr_long[observers, subjects],
            pfr_short[observers, subjects] - pfr_long[observers, subjects],
            drop_large[observers, subjects],
            drop_small[observers, subjects],
            # Latency is scaled into roughly [0, 1] so no single feature
            # dominates the discriminator's input purely by magnitude.
            latency_mean[observers, subjects] / max(1.0, config.HOP_DELAY_MAX),
            latency_std[observers, subjects] / max(1.0, config.HOP_DELAY_MAX),
            disagreement[observers, subjects],
        ]).astype(np.float64)

        features = np.nan_to_num(features, nan=0.0, posinf=1.0, neginf=0.0)
        features = np.clip(features, -1.0, 2.0)

        if update_snapshot:
            self.snapshot(nodes)
        return features, observers.astype(np.int64), subjects.astype(np.int64)


def aggregate_by_subject(scores, subjects, n_nodes, reduce="mean"):
    """Collapse per-pair anomaly scores into one score per node.

    Mean rather than max: a single hostile observer should not be able to get an
    honest node flagged on its own, which is precisely the slander attack.
    """
    totals = np.zeros(n_nodes)
    counts = np.zeros(n_nodes)
    np.add.at(totals, subjects, scores)
    np.add.at(counts, subjects, 1.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        if reduce == "max":
            out = np.zeros(n_nodes)
            np.maximum.at(out, subjects, scores)
            return out
        return np.where(counts > 0, totals / np.maximum(counts, 1.0), 0.0)


def scenario_description(attack, malicious_fraction):
    """Human-readable label for a threat configuration."""
    return {
        "attack": attack,
        "malicious_fraction": float(malicious_fraction),
        "label": adversary.describe(attack, malicious_fraction),
        "behaviour": {
            "blackhole": "advertises a perfect route, drops 100% of data",
            "grayhole": "relays control and small packets, drops {0:.0%} of "
                        "packets >= {1} bytes".format(
                            config.GRAYHOLE_LARGE_DROP_P,
                            config.LARGE_PACKET_THRESHOLD),
            "trust_poisoning": "gray-hole that also publishes false PT_GID "
                               "reports (1.0 for colluders, 0.15 for the most "
                               "trusted honest nodes)",
            "none": "no adversary",
        }.get(str(attack).lower(), "unknown"),
    }


In [ ]:
%%writefile src/c_ar_eaurp/gan_detector.py
"""GAN-based anomaly detection -- AR-EAURP step 2.

The doc asks for a GAN that "learns the distribution of legitimate node
behaviour"; if an observed forwarding pattern deviates from it, the node is
flagged "contested" and its trust weight halved.

DESIGN NOTES WORTH DEFENDING AT THE REVIEW
------------------------------------------
1. **Trained on honest behaviour only.** The discriminator never sees an attack
   during training. Detection is therefore one-class: we model what normal looks
   like and flag what does not fit. That matters because it means the detector
   is not tuned to the specific attacks we then evaluate it on.

2. **A discriminator alone is a poor anomaly detector.** D is trained to
   separate *real from generated*, not *normal from abnormal*; once G is good, D
   is near chance everywhere. So we use the AnoGAN-style combination: a
   reconstruction term (how far is this sample from anything the generator can
   produce) plus the discriminator term. Reconstruction dominates by default.
   The reconstruction search is a nearest-neighbour lookup against a fixed bank
   of generated samples rather than per-sample latent optimisation -- same
   signal, a few microseconds instead of a few seconds, which matters when it
   runs inside a simulation loop.

3. **Mode collapse is assumed, not hoped against.** Label smoothing and a
   held-out clean validation split are used, and if validation AUC comes out
   below 0.6 the detector says so loudly and falls back to a percentile
   threshold on the size-conditioned drop rate. The fallback is reported
   whenever it triggers -- it is never allowed to masquerade as the GAN working.

4. **The fallback is also the baseline.** Reporting GAN-vs-fallback is what
   makes "the GAN helped" a measured claim rather than an assumed one.
"""

import numpy as np

from ..common import config
from .threat_model import FEATURE_NAMES

try:  # pragma: no cover - depends on the runtime environment
    import torch
    import torch.nn as nn

    TORCH_AVAILABLE = True
except Exception:  # pragma: no cover
    TORCH_AVAILABLE = False

# Weight on the reconstruction term in the AnoGAN-style score.
#
# Set to 1.0 on evidence, not by preference. Measured on held-out attacked
# traffic, with the feature scaling fixed:
#
#     reconstruction residual   AUC 0.969
#     discriminator             AUC 0.056   <- anti-correlated
#     combined at 0.9/0.1       AUC 0.969
#
# The discriminator is not merely weak, it is backwards: it assigns a *higher*
# "this is real" probability to attacked behaviour than to honest behaviour.
# That is not a bug in the training loop, it is what a discriminator is for --
# it separates real from generated, never normal from abnormal, and attacked
# samples are real. Giving it any weight can only drag the score down.
#
# The generator still does the work, which is what makes this a GAN-based
# detector in the AnoGAN sense: G learns the manifold of legitimate behaviour
# and the residual distance to that manifold is the anomaly score. D exists to
# train G. Its own score is still computed and reported in `summary()` so this
# claim stays auditable rather than asserted.
RECONSTRUCTION_WEIGHT = 1.0
GENERATED_BANK_SIZE = 512
MIN_TRAIN_SAMPLES = 32

# Floor under the per-feature standard deviation. See _fit_scaler for why this
# exists and why setting it to 1.0 (the usual guard) silently breaks detection.
MIN_FEATURE_SCALE = 0.02

# A detector at AUC 0.61 is noise wearing a lab coat. The gate used to sit at
# 0.60, which let exactly that through: a GAN scoring 0.614 skipped the fallback
# and dropped gray-hole recall from 0.38 to 0.03. If the learned model cannot
# clear this bar it must defer to the simple percentile detector, loudly.
MIN_ACCEPTABLE_AUC = 0.70


def roc_auc(scores, labels):
    """Mann-Whitney AUC. Implemented directly so sklearn is not required."""
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels).astype(bool)
    positives = int(labels.sum())
    negatives = int((~labels).sum())
    if positives == 0 or negatives == 0:
        return float("nan")
    order = np.argsort(scores, kind="mergesort")
    ranks = np.empty(len(scores), dtype=float)
    ranks[order] = np.arange(1, len(scores) + 1, dtype=float)
    # Average ranks within ties.
    sorted_scores = scores[order]
    start = 0
    while start < len(sorted_scores):
        stop = start
        while stop + 1 < len(sorted_scores) and sorted_scores[stop + 1] == sorted_scores[start]:
            stop += 1
        if stop > start:
            ranks[order[start:stop + 1]] = np.mean(
                np.arange(start + 1, stop + 2, dtype=float)
            )
        start = stop + 1
    rank_sum = ranks[labels].sum()
    return float((rank_sum - positives * (positives + 1) / 2.0) / (positives * negatives))


if TORCH_AVAILABLE:

    class _Generator(nn.Module):
        def __init__(self, latent_dim, hidden, feature_dim):
            super(_Generator, self).__init__()
            self.net = nn.Sequential(
                nn.Linear(latent_dim, hidden),
                nn.LeakyReLU(0.2),
                nn.Linear(hidden, hidden),
                nn.LeakyReLU(0.2),
                nn.Linear(hidden, feature_dim),
            )

        def forward(self, z):
            return self.net(z)

    class _Discriminator(nn.Module):
        def __init__(self, feature_dim, hidden):
            super(_Discriminator, self).__init__()
            self.body = nn.Sequential(
                nn.Linear(feature_dim, hidden),
                nn.LeakyReLU(0.2),
                nn.Linear(hidden, hidden),
                nn.LeakyReLU(0.2),
            )
            self.head = nn.Linear(hidden, 1)

        def forward(self, x, return_features=False):
            features = self.body(x)
            logits = self.head(features)
            if return_features:
                return logits, features
            return logits


class GanAnomalyDetector(object):
    """One-class behaviour model over the features in ``threat_model``."""

    def __init__(self, feature_dim=None, latent_dim=None, hidden=None,
                 lr=None, epochs=None, batch_size=None, seed=0):
        self.feature_dim = config.GAN_FEATURE_DIM if feature_dim is None else int(feature_dim)
        self.latent_dim = config.GAN_LATENT_DIM if latent_dim is None else int(latent_dim)
        self.hidden = config.GAN_HIDDEN if hidden is None else int(hidden)
        self.lr = config.GAN_LR if lr is None else float(lr)
        self.epochs = config.GAN_EPOCHS if epochs is None else int(epochs)
        self.batch_size = config.GAN_BATCH if batch_size is None else int(batch_size)
        self.seed = int(seed)

        self.backend = "torch" if TORCH_AVAILABLE else "percentile-fallback"
        self.using_fallback = not TORCH_AVAILABLE
        self.fallback_reason = None if TORCH_AVAILABLE else "torch unavailable"

        self.generator = None
        self.discriminator = None
        self._bank = None
        self._mean = None
        self._std = None
        self.threshold = 0.5
        self.history = {"d_loss": [], "g_loss": []}
        self.validation = {}
        self.trained = False

    # -- normalisation -----------------------------------------------------

    def _fit_scaler(self, features):
        """Standardise, with a *small* floor under the standard deviation.

        This is the single most important line in the detector, and getting it
        wrong silently destroys the signal. Several features have exactly zero
        variance across honest behaviour -- an honest relay never drops a large
        packet, and never disagrees with its peers about a neighbour. Those are
        precisely the informative ones.

        The obvious guard, ``std[std == 0] = 1.0``, is wrong here: it says
        "this feature varies by 1.0 among honest nodes", so a gray-hole's 0.20
        drop rate becomes a z-score of 0.20 and is drowned out by nuisance
        features like forwarding latency, whose honest variance really is ~0.16.
        Measured, that mistake put discriminator AUC at 0.46 -- worse than
        chance -- and gray-hole recall at 0.03.

        A small floor encodes the opposite, correct intuition: if honest nodes
        never vary on a feature, then *any* deviation on it is strongly
        anomalous, so the denominator should be small rather than large.
        """
        self._mean = features.mean(axis=0)
        self._std = features.std(axis=0)
        self._std = np.maximum(self._std, MIN_FEATURE_SCALE)

    def _scale(self, features):
        if self._mean is None:
            return np.asarray(features, dtype=np.float32)
        return ((np.asarray(features, dtype=float) - self._mean) / self._std).astype(
            np.float32
        )

    # -- training ----------------------------------------------------------

    def fit(self, clean_features, validation_features=None,
            validation_labels=None, epochs=None):
        """Train on honest-only behaviour windows.

        ``validation_features``/``validation_labels`` are optional and used only
        to measure quality -- never to fit anything.
        """
        clean = np.asarray(clean_features, dtype=float)
        clean = clean.reshape(-1, self.feature_dim) if clean.size else clean

        if clean.shape[0] < MIN_TRAIN_SAMPLES:
            self.using_fallback = True
            self.fallback_reason = (
                "only {0} clean training windows (need {1})".format(
                    clean.shape[0], MIN_TRAIN_SAMPLES)
            )
            self._fit_fallback(clean)
            return self.summary()

        self._fit_scaler(clean)

        if not TORCH_AVAILABLE:
            self._fit_fallback(clean)
            return self.summary()

        epochs = self.epochs if epochs is None else int(epochs)
        torch.manual_seed(self.seed)

        self.generator = _Generator(self.latent_dim, self.hidden, self.feature_dim)
        self.discriminator = _Discriminator(self.feature_dim, self.hidden)
        opt_g = torch.optim.Adam(self.generator.parameters(), lr=self.lr,
                                 betas=(0.5, 0.999))
        opt_d = torch.optim.Adam(self.discriminator.parameters(), lr=self.lr,
                                 betas=(0.5, 0.999))
        criterion = torch.nn.BCEWithLogitsLoss()

        data = torch.from_numpy(self._scale(clean))
        n = data.shape[0]
        batch = min(self.batch_size, n)
        self.history = {"d_loss": [], "g_loss": []}

        for _ in range(epochs):
            index = torch.randperm(n)[:batch]
            real = data[index]

            # --- discriminator ---
            z = torch.randn(batch, self.latent_dim)
            fake = self.generator(z).detach()
            d_real = self.discriminator(real)
            d_fake = self.discriminator(fake)
            # One-sided label smoothing: real targets 0.9, not 1.0.
            loss_d = criterion(
                d_real, torch.full_like(d_real, config.GAN_LABEL_SMOOTH)
            ) + criterion(d_fake, torch.zeros_like(d_fake))
            opt_d.zero_grad()
            loss_d.backward()
            opt_d.step()

            # --- generator, with feature matching to discourage collapse ---
            z = torch.randn(batch, self.latent_dim)
            generated = self.generator(z)
            g_logits, g_features = self.discriminator(generated, return_features=True)
            _, real_features = self.discriminator(real, return_features=True)
            adversarial = criterion(g_logits, torch.ones_like(g_logits))
            feature_match = torch.mean(
                (g_features.mean(dim=0) - real_features.mean(dim=0).detach()) ** 2
            )
            loss_g = adversarial + feature_match
            opt_g.zero_grad()
            loss_g.backward()
            opt_g.step()

            self.history["d_loss"].append(float(loss_d.item()))
            self.history["g_loss"].append(float(loss_g.item()))

        # Fixed bank of generated samples for the reconstruction term.
        with torch.no_grad():
            z = torch.randn(GENERATED_BANK_SIZE, self.latent_dim)
            self._bank = self.generator(z).numpy()

        self.trained = True
        self.threshold = float(
            np.percentile(self.score(clean), config.GAN_THRESHOLD_PERCENTILE)
        )

        self._validate(clean, validation_features, validation_labels)
        return self.summary()

    def _validate(self, clean, validation_features, validation_labels):
        """Measure quality and trip the fallback if the GAN did not learn."""
        if validation_features is None or validation_labels is None:
            return
        features = np.asarray(validation_features, dtype=float)
        labels = np.asarray(validation_labels).astype(bool)
        if features.size == 0 or labels.sum() == 0 or (~labels).sum() == 0:
            return

        auc = roc_auc(self.score(features), labels)
        self.validation = {"auc": auc, "n": int(features.shape[0]),
                           "n_anomalous": int(labels.sum())}

        if not np.isnan(auc) and auc < MIN_ACCEPTABLE_AUC:
            self.using_fallback = True
            self.fallback_reason = (
                "GAN validation AUC {0:.3f} < {1} -- likely mode collapse; "
                "falling back to a percentile threshold on the "
                "size-conditioned drop rate".format(auc, MIN_ACCEPTABLE_AUC)
            )
            print("[gan_detector] WARNING: " + self.fallback_reason)
            self._fit_fallback(clean)

    def _fit_fallback(self, clean):
        """Percentile threshold on ``drop_rate_large`` (feature index 3)."""
        self.trained = True
        self.backend = "percentile-fallback"
        if clean.size:
            column = clean.reshape(-1, self.feature_dim)[:, 3]
            self._fallback_threshold = float(
                np.percentile(column, config.GAN_THRESHOLD_PERCENTILE)
            )
        else:
            self._fallback_threshold = 0.1
        self.threshold = self._fallback_threshold

    # -- scoring -----------------------------------------------------------

    def score(self, features):
        """Anomaly score per row; higher means less like honest behaviour."""
        features = np.asarray(features, dtype=float)
        if features.size == 0:
            return np.zeros(0)
        features = features.reshape(-1, self.feature_dim)

        if self.using_fallback or not TORCH_AVAILABLE or self.generator is None:
            # The size-conditioned drop rate, directly.
            return np.clip(features[:, 3], 0.0, 1.0)

        scaled = self._scale(features)
        with torch.no_grad():
            tensor = torch.from_numpy(np.ascontiguousarray(scaled))
            logits = self.discriminator(tensor).numpy().ravel()
        # 1 - sigmoid(logits), written so a large negative logit cannot overflow.
        discriminator_score = 1.0 / (1.0 + np.exp(np.clip(logits, -60.0, 60.0)))

        # Reconstruction: distance to the closest sample the generator can make.
        # Computed in chunks -- the naive (n, bank, dim) broadcast allocates
        # n * 512 * 8 floats at once, which is ~80 MB for a few thousand
        # observation windows and is the kind of thing that quietly kills a
        # Colab kernel mid-sweep.
        bank = self._bank
        distances = np.empty(scaled.shape[0], dtype=np.float64)
        chunk = 256
        for start in range(0, scaled.shape[0], chunk):
            block = scaled[start:start + chunk]
            diff = block[:, None, :] - bank[None, :, :]
            distances[start:start + chunk] = np.sqrt(
                (diff * diff).sum(axis=-1)
            ).min(axis=1)
        # Squash into [0, 1) so the two terms are commensurate.
        reconstruction = distances / (1.0 + distances)

        # Kept for reporting even at zero weight, so the claim in the comment on
        # RECONSTRUCTION_WEIGHT can be re-checked from any run.
        self.last_discriminator_score = discriminator_score
        self.last_reconstruction_score = reconstruction

        return (
            RECONSTRUCTION_WEIGHT * reconstruction
            + (1.0 - RECONSTRUCTION_WEIGHT) * discriminator_score
        )

    def flag(self, features):
        """Boolean "contested" verdict per row."""
        scores = self.score(features)
        return scores > self.threshold, scores

    def set_threshold_from_clean(self, clean_features, percentile=None):
        percentile = (
            config.GAN_THRESHOLD_PERCENTILE if percentile is None else float(percentile)
        )
        scores = self.score(clean_features)
        if scores.size:
            self.threshold = float(np.percentile(scores, percentile))
        return self.threshold

    # -- reporting ---------------------------------------------------------

    def summary(self):
        return {
            "gan_backend": self.backend,
            "gan_using_fallback": bool(self.using_fallback),
            "gan_fallback_reason": self.fallback_reason,
            "gan_threshold": float(self.threshold),
            "gan_epochs_run": len(self.history.get("d_loss", [])),
            "gan_final_d_loss": (
                float(self.history["d_loss"][-1]) if self.history.get("d_loss") else None
            ),
            "gan_final_g_loss": (
                float(self.history["g_loss"][-1]) if self.history.get("g_loss") else None
            ),
            "gan_val_auc": self.validation.get("auc"),
            "feature_names": list(FEATURE_NAMES),
        }


In [ ]:
%%writefile src/c_ar_eaurp/lstm_energy.py
"""LSTM predictive energy harvesting -- AR-EAURP step 3.

Replaces the linear depletion of Lekha.pdf Eq. (8) with a solar/kinetic model,
and forecasts the energy each node will harvest over the next
``LSTM_HORIZON`` rounds. That forecast becomes a new DRL state feature and a
route-scoring term, so the agent can prefer relays that are *about to be*
energy-rich rather than only those that are rich now.

WHY THE BASELINES ARE NOT OPTIONAL
----------------------------------
An LSTM that beats nothing has proved nothing. The harvest signal here is a
diurnal sinusoid times an AR(1) cloud process, and a sinusoid is extremely
predictable -- a "same time yesterday" lookup already does well on it. So every
run reports three numbers:

    persistence     last observed harvest, held for the horizon
    seasonal naive  the same window exactly one day earlier
    LSTM            the learned model

If the LSTM does not beat both, the report says so. This is exactly the concern
raised in the Review-1 critique ("LSTM prediction is only as good as the traces
it is trained on"), so it is measured rather than asserted.
"""

import numpy as np

from ..common import config

try:  # pragma: no cover - depends on the runtime environment
    import torch
    import torch.nn as nn

    TORCH_AVAILABLE = True
except Exception:  # pragma: no cover
    TORCH_AVAILABLE = False

N_FEATURES = 4  # harvest, normalised energy, sin(phase), cos(phase)


def generate_harvest_traces(n_nodes, n_rounds, rng, day_length=None,
                            panel_fraction=None):
    """Simulate the solar/kinetic harvest process on its own.

    Used to build the LSTM's training set without running a full network
    simulation. Uses the same generative process as
    ``common.energy.SolarHarvesting`` so the model trains on the signal it will
    actually see.
    """
    day_length = config.HARVEST_DAY_LENGTH if day_length is None else float(day_length)
    panel_fraction = (
        config.HARVEST_PANEL_FRACTION if panel_fraction is None else float(panel_fraction)
    )

    has_panel = rng.random(n_nodes) < panel_fraction
    gains = rng.uniform(config.HARVEST_PEAK_MIN, config.HARVEST_PEAK_MAX, n_nodes)
    panel_gain = np.where(has_panel, gains, 0.0)

    cloud = np.ones(n_nodes)
    harvest = np.zeros((n_rounds, n_nodes))
    energy = np.full(n_nodes, config.INITIAL_ENERGY)
    energy_trace = np.zeros((n_rounds, n_nodes))

    for step in range(n_rounds):
        noise = rng.normal(0.0, config.HARVEST_CLOUD_SIGMA, n_nodes)
        cloud = (
            config.HARVEST_CLOUD_RHO * cloud
            + (1.0 - config.HARVEST_CLOUD_RHO) * 1.0
            + noise * (1.0 - config.HARVEST_CLOUD_RHO)
        )
        cloud = np.clip(cloud, 0.0, 1.5)

        irradiance = max(0.0, float(np.sin(2.0 * np.pi * step / day_length)))
        gain = np.clip(panel_gain * irradiance * cloud, 0.0, None)
        harvest[step] = gain

        drain = rng.uniform(config.IDLE_DRAIN_MIN, config.IDLE_DRAIN_MAX, n_nodes)
        energy = np.clip(energy + gain - drain, 0.0, config.MAX_ENERGY_CAP)
        energy_trace[step] = energy

    return harvest, energy_trace, panel_gain


def phase_features(rounds, day_length=None):
    """Time-of-day encoding, as sin/cos so midnight is not a discontinuity."""
    day_length = config.HARVEST_DAY_LENGTH if day_length is None else float(day_length)
    phase = 2.0 * np.pi * (np.asarray(rounds, dtype=float) / day_length)
    return np.sin(phase), np.cos(phase)


def build_dataset(harvest, energy_trace, input_len=None, horizon=None,
                  day_length=None):
    """Windows of past behaviour -> total harvest over the next ``horizon``."""
    input_len = config.LSTM_INPUT_LEN if input_len is None else int(input_len)
    horizon = config.LSTM_HORIZON if horizon is None else int(horizon)

    n_rounds, n_nodes = harvest.shape
    sin_phase, cos_phase = phase_features(np.arange(n_rounds), day_length)

    inputs, targets = [], []
    last_values = []
    seasonal = []
    day_length = config.HARVEST_DAY_LENGTH if day_length is None else float(day_length)
    day_steps = int(round(day_length))

    for start in range(n_rounds - input_len - horizon):
        stop = start + input_len
        window_harvest = harvest[start:stop, :]
        window_energy = energy_trace[start:stop, :] / config.MAX_ENERGY_CAP
        target = harvest[stop:stop + horizon, :].sum(axis=0)

        window = np.stack(
            [
                window_harvest,
                window_energy,
                np.repeat(sin_phase[start:stop, None], n_nodes, axis=1),
                np.repeat(cos_phase[start:stop, None], n_nodes, axis=1),
            ],
            axis=-1,
        )  # (input_len, n_nodes, N_FEATURES)

        inputs.append(np.transpose(window, (1, 0, 2)))  # (n_nodes, input_len, F)
        targets.append(target)
        last_values.append(window_harvest[-1, :] * horizon)

        # "Same window, one day ago" -- the seasonal-naive baseline.
        prior = stop - day_steps
        if prior >= 0:
            seasonal.append(harvest[prior:prior + horizon, :].sum(axis=0))
        else:
            seasonal.append(window_harvest[-1, :] * horizon)

    if not inputs:
        empty = np.zeros((0, input_len, N_FEATURES))
        return empty, np.zeros(0), np.zeros(0), np.zeros(0)

    X = np.concatenate(inputs, axis=0)
    y = np.concatenate(targets, axis=0)
    persistence = np.concatenate(last_values, axis=0)
    seasonal_naive = np.concatenate(seasonal, axis=0)
    return X, y, persistence, seasonal_naive


if TORCH_AVAILABLE:

    class _LstmRegressor(nn.Module):
        def __init__(self, n_features, hidden):
            super(_LstmRegressor, self).__init__()
            self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
            self.head = nn.Linear(hidden, 1)

        def forward(self, x):
            output, _ = self.lstm(x)
            return self.head(output[:, -1, :]).squeeze(-1)


class EnergyHarvestPredictor(object):
    """Forecasts cumulative harvest over the next ``LSTM_HORIZON`` rounds."""

    def __init__(self, hidden=None, input_len=None, horizon=None, lr=None,
                 epochs=None, seed=0):
        self.hidden = config.LSTM_HIDDEN if hidden is None else int(hidden)
        self.input_len = config.LSTM_INPUT_LEN if input_len is None else int(input_len)
        self.horizon = config.LSTM_HORIZON if horizon is None else int(horizon)
        self.lr = config.LSTM_LR if lr is None else float(lr)
        self.epochs = config.LSTM_EPOCHS if epochs is None else int(epochs)
        self.seed = int(seed)

        self.model = None
        self.backend = "torch" if TORCH_AVAILABLE else "persistence-fallback"
        self.using_fallback = not TORCH_AVAILABLE
        self.trained = False
        self.history = []
        self.evaluation = {}
        self._y_scale = 1.0

    def fit(self, n_nodes=60, n_rounds=1200, seed=None, epochs=None,
            validation_split=0.25):
        """Train on freshly generated harvest traces."""
        rng = np.random.default_rng(self.seed if seed is None else int(seed))
        harvest, energy_trace, _ = generate_harvest_traces(n_nodes, n_rounds, rng)
        X, y, persistence, seasonal = build_dataset(
            harvest, energy_trace, self.input_len, self.horizon
        )
        if X.shape[0] < 64:
            self.using_fallback = True
            self.trained = True
            return self.summary()

        split = int((1.0 - validation_split) * X.shape[0])
        order = rng.permutation(X.shape[0])
        train_idx, test_idx = order[:split], order[split:]

        self._y_scale = float(np.abs(y[train_idx]).max()) or 1.0

        if TORCH_AVAILABLE:
            torch.manual_seed(self.seed)
            self.model = _LstmRegressor(N_FEATURES, self.hidden)
            optimiser = torch.optim.Adam(self.model.parameters(), lr=self.lr)
            loss_fn = torch.nn.MSELoss()

            X_train = torch.from_numpy(X[train_idx].astype(np.float32))
            y_train = torch.from_numpy((y[train_idx] / self._y_scale).astype(np.float32))

            epochs = self.epochs if epochs is None else int(epochs)
            batch = 256
            self.history = []
            for _ in range(epochs):
                permutation = torch.randperm(X_train.shape[0])
                epoch_loss = 0.0
                batches = 0
                for start in range(0, X_train.shape[0], batch):
                    index = permutation[start:start + batch]
                    prediction = self.model(X_train[index])
                    loss = loss_fn(prediction, y_train[index])
                    optimiser.zero_grad()
                    loss.backward()
                    optimiser.step()
                    epoch_loss += float(loss.item())
                    batches += 1
                self.history.append(epoch_loss / max(1, batches))
            self.trained = True

        self.evaluation = self.evaluate(
            X[test_idx], y[test_idx], persistence[test_idx], seasonal[test_idx]
        )
        return self.summary()

    def predict_batch(self, X):
        """Predicted cumulative harvest for a batch of windows."""
        X = np.asarray(X, dtype=np.float32)
        if X.size == 0:
            return np.zeros(0)
        if self.using_fallback or self.model is None:
            # Persistence: hold the last observed harvest for the horizon.
            return X[:, -1, 0] * self.horizon
        with torch.no_grad():
            output = self.model(torch.from_numpy(X)).numpy()
        return output * self._y_scale

    def evaluate(self, X, y, persistence, seasonal):
        """MAE / RMSE for the LSTM and both baselines."""
        prediction = self.predict_batch(X)

        def errors(pred):
            residual = np.asarray(pred, dtype=float) - np.asarray(y, dtype=float)
            return float(np.abs(residual).mean()), float(np.sqrt((residual ** 2).mean()))

        lstm_mae, lstm_rmse = errors(prediction)
        pers_mae, pers_rmse = errors(persistence)
        seas_mae, seas_rmse = errors(seasonal)

        best_baseline = min(pers_mae, seas_mae)
        # A margin, because in fallback mode the "model" *is* persistence and a
        # bit-level float difference must not be reported as beating it.
        margin = 1e-6 + 0.001 * max(pers_mae, seas_mae)
        return {
            "n_test": int(len(y)),
            "lstm_mae": lstm_mae,
            "lstm_rmse": lstm_rmse,
            "persistence_mae": pers_mae,
            "persistence_rmse": pers_rmse,
            "seasonal_naive_mae": seas_mae,
            "seasonal_naive_rmse": seas_rmse,
            "beats_persistence": bool(lstm_mae < pers_mae - margin),
            "beats_seasonal_naive": bool(lstm_mae < seas_mae - margin),
            "improvement_over_best_baseline": (
                float((best_baseline - lstm_mae) / best_baseline)
                if best_baseline > 0 else 0.0
            ),
        }

    # -- online use inside the simulation ---------------------------------

    def predicted_surplus(self, harvest_history, energy_history, round_idx):
        """Per-node forecast from live simulation history.

        ``harvest_history`` and ``energy_history`` are ``(T, n_nodes)`` arrays of
        the most recent rounds. Falls back to persistence until enough history
        has accumulated.
        """
        harvest_history = np.asarray(harvest_history, dtype=float)
        if harvest_history.ndim != 2 or harvest_history.shape[0] < self.input_len:
            if harvest_history.size == 0:
                return None
            return harvest_history[-1] * self.horizon

        window_harvest = harvest_history[-self.input_len:]
        window_energy = (
            np.asarray(energy_history, dtype=float)[-self.input_len:]
            / config.MAX_ENERGY_CAP
        )
        rounds = np.arange(round_idx - self.input_len + 1, round_idx + 1)
        sin_phase, cos_phase = phase_features(rounds)
        n_nodes = window_harvest.shape[1]

        window = np.stack(
            [
                window_harvest,
                window_energy,
                np.repeat(sin_phase[:, None], n_nodes, axis=1),
                np.repeat(cos_phase[:, None], n_nodes, axis=1),
            ],
            axis=-1,
        )
        X = np.transpose(window, (1, 0, 2))
        return self.predict_batch(X)

    def summary(self):
        row = {
            "lstm_backend": self.backend,
            "lstm_using_fallback": bool(self.using_fallback),
            "lstm_epochs_run": len(self.history),
            "lstm_final_loss": float(self.history[-1]) if self.history else None,
            "lstm_horizon": self.horizon,
        }
        row.update({"lstm_eval_" + k: v for k, v in self.evaluation.items()})
        return row


In [ ]:
%%writefile src/c_ar_eaurp/cmdp_agent.py
"""Constrained-MDP routing agent -- AR-EAURP step 4.

The doc asks for a hard constraint: PDR must stay above 90% even when the
adversary controls 30% of the nodes, with a large negative reward on violation.
Implemented as a Lagrangian CMDP, which is the standard way to turn "hard
constraint" into something a policy-gradient-free learner can actually optimise:

    r = +/-1  -  w_e * energy_cost  -  lambda * max(0, 0.90 - PDR_window) * C
    lambda <- clip(lambda + eta * (0.90 - PDR_window), 0, lambda_max)

``lambda`` rises while the constraint is violated, so the penalty grows until
the policy either satisfies the floor or saturates. That saturation is itself
the result worth reporting: the Review-1 critique flags the 90% floor as
possibly infeasible under sustained attack, and a lambda pinned at its ceiling
with the constraint still violated is direct evidence of exactly that. The
violation rate and the lambda trajectory are both logged.

STATE (extends Lekha.pdf Eq. 29 from three dimensions to six):

    T_robust          mean trust after the GAN's contested-node down-weighting
    E                 mean normalised residual energy
    M                 mean mobility factor
    E_pred_surplus    mean LSTM-predicted harvest over the next 10 rounds
    contested_ratio   fraction of nodes currently flagged by the detector
    pdr_window        running delivery ratio -- the constraint signal

ACTIONS (answering the Review-1 critique that a binary action space is coarse):

    0  route on trust
    1  route on predicted energy surplus
    2  route on the balanced EAURP R_Score
    3  route on a diversified, node-disjoint path

A two-action mode is kept for direct parity with the base paper.
"""

import numpy as np

from ..common import config

try:  # pragma: no cover - depends on the runtime environment
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    TORCH_AVAILABLE = True
except Exception:  # pragma: no cover
    TORCH_AVAILABLE = False

# One gradient step per packet is wasteful and, over a 90-configuration attack
# sweep, is the single biggest cost in the whole evaluation: 400 rounds x 4
# packets x 5 runs x 90 configs is ~720k backward passes. Training every Nth
# transition keeps the same learning signal (every transition still enters the
# replay buffer) at a quarter of the cost.
TRAIN_EVERY = 4

STATE_DIM = 6
ACTION_TRUST = 0
ACTION_ENERGY = 1
ACTION_BALANCED = 2
ACTION_DIVERSE = 3
ACTION_NAMES = ("trust", "energy_surplus", "balanced", "diversified")

ENERGY_COST_WEIGHT = 0.05


if TORCH_AVAILABLE:

    class _QNetwork(nn.Module):
        def __init__(self, state_dim, hidden, n_actions):
            super(_QNetwork, self).__init__()
            self.net = nn.Sequential(
                nn.Linear(state_dim, hidden),
                nn.ReLU(),
                nn.Linear(hidden, hidden),
                nn.ReLU(),
                nn.Linear(hidden, n_actions),
            )

        def forward(self, x):
            return self.net(x)


class ReplayBuffer(object):
    """Fixed-capacity circular buffer of transitions."""

    def __init__(self, capacity, state_dim):
        self.capacity = int(capacity)
        self.states = np.zeros((self.capacity, state_dim), dtype=np.float32)
        self.actions = np.zeros(self.capacity, dtype=np.int64)
        self.rewards = np.zeros(self.capacity, dtype=np.float32)
        self.next_states = np.zeros((self.capacity, state_dim), dtype=np.float32)
        self.size = 0
        self.cursor = 0

    def add(self, state, action, reward, next_state):
        index = self.cursor
        self.states[index] = state
        self.actions[index] = action
        self.rewards[index] = reward
        self.next_states[index] = next_state
        self.cursor = (self.cursor + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size, rng):
        count = min(int(batch_size), self.size)
        index = rng.integers(0, self.size, count)
        return (
            self.states[index],
            self.actions[index],
            self.rewards[index],
            self.next_states[index],
        )


class CmdpRoutingAgent(object):
    """DQN with a Lagrangian PDR constraint; tabular fallback without torch."""

    def __init__(self, n_actions=4, hidden=None, lr=None, gamma=None,
                 buffer_size=None, batch_size=None, target_sync=None,
                 pdr_floor=None, seed=0):
        self.n_actions = int(n_actions)
        self.hidden = config.DQN_HIDDEN if hidden is None else int(hidden)
        self.lr = config.DQN_LR if lr is None else float(lr)
        self.gamma = config.RL_GAMMA if gamma is None else float(gamma)
        self.batch_size = config.DQN_BATCH if batch_size is None else int(batch_size)
        self.target_sync = (
            config.DQN_TARGET_SYNC if target_sync is None else int(target_sync)
        )
        self.pdr_floor = (
            config.CMDP_PDR_FLOOR if pdr_floor is None else float(pdr_floor)
        )
        self.seed = int(seed)

        self.backend = "dqn" if TORCH_AVAILABLE else "tabular-fallback"
        self.using_fallback = not TORCH_AVAILABLE

        self.buffer = ReplayBuffer(
            config.DQN_BUFFER if buffer_size is None else int(buffer_size), STATE_DIM
        )
        self.online = None
        self.target = None
        self.optimiser = None
        self._table = {}

        self.steps = 0
        self.train_steps = 0
        self._since_train = 0
        self.epsilon = config.DQN_EPS_START
        self.lam = 0.0
        self.lambda_trace = []
        self.action_counts = np.zeros(self.n_actions, dtype=np.int64)
        self.losses = []
        self.constraint_checks = 0
        self.constraint_violations = 0

        if TORCH_AVAILABLE:
            torch.manual_seed(self.seed)
            self.online = _QNetwork(STATE_DIM, self.hidden, self.n_actions)
            self.target = _QNetwork(STATE_DIM, self.hidden, self.n_actions)
            self.target.load_state_dict(self.online.state_dict())
            self.optimiser = torch.optim.Adam(self.online.parameters(), lr=self.lr)

    # -- policy ------------------------------------------------------------

    def _discretise(self, state):
        """Tabular fallback key: one decimal per dimension."""
        return tuple(round(float(v), 1) for v in state)

    def q_values(self, state):
        if self.using_fallback or self.online is None:
            return np.asarray(
                self._table.setdefault(self._discretise(state),
                                       [0.0] * self.n_actions),
                dtype=float,
            )
        with torch.no_grad():
            tensor = torch.from_numpy(np.asarray(state, dtype=np.float32)[None, :])
            return self.online(tensor).numpy().ravel()

    def act(self, state, rng):
        """Epsilon-greedy with a linear decay on epsilon."""
        self.steps += 1
        self.epsilon = max(
            config.DQN_EPS_END,
            config.DQN_EPS_START
            - (config.DQN_EPS_START - config.DQN_EPS_END)
            * self.steps / float(config.DQN_EPS_DECAY),
        )
        if rng.random() < self.epsilon:
            action = int(rng.integers(0, self.n_actions))
        else:
            action = int(np.argmax(self.q_values(state)))
        self.action_counts[action] += 1
        return action

    # -- CMDP reward -------------------------------------------------------

    def shaped_reward(self, delivered, energy_cost, pdr_window):
        """Eq. (37) plus the energy term and the Lagrangian constraint penalty."""
        base = config.REWARD_SUCCESS if delivered else config.REWARD_FAILURE
        violation = max(0.0, self.pdr_floor - float(pdr_window))

        self.constraint_checks += 1
        if violation > 0.0:
            self.constraint_violations += 1

        penalty = self.lam * violation * config.CMDP_PENALTY
        return float(base - ENERGY_COST_WEIGHT * float(energy_cost) - penalty), violation

    def update_lambda(self, pdr_window):
        """Dual ascent on the constraint multiplier."""
        gap = self.pdr_floor - float(pdr_window)
        self.lam = float(
            np.clip(self.lam + config.CMDP_LAMBDA_LR * gap, 0.0, config.CMDP_LAMBDA_MAX)
        )
        self.lambda_trace.append(self.lam)
        return self.lam

    # -- learning ----------------------------------------------------------

    def observe(self, state, action, reward, next_state, rng):
        if self.using_fallback or self.online is None:
            key = self._discretise(state)
            next_key = self._discretise(next_state)
            values = self._table.setdefault(key, [0.0] * self.n_actions)
            next_values = self._table.setdefault(next_key, [0.0] * self.n_actions)
            values[action] += config.RL_ALPHA * (
                reward + self.gamma * max(next_values) - values[action]
            )
            self.train_steps += 1
            return

        self.buffer.add(state, action, reward, next_state)
        if self.buffer.size < max(self.batch_size, 64):
            return

        # Every transition is stored; only every TRAIN_EVERY-th one triggers a
        # gradient step. See the note on TRAIN_EVERY above.
        self._since_train += 1
        if self._since_train < TRAIN_EVERY:
            return
        self._since_train = 0

        states, actions, rewards, next_states = self.buffer.sample(
            self.batch_size, rng
        )
        states_t = torch.from_numpy(states)
        actions_t = torch.from_numpy(actions)
        rewards_t = torch.from_numpy(rewards)
        next_states_t = torch.from_numpy(next_states)

        q_selected = self.online(states_t).gather(1, actions_t[:, None]).squeeze(1)
        with torch.no_grad():
            target_q = rewards_t + self.gamma * self.target(next_states_t).max(dim=1)[0]

        loss = F.mse_loss(q_selected, target_q)
        self.optimiser.zero_grad()
        loss.backward()
        self.optimiser.step()

        self.losses.append(float(loss.item()))
        self.train_steps += 1
        if self.train_steps % self.target_sync == 0:
            self.target.load_state_dict(self.online.state_dict())

    # -- reporting ---------------------------------------------------------

    def summary(self):
        total = int(self.action_counts.sum()) or 1
        row = {
            "cmdp_backend": self.backend,
            "cmdp_using_fallback": bool(self.using_fallback),
            "cmdp_lambda_final": float(self.lam),
            "cmdp_lambda_max_seen": (
                float(max(self.lambda_trace)) if self.lambda_trace else 0.0
            ),
            "cmdp_lambda_saturated": bool(
                self.lambda_trace and max(self.lambda_trace) >= config.CMDP_LAMBDA_MAX - 1e-6
            ),
            "cmdp_constraint_checks": self.constraint_checks,
            "cmdp_constraint_violations": self.constraint_violations,
            "cmdp_violation_rate": (
                self.constraint_violations / float(self.constraint_checks)
                if self.constraint_checks else 0.0
            ),
            "dqn_train_steps": self.train_steps,
            "dqn_final_loss": float(self.losses[-1]) if self.losses else None,
            "dqn_epsilon_final": float(self.epsilon),
        }
        for index, name in enumerate(ACTION_NAMES[: self.n_actions]):
            row["action_share_" + name] = self.action_counts[index] / float(total)
        return row


In [ ]:
%%writefile src/c_ar_eaurp/protocol.py
"""AR-EAURP -- the advancement, assembled (docx step 5).

Built on the senior's DRL-EAURP: the Q-learning routing agent is kept and
extended, rather than replaced.

    base paper (B)          AR-EAURP (C)
    ----------------------  -----------------------------------------------
    trust = scalar PFR      trust down-weighted 50% for GAN-contested nodes
    state <T, E, M>         + predicted surplus, contested ratio, PDR window
    2 actions (+0.08/+0.02) 4 real route strategies
    reward +/-1             + energy term + Lagrangian PDR>=0.90 constraint
    linear energy drain     solar/kinetic harvesting with a 10-round forecast

TRAIN/TEST SEPARATION. The GAN is trained during a separate *clean*
commissioning run with no adversaries present, then frozen. It never sees an
attack before being evaluated on one. This matters: a detector tuned on the
attacks it is then scored against proves nothing.

DEFENCE ORDERING. A black-hole lies in its route reply, advertising a perfect
score. Lowering its measured trust therefore does nothing on its own -- the lie
overwrites it. So the contested down-weight is applied *after* the advertised
score, which is the only ordering in which the defence actually bites.
"""

import numpy as np

from ..common import config
from ..common.simulator import RoutingPolicy, SimParams, Simulation
from ..routing import aodv
from . import cmdp_agent as cmdp
from .gan_detector import GanAnomalyDetector
from .lstm_energy import EnergyHarvestPredictor
from .threat_model import BehaviourFeatureExtractor, aggregate_by_subject


class FeatureCollectorPolicy(RoutingPolicy):
    """Drives the clean commissioning run that produces the GAN's training set."""

    name = "collector"

    def __init__(self, detect_period=None):
        self.detect_period = (
            config.DETECT_PERIOD if detect_period is None else int(detect_period)
        )
        self.extractor = None
        self.windows = []

    def reset(self, sim):
        self.extractor = BehaviourFeatureExtractor(sim.nodes.n)
        self.windows = []
        sim.refresh_scores()

    def on_round_end(self, sim, round_idx):
        if round_idx > 0 and round_idx % self.detect_period == 0:
            features, _, _ = self.extractor.extract(sim.nodes)
            if features.shape[0]:
                self.windows.append(features)

    def collected(self):
        if not self.windows:
            return np.zeros((0, config.GAN_FEATURE_DIM))
        return np.concatenate(self.windows, axis=0)


def collect_clean_features(n_nodes=100, speed=20000, rounds=400, seed=9001,
                           energy_model="linear"):
    """Run an adversary-free simulation and return its behaviour windows."""
    policy = FeatureCollectorPolicy()
    params = SimParams(
        n_nodes=n_nodes, speed=speed, rounds=rounds, seed=seed,
        attack="none", malicious_fraction=0.0, energy_model=energy_model,
    )
    Simulation(policy, params).run()
    return policy.collected()


def collect_labelled_features(n_nodes=100, speed=20000, rounds=400, seed=9002,
                              attack="grayhole", malicious_fraction=0.3):
    """A held-out attacked run, used only to *measure* detector quality."""
    policy = FeatureCollectorPolicy()
    params = SimParams(
        n_nodes=n_nodes, speed=speed, rounds=rounds, seed=seed,
        attack=attack, malicious_fraction=malicious_fraction,
    )
    simulation = Simulation(policy, params)
    labels = []
    windows = []

    extractor = BehaviourFeatureExtractor(n_nodes)
    policy.extractor = extractor

    class _Labelling(FeatureCollectorPolicy):
        def on_round_end(self, inner_sim, round_idx):
            if round_idx > 0 and round_idx % self.detect_period == 0:
                features, _, subjects = self.extractor.extract(inner_sim.nodes)
                if features.shape[0]:
                    windows.append(features)
                    labels.append(inner_sim.nodes.malicious_mask()[subjects])

    simulation.policy = _Labelling()
    simulation.run()

    if not windows:
        return np.zeros((0, config.GAN_FEATURE_DIM)), np.zeros(0, dtype=bool)
    return np.concatenate(windows, axis=0), np.concatenate(labels, axis=0)


class ArEaurpPolicy(RoutingPolicy):
    """AR-EAURP: GAN-hardened trust + LSTM energy foresight + CMDP routing."""

    name = "C:AR-EAURP"

    def __init__(self, detector=None, predictor=None, agent=None,
                 use_gan=True, use_lstm=True, use_cmdp=True,
                 detect_period=None, n_actions=4):
        self.use_gan = bool(use_gan)
        self.use_lstm = bool(use_lstm)
        self.use_cmdp = bool(use_cmdp)
        self.detect_period = (
            config.DETECT_PERIOD if detect_period is None else int(detect_period)
        )

        self.detector = detector
        self.predictor = predictor
        self.agent = agent if agent is not None else cmdp.CmdpRoutingAgent(
            n_actions=int(n_actions)
        )

        self.extractor = None
        self.caches = {}
        self._pending = None
        self._surplus = None
        self._harvest_history = []
        self._energy_history = []
        self.contested_events = 0
        self.detection_passes = 0

    # -- setup -------------------------------------------------------------

    def pretrain(self, n_nodes=100, speed=20000, rounds=400, seed=9001,
                 gan_epochs=None, lstm_epochs=None, validate=True,
                 energy_model="linear", verbose=True):
        """Commissioning phase: train the detector and the forecaster.

        Both are trained on data that contains no attack, then frozen.
        """
        summary = {}

        if self.use_gan and self.detector is None:
            clean = collect_clean_features(
                n_nodes=n_nodes, speed=speed, rounds=rounds, seed=seed,
                energy_model=energy_model,
            )
            validation_features, validation_labels = (None, None)
            if validate:
                validation_features, validation_labels = collect_labelled_features(
                    n_nodes=n_nodes, speed=speed, rounds=rounds, seed=seed + 1,
                    attack="grayhole", malicious_fraction=0.3,
                )
            self.detector = GanAnomalyDetector(seed=seed)
            summary.update(
                self.detector.fit(
                    clean,
                    validation_features=validation_features,
                    validation_labels=validation_labels,
                    epochs=gan_epochs,
                )
            )
            summary["gan_clean_windows"] = int(clean.shape[0])
            if verbose:
                print("  [C] GAN trained on {0} clean windows; backend={1}; "
                      "val AUC={2}".format(
                          clean.shape[0], self.detector.backend,
                          summary.get("gan_val_auc")))

        if self.use_lstm and self.predictor is None:
            self.predictor = EnergyHarvestPredictor(seed=seed)
            summary.update(
                self.predictor.fit(n_nodes=60, n_rounds=1200, epochs=lstm_epochs)
            )
            if verbose:
                evaluation = self.predictor.evaluation
                print("  [C] LSTM backend={0}; MAE {1:.5f} vs persistence "
                      "{2:.5f} / seasonal {3:.5f}".format(
                          self.predictor.backend,
                          evaluation.get("lstm_mae", float("nan")),
                          evaluation.get("persistence_mae", float("nan")),
                          evaluation.get("seasonal_naive_mae", float("nan"))))
        return summary

    # -- harness hooks -----------------------------------------------------

    def reset(self, sim):
        self.extractor = BehaviourFeatureExtractor(sim.nodes.n)
        self.caches = {
            action: aodv.RouteCache() for action in range(self.agent.n_actions)
        }
        self._pending = None
        self._surplus = None
        self._harvest_history = []
        self._energy_history = []
        self.contested_events = 0
        self.detection_passes = 0
        self._refresh(sim)

    def robust_trust(self, sim):
        """Mean observed trust per node, before the contested down-weight."""
        return sim.nodes.network_trust()

    def _score_vectors(self, sim):
        """Three score vectors: trust-led, surplus-led and balanced."""
        nodes = sim.nodes
        trust = self.robust_trust(sim)
        energy = nodes.normalised_energy()

        surplus = energy
        if self.use_lstm and self._surplus is not None:
            span = float(np.ptp(self._surplus))
            if span > 1e-9:
                normalised = (self._surplus - self._surplus.min()) / span
            else:
                normalised = np.zeros_like(self._surplus)
            # Blend present energy with forecast headroom: a node that is low
            # now but about to recharge is a reasonable relay, one that is low
            # now and staying low is not.
            surplus = np.clip(0.5 * energy + 0.5 * normalised, 0.0, 1.0)

        vectors = {
            cmdp.ACTION_TRUST: aodv.route_scores(nodes, trust=trust, energy=energy * 0.0 + 1.0),
            cmdp.ACTION_ENERGY: aodv.route_scores(nodes, trust=trust, energy=surplus),
            cmdp.ACTION_BALANCED: aodv.route_scores(nodes, trust=trust, energy=energy),
        }
        vectors[cmdp.ACTION_DIVERSE] = vectors[cmdp.ACTION_BALANCED]

        # The contested down-weight is applied *after* route_scores, so that it
        # also bites on a black-hole's advertised (lied) score. Applying it
        # before would be overwritten by the lie and achieve nothing.
        if self.use_gan:
            contested = sim.nodes.contested
            if contested.any():
                for key in vectors:
                    vectors[key] = vectors[key].copy()
                    vectors[key][contested] *= config.CONTESTED_TRUST_WEIGHT
        return vectors

    def _refresh(self, sim):
        self._vectors = self._score_vectors(sim)
        sim.scores = self._vectors[cmdp.ACTION_BALANCED]

    def state_of(self, sim):
        """The six-dimensional CMDP state."""
        nodes = sim.nodes
        alive = nodes.alive
        if not alive.any():
            return np.zeros(cmdp.STATE_DIM, dtype=np.float32)

        trust = self.robust_trust(sim)
        if self.use_gan and nodes.contested.any():
            trust = trust.copy()
            trust[nodes.contested] *= config.CONTESTED_TRUST_WEIGHT

        surplus = 0.0
        if self.use_lstm and self._surplus is not None:
            surplus = float(np.clip(self._surplus[alive].mean() * 10.0, 0.0, 1.0))

        return np.array(
            [
                float(trust[alive].mean()),
                float(nodes.normalised_energy()[alive].mean()),
                float(nodes.mobility_factor()[alive].mean()),
                surplus,
                float(nodes.contested.mean()),
                float(sim.metrics.window_pdr()),
            ],
            dtype=np.float32,
        )

    def on_round_start(self, sim, round_idx):
        nodes = sim.nodes

        # Expire contested flags whose time-to-live has run out.
        if nodes.contested.any():
            expired = nodes.contested & (nodes.contested_until <= round_idx)
            if expired.any():
                nodes.contested[expired] = False

        # Track harvest/energy history for the forecaster.
        if self.use_lstm:
            self._harvest_history.append(nodes.harvest_last.copy())
            self._energy_history.append(nodes.energy.copy())
            keep = max(self.predictor.input_len if self.predictor else 20, 20)
            if len(self._harvest_history) > keep:
                self._harvest_history.pop(0)
                self._energy_history.pop(0)

        if round_idx > 0 and round_idx % self.detect_period == 0:
            self._detection_pass(sim, round_idx)
            self._forecast(sim, round_idx)
            self._refresh(sim)

    def _detection_pass(self, sim, round_idx):
        """Score recent behaviour and mark deviating nodes contested."""
        if not self.use_gan or self.detector is None or self.extractor is None:
            return
        features, observers, subjects = self.extractor.extract(sim.nodes)
        if features.shape[0] == 0:
            return

        self.detection_passes += 1
        flags, scores = self.detector.flag(features)
        per_node = aggregate_by_subject(
            scores.astype(float), subjects, sim.nodes.n, reduce="mean"
        )
        sim.nodes.anomaly_score = per_node

        # A node is contested when its *mean* score across observers clears the
        # threshold. Using the mean rather than the max is deliberate: it stops
        # a lone slanderer getting an honest node flagged by itself.
        contested = per_node > self.detector.threshold
        newly = contested & (~sim.nodes.contested)
        self.contested_events += int(newly.sum())

        sim.nodes.contested[contested] = True
        sim.nodes.contested_until[contested] = int(round_idx) + config.CONTESTED_TTL

    def _forecast(self, sim, round_idx):
        if not self.use_lstm or self.predictor is None:
            return
        if len(self._harvest_history) < 2:
            return
        prediction = self.predictor.predicted_surplus(
            np.asarray(self._harvest_history),
            np.asarray(self._energy_history),
            round_idx,
        )
        if prediction is not None:
            self._surplus = np.asarray(prediction, dtype=float)

    def select_route(self, sim, packet):
        state = self.state_of(sim)
        action = self.agent.act(state, sim.seeds.policy)

        scores = self._vectors.get(action, sim.scores)
        cache = self.caches.get(action)

        path = cache.get(sim.nodes, packet.src, packet.dst) if cache else None
        if path is None:
            if action == cmdp.ACTION_DIVERSE:
                primary = aodv.find_route(
                    sim.nodes, packet.src, packet.dst, scores, mode=aodv.MODE_EAURP
                )
                path = aodv.find_diverse_route(
                    sim.nodes, packet.src, packet.dst, scores, primary=primary
                )
            else:
                path = aodv.find_route(
                    sim.nodes, packet.src, packet.dst, scores, mode=aodv.MODE_EAURP
                )
            sim.metrics.record_discovery(path is not None)
            if path is not None and cache:
                cache.put(packet.src, packet.dst, path)

        self._pending = (state, action)
        return path

    def on_result(self, sim, packet, delivered, path, hops, reason):
        if self._pending is None:
            return
        state, action = self._pending

        energy_cost = float(hops) * (config.ENERGY_TX + config.ENERGY_RX)
        pdr_window = sim.metrics.window_pdr()

        if self.use_cmdp:
            reward, violation = self.agent.shaped_reward(
                delivered, energy_cost, pdr_window
            )
            sim.metrics.record_cmdp_check(violation <= 0.0)
        else:
            reward = config.REWARD_SUCCESS if delivered else config.REWARD_FAILURE

        next_state = self.state_of(sim)
        self.agent.observe(state, action, reward, next_state, sim.seeds.policy)
        self._pending = None

    def on_round_end(self, sim, round_idx):
        if self.use_cmdp and round_idx % 10 == 0:
            self.agent.update_lambda(sim.metrics.window_pdr())

    def detected_mask(self, sim):
        """C flags a node either by GAN contest or by EAURP revocation."""
        return sim.nodes.contested | sim.nodes.revoked

    def extra_metrics(self, sim):
        row = {
            "c_use_gan": self.use_gan,
            "c_use_lstm": self.use_lstm,
            "c_use_cmdp": self.use_cmdp,
            "c_contested_events": self.contested_events,
            "c_detection_passes": self.detection_passes,
            "c_contested_final": int(sim.nodes.contested.sum()),
        }
        row.update(self.agent.summary())
        if self.detector is not None:
            row.update(self.detector.summary())
        if self.predictor is not None:
            row.update(self.predictor.summary())
        # feature_names is a list; drop it from the CSV row.
        row.pop("feature_names", None)
        return row


In [ ]:
%%writefile experiments/__init__.py
"""Experiment drivers for the AR-EAURP Review-2 evaluation."""


In [ ]:
%%writefile experiments/harness.py
"""Shared experiment plumbing: policy registry, sweeps, CSV checkpointing.

Checkpointing matters on Colab. Every sweep writes its CSV as soon as it
finishes and checks for that file on start, so a session that disconnects
half-way through costs only the sweep that was in flight rather than the whole
run. Set ``AR_EAURP_RESULTS`` (or pass ``out_dir``) to a Google Drive path and
results survive the session entirely.
"""

import csv
import os
import time

import numpy as np

from src.a_drl_eaurp.policy_common import TabularDrlPolicy
from src.b_senior.policy_common import SeniorDrlPolicy
from src.c_ar_eaurp.protocol import ArEaurpPolicy
from src.common import config, metrics as metrics_mod, seeding
from src.common.simulator import SimParams, Simulation
from src.routing.eaurp import AodvPolicy, EaurpPolicy

DEFAULT_OUT = os.environ.get("AR_EAURP_RESULTS", "results")
BASE_SEED = 20260909


def out_path(*parts, **kwargs):
    """Build a path under the results directory, creating parents."""
    root = kwargs.get("out_dir") or DEFAULT_OUT
    path = os.path.join(root, *parts)
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    return path


def csv_path(name, out_dir=None):
    return out_path("csv", name, out_dir=out_dir)


# --------------------------------------------------------------------------
# Policy registry
# --------------------------------------------------------------------------

def make_policies(shared_c=None):
    """The Track-2 line-up. ``shared_c`` reuses one pre-trained AR-EAURP."""
    return [
        ("AODV (reference baseline)", lambda: AodvPolicy()),
        ("EAURP (base.pdf substrate)", lambda: EaurpPolicy()),
        ("A: DRL-EAURP (paper)", lambda: TabularDrlPolicy()),
        ("B: DRL-EAURP (senior code)", lambda: SeniorDrlPolicy()),
        ("C: AR-EAURP (advancement)", (lambda: shared_c) if shared_c else
         (lambda: ArEaurpPolicy())),
    ]


def pretrained_c(profile, n_nodes=100, seed=9001, verbose=True, **kwargs):
    """Train C's detector and forecaster once and reuse across the sweeps."""
    policy = ArEaurpPolicy(**kwargs)
    summary = policy.pretrain(
        n_nodes=n_nodes,
        rounds=max(200, profile.rounds),
        seed=seed,
        gan_epochs=profile.gan_epochs,
        lstm_epochs=profile.lstm_epochs,
        verbose=verbose,
    )
    return policy, summary


# --------------------------------------------------------------------------
# Sweeps
# --------------------------------------------------------------------------

def run_config(policy_factory, params, runs, base_seed=BASE_SEED):
    """Run one configuration over ``runs`` seeds and average the results."""
    rows = []
    for run_index in range(int(runs)):
        params.seed = seeding.run_seed(base_seed, run_index)
        policy = policy_factory()
        rows.append(Simulation(policy, params).run())
    aggregated = metrics_mod.aggregate(rows)
    return aggregated


def sweep(name, configs, runs, out_dir=None, resume=True, verbose=True,
          base_seed=BASE_SEED):
    """Run a list of ``(label, policy_factory, SimParams, extra)`` configs.

    Returns the rows and writes them to ``csv/<name>.csv``.
    """
    path = csv_path(name + ".csv", out_dir=out_dir)
    if resume and os.path.exists(path):
        if verbose:
            print("  [skip] {0} already present at {1}".format(name, path))
        with open(path, "r", encoding="utf-8", newline="") as handle:
            return list(csv.DictReader(handle)), path

    rows = []
    started = time.perf_counter()
    for index, (label, factory, params, extra) in enumerate(configs, start=1):
        row = run_config(factory, params, runs, base_seed=base_seed)
        row["label"] = label
        row.update(extra or {})
        rows.append(row)
        if verbose:
            print("  [{0}/{1}] {2:44s} PDR={3:.3f} routable={4:.3f} "
                  "delay={5:6.1f} TPR={6:.2f} FPR={7:.2f}".format(
                      index, len(configs), label[:44],
                      row.get("pdr", 0.0), row.get("pdr_routable", 0.0),
                      row.get("avg_delay_ms", 0.0),
                      row.get("det_tpr", 0.0), row.get("det_fpr", 0.0)))

    write_rows(rows, path)
    if verbose:
        print("  -> {0}  ({1:.1f}s)".format(path, time.perf_counter() - started))
    return rows, path


def write_rows(rows, path):
    """Write dict rows to CSV, unioning keys across rows."""
    if not rows:
        return path
    fieldnames = []
    seen = set()
    for row in rows:
        for key in row:
            if key not in seen:
                seen.add(key)
                fieldnames.append(key)
    directory = os.path.dirname(os.path.abspath(path))
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, "") for key in fieldnames})
    return path


def read_rows(path):
    if not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def as_float(rows, key, default=0.0):
    """Pull a numeric column out of CSV-read (string-valued) rows."""
    out = []
    for row in rows:
        try:
            out.append(float(row.get(key, default)))
        except (TypeError, ValueError):
            out.append(default)
    return np.asarray(out)


In [ ]:
%%writefile experiments/sweeps.py
"""The six Track-2 experiments (E1-E6).

Kept in one module rather than six near-identical files; each experiment is a
function so the notebook can run them individually and the CLI can run them all.

    E1  speed sweep            all policies, 10k-40k (plus a realistic annex)
    E2  density sweep          all policies, 60-200 nodes
    E3  ATTACK SWEEP           A / B / C under 0-50% malicious x 3 attacks
    E4  energy + lifetime      linear vs solar, LSTM accuracy, network lifetime
    E5  ablation               C without GAN / LSTM / CMDP, plus a floor sweep
    E6  overhead               control traffic, wall-clock, ECC cost

E3 is the headline: it is the only experiment in which the advancement's threat
model is actually exercised, and the only one the base papers cannot run at all.
"""

import time

import numpy as np

from src.c_ar_eaurp.protocol import ArEaurpPolicy
from src.common import config
from src.common.simulator import SimParams, Simulation
from src.routing import ecc

from .harness import (
    BASE_SEED, csv_path, make_policies, pretrained_c, run_config, sweep,
    write_rows,
)


def _params(profile, **kwargs):
    base = dict(
        n_nodes=config.DEFAULT_NODES,
        speed=config.DEFAULT_SPEED,
        rounds=profile.rounds,
        packets_per_round=profile.packets_per_round,
    )
    base.update(kwargs)
    return SimParams(**base)


# --------------------------------------------------------------------------
# E1 -- speed sweep
# --------------------------------------------------------------------------

def e1_speed(profile, shared_c=None, out_dir=None, realistic=True, **kw):
    print("E1  speed sweep (paper range 10k-40k)")
    configs = []
    for label, factory in make_policies(shared_c):
        for speed in config.SPEEDS_PAPER:
            configs.append((
                "{0} @ v={1}".format(label, speed),
                factory,
                _params(profile, speed=speed),
                {"policy_label": label, "sweep": "speed", "speed_value": speed},
            ))
    rows, path = sweep("e1_speed", configs, profile.runs, out_dir=out_dir, **kw)

    if realistic:
        print("E1b speed sweep (physically realistic 1-20 m/round annex)")
        annex = []
        for label, factory in make_policies(shared_c):
            for speed in config.SPEEDS_REALISTIC:
                annex.append((
                    "{0} @ v={1}".format(label, speed),
                    factory,
                    _params(profile, speed=speed),
                    {"policy_label": label, "sweep": "speed_realistic",
                     "speed_value": speed},
                ))
        sweep("e1b_speed_realistic", annex, profile.runs, out_dir=out_dir, **kw)
    return rows, path


# --------------------------------------------------------------------------
# E2 -- density sweep
# --------------------------------------------------------------------------

def e2_density(profile, shared_c=None, out_dir=None, **kw):
    print("E2  node-density sweep (60-200 nodes)")
    configs = []
    for label, factory in make_policies(shared_c):
        for count in config.NODE_COUNTS:
            configs.append((
                "{0} @ N={1}".format(label, count),
                factory,
                _params(profile, n_nodes=count),
                {"policy_label": label, "sweep": "density", "node_count": count},
            ))
    return sweep("e2_density", configs, profile.runs, out_dir=out_dir, **kw)


# --------------------------------------------------------------------------
# E3 -- attack sweep (the headline)
# --------------------------------------------------------------------------

def e3_attack(profile, shared_c=None, out_dir=None, **kw):
    print("E3  ATTACK SWEEP -- 0-50% malicious x {blackhole, grayhole, "
          "trust_poisoning}")
    configs = []
    for label, factory in make_policies(shared_c):
        for attack in config.ATTACKS:
            for fraction in config.MALICIOUS_FRACTIONS:
                configs.append((
                    "{0} | {1} @ {2:.0f}%".format(label, attack, 100 * fraction),
                    factory,
                    _params(profile, attack=attack, malicious_fraction=fraction),
                    {
                        "policy_label": label,
                        "sweep": "attack",
                        "attack_type": attack,
                        "malicious_pct": 100.0 * fraction,
                    },
                ))
    return sweep("e3_attack", configs, profile.runs, out_dir=out_dir, **kw)


# --------------------------------------------------------------------------
# E4 -- energy model, lifetime and forecast accuracy
# --------------------------------------------------------------------------

def e4_energy(profile, shared_c=None, out_dir=None, **kw):
    print("E4  energy model, network lifetime and LSTM forecast accuracy")

    configs = []
    for label, factory in make_policies(shared_c):
        for model in ("linear", "solar"):
            configs.append((
                "{0} | {1} energy".format(label, model),
                factory,
                _params(profile, energy_model=model),
                {"policy_label": label, "sweep": "energy", "energy_mode": model},
            ))
    rows, path = sweep("e4_energy", configs, profile.runs, out_dir=out_dir, **kw)

    # Network lifetime needs a much longer horizon than the performance sweeps:
    # under the papers' own drain the first death is around round 1250.
    print("E4b network lifetime (long horizon, {0} rounds)".format(
        config.LIFETIME_ROUNDS))
    lifetime_configs = []
    for label, factory in make_policies(shared_c):
        for model in ("linear", "solar"):
            lifetime_configs.append((
                "{0} | {1}".format(label, model),
                factory,
                _params(profile, rounds=config.LIFETIME_ROUNDS,
                        energy_model=model, stop_when_dead=True),
                {"policy_label": label, "sweep": "lifetime", "energy_mode": model},
            ))
    sweep("e4b_lifetime", lifetime_configs, max(2, profile.runs // 2),
          out_dir=out_dir, **kw)

    # Forecast quality, on its own, against both naive baselines.
    if shared_c is not None and shared_c.predictor is not None:
        evaluation = dict(shared_c.predictor.evaluation)
        evaluation.update(shared_c.predictor.summary())
        write_rows([evaluation], csv_path("e4c_lstm_accuracy.csv", out_dir=out_dir))
        print("  LSTM MAE {0:.5f} | persistence {1:.5f} | seasonal-naive "
              "{2:.5f} | beats both: {3}".format(
                  evaluation.get("lstm_mae", float("nan")),
                  evaluation.get("persistence_mae", float("nan")),
                  evaluation.get("seasonal_naive_mae", float("nan")),
                  bool(evaluation.get("beats_persistence"))
                  and bool(evaluation.get("beats_seasonal_naive"))))
    return rows, path


# --------------------------------------------------------------------------
# E5 -- ablation of C, and the CMDP feasibility sweep
# --------------------------------------------------------------------------

def e5_ablation(profile, out_dir=None, attack="grayhole", fraction=0.3, **kw):
    print("E5  ablation of C  (-GAN, -LSTM, -CMDP) under {0} @ {1:.0f}%".format(
        attack, 100 * fraction))

    variants = [
        ("C full", dict(use_gan=True, use_lstm=True, use_cmdp=True)),
        ("C without GAN", dict(use_gan=False, use_lstm=True, use_cmdp=True)),
        ("C without LSTM", dict(use_gan=True, use_lstm=False, use_cmdp=True)),
        ("C without CMDP", dict(use_gan=True, use_lstm=True, use_cmdp=False)),
        ("C bare (none)", dict(use_gan=False, use_lstm=False, use_cmdp=False)),
    ]

    configs = []
    for label, flags in variants:
        policy, _ = pretrained_c(profile, verbose=False, **flags)
        configs.append((
            label,
            (lambda captured=policy: captured),
            _params(profile, attack=attack, malicious_fraction=fraction),
            {"policy_label": label, "sweep": "ablation", "attack_type": attack,
             "malicious_pct": 100.0 * fraction},
        ))
    rows, path = sweep("e5_ablation", configs, profile.runs, out_dir=out_dir, **kw)

    # How high can the CMDP floor go before it stops being satisfiable?
    # The Review-1 critique predicts the doc's 0.90 is infeasible; this measures
    # where the real ceiling is instead of asserting one.
    print("E5b CMDP feasibility -- how high a PDR floor is actually reachable")
    from src.c_ar_eaurp.cmdp_agent import CmdpRoutingAgent

    # Train the detector and forecaster once and share them across every floor
    # setting: only the constraint changes, so retraining per floor would burn
    # minutes to produce identical models.
    reference, _ = pretrained_c(profile, verbose=False)

    floor_configs = []
    for floor in (0.5, 0.6, 0.7, 0.8, 0.9):
        policy = ArEaurpPolicy(
            detector=reference.detector,
            predictor=reference.predictor,
            agent=CmdpRoutingAgent(n_actions=4, pdr_floor=floor),
        )
        floor_configs.append((
            "CMDP floor {0:.2f}".format(floor),
            (lambda captured=policy: captured),
            _params(profile, attack=attack, malicious_fraction=fraction),
            {"policy_label": "C", "sweep": "cmdp_floor", "pdr_floor": floor},
        ))
    sweep("e5b_cmdp_floor", floor_configs, max(2, profile.runs // 2),
          out_dir=out_dir, **kw)
    return rows, path


# --------------------------------------------------------------------------
# E6 -- overhead
# --------------------------------------------------------------------------

def e6_overhead(profile, shared_c=None, out_dir=None, **kw):
    print("E6  overhead -- control traffic, wall-clock and ECC cost")
    rows = []
    for label, factory in make_policies(shared_c):
        params = _params(profile, attack="grayhole", malicious_fraction=0.3)
        started = time.perf_counter()
        row = run_config(factory, params, max(2, profile.runs // 2))
        elapsed = time.perf_counter() - started
        row["label"] = label
        row["policy_label"] = label
        row["wall_clock_seconds"] = elapsed
        row["wall_clock_per_round_ms"] = 1000.0 * elapsed / max(
            1.0, row.get("rounds_completed", profile.rounds)
        )
        row["control_per_round"] = row.get("control_packets", 0.0) / max(
            1.0, row.get("rounds_completed", 1.0)
        )
        rows.append(row)
        print("  {0:44s} control/round={1:7.1f}  wall={2:5.1f}s".format(
            label[:44], row["control_per_round"], elapsed))

    bench = ecc.benchmark()
    bench["label"] = "ECC benchmark (standalone)"
    bench["policy_label"] = "ECC"
    rows.append(bench)
    print("  ECC: {0} backend, {1:.1f} us/packet, {2:.1f} us/key-agreement".format(
        bench["ecc_backend"], 1e6 * bench["bench_seconds_per_packet"],
        1e6 * bench["ecc_seconds_per_agreement"]))

    path = csv_path("e6_overhead.csv", out_dir=out_dir)
    write_rows(rows, path)
    print("  ->", path)
    return rows, path


EXPERIMENTS = {
    "e1": e1_speed,
    "e2": e2_density,
    "e3": e3_attack,
    "e4": e4_energy,
    "e5": e5_ablation,
    "e6": e6_overhead,
}


In [ ]:
%%writefile experiments/plots.py
"""Figures for the Review-2 report.

Reads the CSVs written by ``sweeps`` and renders the figures. Kept separate
from the experiments so plots can be regenerated without re-running anything,
and so the sweeps stay importable on a machine without matplotlib.
"""

import os

import numpy as np

from src.common import config

from .harness import as_float, out_path, read_rows

try:  # pragma: no cover - depends on the runtime environment
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    MATPLOTLIB_AVAILABLE = True
except Exception:  # pragma: no cover
    MATPLOTLIB_AVAILABLE = False

# One colour per implementation, used consistently across every figure.
COLOURS = {
    "AODV (reference baseline)": "#8c8c8c",
    "EAURP (base.pdf substrate)": "#6a9fb5",
    "A: DRL-EAURP (paper)": "#2f6f9f",
    "B: DRL-EAURP (senior code)": "#e08a1e",
    "C: AR-EAURP (advancement)": "#c0392b",
}
MARKERS = {
    "AODV (reference baseline)": "o",
    "EAURP (base.pdf substrate)": "s",
    "A: DRL-EAURP (paper)": "^",
    "B: DRL-EAURP (senior code)": "D",
    "C: AR-EAURP (advancement)": "v",
}
ORDER = list(COLOURS)

# Short forms for cramped bar-chart axes, so labels are never truncated
# mid-word into things like "EAURP (base.pd".
SHORT_NAMES = {
    "AODV (reference baseline)": "AODV",
    "EAURP (base.pdf substrate)": "EAURP",
    "A: DRL-EAURP (paper)": "A (paper)",
    "B: DRL-EAURP (senior code)": "B (senior)",
    "C: AR-EAURP (advancement)": "C (AR-EAURP)",
}


def _require():
    if not MATPLOTLIB_AVAILABLE:
        raise RuntimeError(
            "matplotlib is not available; figures cannot be rendered here"
        )


def _style(ax, title, xlabel, ylabel):
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(alpha=0.25, linewidth=0.6)
    ax.tick_params(labelsize=8)


def _series(rows, policy, x_key, y_key):
    subset = [row for row in rows if row.get("policy_label") == policy]
    if not subset:
        return None, None
    x = as_float(subset, x_key)
    y = as_float(subset, y_key)
    order = np.argsort(x)
    return x[order], y[order]


def _save(fig, name, out_dir=None):
    path = out_path("figures", name, out_dir=out_dir)
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print("  figure ->", path)
    return path


# --------------------------------------------------------------------------

def plot_speed(out_dir=None):
    """E1: the five metrics both papers report, against node speed."""
    _require()
    rows = read_rows(out_path("csv", "e1_speed.csv", out_dir=out_dir))
    if not rows:
        return None

    # Note the absence of a network-lifetime panel. Over a performance-sweep
    # run no node has drained yet, so "lifetime" would read as the run length
    # for every policy -- a flat line that looks like a result and is not one.
    # Lifetime gets its own long-horizon experiment; see plot_lifetime (E4b).
    panels = [
        ("pdr", "Packet Delivery Ratio", "PDR"),
        ("pdr_routable", "PDR among routable packets", "PDR (routable)"),
        ("avg_delay_ms", "Average end-to-end delay", "delay (ms)"),
        ("packet_loss_bytes", "Packet loss", "bytes lost"),
        ("throughput", "Throughput", "bits/round / 1000"),
        ("loss_no_route", "Packets with no route at all", "packets"),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, (key, title, ylabel) in zip(axes.ravel(), panels):
        for policy in ORDER:
            x, y = _series(rows, policy, "speed_value", key)
            if x is None:
                continue
            ax.plot(x, y, marker=MARKERS[policy], color=COLOURS[policy],
                    label=policy, linewidth=1.6, markersize=5)
        _style(ax, title, "node speed", ylabel)
    axes[0, 0].legend(fontsize=7, loc="best")
    fig.suptitle("E1 - Track 2: all implementations on one harness, identical "
                 "seeds.\nNetwork lifetime is measured separately over a long "
                 "horizon (E4b) - within this window no node has drained.",
                 fontsize=12)
    fig.tight_layout()
    return _save(fig, "e1_speed.png", out_dir=out_dir)


def plot_density(out_dir=None):
    """E2: performance against node count."""
    _require()
    rows = read_rows(out_path("csv", "e2_density.csv", out_dir=out_dir))
    if not rows:
        return None

    panels = [
        ("pdr", "Packet Delivery Ratio", "PDR"),
        ("loss_no_route", "Packets with no route at all", "packets"),
        ("avg_delay_ms", "Average delay", "delay (ms)"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
    for ax, (key, title, ylabel) in zip(axes, panels):
        for policy in ORDER:
            x, y = _series(rows, policy, "node_count", key)
            if x is None:
                continue
            ax.plot(x, y, marker=MARKERS[policy], color=COLOURS[policy],
                    label=policy, linewidth=1.6, markersize=5)
        _style(ax, title, "number of nodes", ylabel)
    axes[0].legend(fontsize=7)
    fig.suptitle("E2 - node density. The 'no route' panel is topology, not "
                 "protocol: at N=60 only ~39% of node pairs are connected at all.",
                 fontsize=11)
    fig.tight_layout()
    return _save(fig, "e2_density.png", out_dir=out_dir)


def plot_lifetime(out_dir=None):
    """E4b: network lifetime, over a horizon long enough for deaths to happen.

    Kept apart from E1 deliberately. Both papers quote a ~1230-round lifetime
    alongside a 0.96 delivery ratio, which is only possible because their energy
    model charges an idle drain and never charges transmission. Once per-hop
    TX/RX cost is modelled the two quantities need different time horizons, and
    reporting them from one sweep hides that.
    """
    _require()
    rows = read_rows(out_path("csv", "e4b_lifetime.csv", out_dir=out_dir))
    if not rows:
        return None

    modes = ["linear", "solar"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    for ax, (key, title, ylabel) in zip(axes, [
        ("network_lifetime", "Round of first node death", "rounds"),
        ("eighty_pct_dead_round", "Round at which 80% of nodes are dead",
         "rounds"),
    ]):
        width = 0.35
        positions = np.arange(len(ORDER))
        for offset, mode in enumerate(modes):
            values = []
            for policy in ORDER:
                subset = [
                    row for row in rows
                    if row.get("policy_label") == policy
                    and row.get("energy_mode") == mode
                ]
                values.append(
                    float(np.mean(as_float(subset, key))) if subset else 0.0
                )
            ax.bar(positions + offset * width, values, width,
                   label="{0} energy".format(mode),
                   color="#6a9fb5" if mode == "linear" else "#c0392b",
                   alpha=0.9)
        ax.set_xticks(positions + width / 2.0)
        ax.set_xticklabels([SHORT_NAMES.get(p, p) for p in ORDER],
                           rotation=15, ha="right", fontsize=8)
        _style(ax, title, "", ylabel)
        ax.legend(fontsize=8)
        # Bars that sit exactly on the horizon are censored, not measured.
        if values and max(values) >= config.LIFETIME_ROUNDS - 1:
            ax.axhline(config.LIFETIME_ROUNDS, color="#666666",
                       linestyle=":", linewidth=1.0)
            ax.annotate("bars touching the dotted line are censored:\n"
                        "the run ended before 80% of nodes had died",
                        xy=(0.03, 0.055), xycoords="axes fraction",
                        ha="left", fontsize=7.5, color="#333333",
                        bbox=dict(boxstyle="round,pad=0.35", fc="#ffffff",
                                  ec="#888888", lw=0.6, alpha=0.92))
    fig.suptitle("E4b - network lifetime over a long horizon. Solar harvesting "
                 "is the advancement's replacement for Eq. (8).", fontsize=11)
    fig.tight_layout()
    return _save(fig, "e4b_lifetime.png", out_dir=out_dir)


def plot_attack(out_dir=None):
    """E3: the headline figure -- PDR and detection against adversary strength."""
    _require()
    rows = read_rows(out_path("csv", "e3_attack.csv", out_dir=out_dir))
    if not rows:
        return None

    attacks = ["blackhole", "grayhole", "trust_poisoning"]
    titles = {
        "blackhole": "Black-hole (drops 100%)",
        "grayhole": "Gray-hole (drops ~15%, PFR stays 0.84)",
        "trust_poisoning": "Trust poisoning (gray-hole + slander)",
    }

    fig, axes = plt.subplots(3, 3, figsize=(16, 12))
    for column, attack in enumerate(attacks):
        subset = [row for row in rows if row.get("attack_type") == attack]
        for row_index, (key, ylabel) in enumerate([
            ("pdr", "PDR"),
            ("det_tpr", "detection recall (TPR)"),
            ("det_fpr", "false-positive rate (FPR)"),
        ]):
            ax = axes[row_index, column]
            for policy in ORDER:
                x, y = _series(subset, policy, "malicious_pct", key)
                if x is None:
                    continue
                ax.plot(x, y, marker=MARKERS[policy], color=COLOURS[policy],
                        label=policy, linewidth=1.8, markersize=5)
            title = titles[attack] if row_index == 0 else ""
            _style(ax, title, "malicious nodes (%)", ylabel)
            if key in ("det_tpr", "det_fpr", "pdr"):
                ax.set_ylim(-0.03, 1.03)
    axes[0, 0].legend(fontsize=7, loc="best")

    fig.suptitle(
        "E3 - ATTACK SWEEP. Middle row is the point of the advancement: under "
        "gray-hole,\nevery prior implementation detects nothing at any attack "
        "level (TPR = 0).",
        fontsize=12,
    )
    fig.tight_layout()
    return _save(fig, "e3_attack.png", out_dir=out_dir)


def plot_ablation(out_dir=None):
    """E5: which component of C is doing the work."""
    _require()
    rows = read_rows(out_path("csv", "e5_ablation.csv", out_dir=out_dir))
    if not rows:
        return None

    labels = [row.get("label", "") for row in rows]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
    for ax, (key, title, ylabel) in zip(axes, [
        ("pdr", "Packet Delivery Ratio", "PDR"),
        ("det_tpr", "Detection recall", "TPR"),
        ("det_fpr", "False-positive rate", "FPR"),
    ]):
        values = as_float(rows, key)
        positions = np.arange(len(labels))
        bars = ax.bar(positions, values, color="#c0392b", alpha=0.85)
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
        # An all-zero column must read as a measured zero, not as a broken
        # axis: with no floor, matplotlib auto-ranges to +/-0.05 and draws
        # nothing at all, which looks like a rendering failure.
        top = max(0.05, float(values.max()) * 1.25) if values.size else 1.0
        ax.set_ylim(0.0, top)
        for bar, value in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2.0,
                    bar.get_height() + top * 0.02,
                    "{0:.3f}".format(value), ha="center", fontsize=7)
        _style(ax, title, "", ylabel)
    fig.suptitle("E5 - ablation of C under gray-hole @ 30%", fontsize=12)
    fig.tight_layout()
    figure = _save(fig, "e5_ablation.png", out_dir=out_dir)

    floors = read_rows(out_path("csv", "e5b_cmdp_floor.csv", out_dir=out_dir))
    if floors:
        fig, ax = plt.subplots(figsize=(7, 4.4))
        x = as_float(floors, "pdr_floor")
        achieved = as_float(floors, "pdr")
        violations = as_float(floors, "cmdp_violation_rate")
        order = np.argsort(x)
        ax.plot(x[order], achieved[order], marker="o", color="#c0392b",
                label="achieved PDR")
        ax.plot(x[order], x[order], linestyle="--", color="#666666",
                label="requested floor")
        ax.plot(x[order], violations[order], marker="s", color="#e08a1e",
                label="constraint violation rate")
        _style(ax, "E5b - is the CMDP floor reachable?", "requested PDR floor",
               "value")
        ax.legend(fontsize=8)
        fig.tight_layout()
        _save(fig, "e5b_cmdp_floor.png", out_dir=out_dir)
    return figure


def plot_track1_comparison(out_dir=None):
    """Track 1: A (from the paper) beside B (the delivered code)."""
    _require()
    a_rows = read_rows(out_path("csv", "a_paper_track1.csv", out_dir=out_dir))
    b_rows = read_rows(out_path("csv", "b_senior_as_is.csv", out_dir=out_dir))
    if not a_rows and not b_rows:
        return None

    models = ["Existing", "EAURP", "ATEAURP", "PSE-EAURP", "DRL-EAURP"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

    for ax, (key, ylabel) in zip(axes, [
        ("pdr", "PDR"),
        ("avg_delay_ms", "delay (ms)"),
        ("network_lifetime", "lifetime (rounds)"),
    ]):
        for source, rows, style in (
            ("A (from paper)", a_rows, "-"),
            ("B (senior code)", b_rows, "--"),
        ):
            if not rows:
                continue
            values = []
            for model in models:
                subset = [
                    row for row in rows
                    if row.get("model") == model
                    and row.get("sweep", "speed") == "speed"
                ]
                values.append(float(np.mean(as_float(subset, key))) if subset else np.nan)
            ax.plot(models, values, style, marker="o",
                    label=source, linewidth=1.8, markersize=5)
            # A's lifetime is censored whenever it comes out flat at the run
            # length: under the quick profile A stops at 600 rounds, before
            # anything has drained, while B always runs the senior's hardcoded
            # 2000. Saying so on the figure stops it being read as a result.
            if key == "network_lifetime" and source.startswith("A"):
                finite = [v for v in values if np.isfinite(v)]
                if finite and (max(finite) - min(finite)) < 0.05 * max(finite):
                    ax.annotate(
                        "A censored: no node died within\n"
                        "the run (use --profile full)",
                        xy=(0.5, 0.12), xycoords="axes fraction",
                        ha="center", fontsize=7.5, color="#2f6f9f",
                        bbox=dict(boxstyle="round,pad=0.3", fc="#eef4fa",
                                  ec="#2f6f9f", lw=0.6))
        _style(ax, {"pdr": "Packet Delivery Ratio",
                    "avg_delay_ms": "Average delay",
                    "network_lifetime": "Network lifetime"}.get(key, key),
               "", ylabel)
        ax.tick_params(axis="x", rotation=20)
    axes[0].legend(fontsize=8)
    fig.suptitle("Track 1 - A (Lekha.pdf equations) vs B (delivered code). "
                 "'Existing' is derived arithmetic in both (Eq. 14-18), "
                 "not a protocol.", fontsize=11)
    fig.tight_layout()
    return _save(fig, "track1_a_vs_b.png", out_dir=out_dir)


def plot_gan_training(detector, out_dir=None):
    """GAN training curves, straight from a fitted detector."""
    _require()
    history = getattr(detector, "history", None)
    if not history or not history.get("d_loss"):
        print("  (no GAN training history -- detector used its fallback)")
        return None
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(history["d_loss"], label="discriminator", color="#2f6f9f", linewidth=1.2)
    ax.plot(history["g_loss"], label="generator", color="#c0392b", linewidth=1.2)
    _style(ax, "GAN training (honest behaviour only)", "epoch", "loss")
    ax.legend(fontsize=8)
    fig.tight_layout()
    return _save(fig, "c_gan_training.png", out_dir=out_dir)


def plot_lstm_accuracy(predictor, out_dir=None):
    """LSTM forecast error beside both naive baselines."""
    _require()
    evaluation = getattr(predictor, "evaluation", None)
    if not evaluation:
        return None
    fallback = bool(getattr(predictor, "using_fallback", False))
    names = ["LSTM (fallback:\npersistence)" if fallback else "LSTM",
             "persistence", "seasonal naive"]
    values = [
        evaluation.get("lstm_mae", 0.0),
        evaluation.get("persistence_mae", 0.0),
        evaluation.get("seasonal_naive_mae", 0.0),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    bars = axes[0].bar(names, values, color=["#c0392b", "#8c8c8c", "#6a9fb5"])
    top = max(values) * 1.25 if max(values) > 0 else 1.0
    axes[0].set_ylim(0.0, top)
    for bar, value in zip(bars, values):
        axes[0].text(bar.get_x() + bar.get_width() / 2.0,
                     bar.get_height() + top * 0.02,
                     "{0:.5f}".format(value), ha="center", fontsize=7.5)
    title = "Harvest forecast error (lower is better)"
    if fallback:
        # Without torch the "model" *is* persistence, so the first two bars are
        # the same number. Say so, or the figure reads as a trained LSTM that
        # merely tied with a naive baseline.
        title += "\nno torch: the first bar IS persistence, not a trained LSTM"
    _style(axes[0], title, "", "MAE")

    history = getattr(predictor, "history", None)
    if history:
        axes[1].plot(history, color="#c0392b", linewidth=1.3)
        _style(axes[1], "LSTM training loss", "epoch", "MSE")
    else:
        axes[1].text(0.5, 0.5, "no training history\n(fallback in use)",
                     ha="center", va="center", fontsize=10)
        axes[1].axis("off")
    fig.tight_layout()
    return _save(fig, "c_lstm_accuracy.png", out_dir=out_dir)


def plot_all(out_dir=None):
    """Render every figure whose CSV exists."""
    _require()
    made = []
    for renderer in (plot_speed, plot_density, plot_attack, plot_lifetime,
                     plot_ablation, plot_track1_comparison):
        try:
            path = renderer(out_dir=out_dir)
            if path:
                made.append(path)
        except Exception as error:  # keep going; a missing sweep is not fatal
            print("  [warn] {0} failed: {1}".format(renderer.__name__, error))
    return made


In [ ]:
%%writefile experiments/report.py
"""The A vs B vs C comparison, and the caveats that make it honest.

Reads the CSVs produced by the sweeps and assembles the comparison tables plus
the "what is and isn't comparable" section. Everything printed here is derived
from measured output; nothing is typed in by hand.
"""

import os

import numpy as np

from .harness import as_float, out_path, read_rows

IMPLEMENTATIONS = [
    "A: DRL-EAURP (paper)",
    "B: DRL-EAURP (senior code)",
    "C: AR-EAURP (advancement)",
]
REFERENCES = ["AODV (reference baseline)", "EAURP (base.pdf substrate)"]


def _mean(rows, key, default=float("nan")):
    if not rows:
        return default
    values = as_float(rows, key)
    return float(np.mean(values)) if values.size else default


def _filter(rows, **conditions):
    out = []
    for row in rows:
        keep = True
        for key, value in conditions.items():
            actual = row.get(key)
            if isinstance(value, float):
                try:
                    keep = keep and abs(float(actual) - value) < 1e-6
                except (TypeError, ValueError):
                    keep = False
            else:
                keep = keep and str(actual) == str(value)
            if not keep:
                break
        if keep:
            out.append(row)
    return out


def build_comparison(out_dir=None):
    """Collect every comparison the report needs into one dict."""
    report = {"available": {}, "caveats": []}

    files = {
        "e1": "e1_speed.csv",
        "e2": "e2_density.csv",
        "e3": "e3_attack.csv",
        "e4": "e4_energy.csv",
        "e4b": "e4b_lifetime.csv",
        "e5": "e5_ablation.csv",
        "e5b": "e5b_cmdp_floor.csv",
        "e6": "e6_overhead.csv",
        "a1": "a_paper_track1.csv",
        "b1": "b_senior_as_is.csv",
        "drain": "a_drain_band_ablation.csv",
    }
    data = {}
    for key, name in files.items():
        path = out_path("csv", name, out_dir=out_dir)
        data[key] = read_rows(path)
        report["available"][key] = bool(data[key])
    report["_data"] = data

    # -- Track 1 headline numbers -----------------------------------------
    track1 = {}
    for label, rows, model_key in (("A", data["a1"], "model"),
                                   ("B", data["b1"], "model")):
        if not rows:
            continue
        block = {}
        for model in ("Existing", "EAURP", "ATEAURP", "PSE-EAURP", "DRL-EAURP"):
            subset = [
                row for row in rows
                if row.get(model_key) == model
                and row.get("sweep", "speed") == "speed"
            ]
            if subset:
                block[model] = {
                    "pdr": _mean(subset, "pdr"),
                    "delay": _mean(subset, "avg_delay_ms"),
                    "lifetime": _mean(subset, "network_lifetime"),
                }
        track1[label] = block
    report["track1"] = track1

    if data["drain"]:
        bands = {row.get("band"): row for row in data["drain"]}
        paper = bands.get("paper_Eq8_0.05-0.15")
        code = bands.get("code_0.04-0.12")
        if paper and code:
            report["drain_band"] = {
                "paper_lifetime": float(paper.get("network_lifetime", 0.0)),
                "code_lifetime": float(code.get("network_lifetime", 0.0)),
                "paper_pdr": float(paper.get("pdr", 0.0)),
                "code_pdr": float(code.get("pdr", 0.0)),
                "delta": float(code.get("network_lifetime", 0.0))
                - float(paper.get("network_lifetime", 0.0)),
            }

    # -- Track 2 clean baseline -------------------------------------------
    clean = {}
    for policy in REFERENCES + IMPLEMENTATIONS:
        subset = _filter(data["e1"], policy_label=policy)
        if subset:
            clean[policy] = {
                "pdr": _mean(subset, "pdr"),
                "pdr_routable": _mean(subset, "pdr_routable"),
                "delay": _mean(subset, "avg_delay_ms"),
                "throughput": _mean(subset, "throughput"),
            }
    report["track2_clean"] = clean

    # -- Track 2 under attack ---------------------------------------------
    attacks = {}
    for attack in ("blackhole", "grayhole", "trust_poisoning"):
        per_policy = {}
        for policy in REFERENCES + IMPLEMENTATIONS:
            points = {}
            for pct in (0.0, 10.0, 20.0, 30.0, 40.0, 50.0):
                subset = _filter(data["e3"], policy_label=policy,
                                 attack_type=attack, malicious_pct=pct)
                if subset:
                    points[pct] = {
                        "pdr": _mean(subset, "pdr"),
                        "tpr": _mean(subset, "det_tpr"),
                        "fpr": _mean(subset, "det_fpr"),
                    }
            if points:
                per_policy[policy] = points
        attacks[attack] = per_policy
    report["track2_attack"] = attacks

    # -- ablation and CMDP feasibility ------------------------------------
    if data["e5"]:
        report["ablation"] = [
            {
                "label": row.get("label"),
                "pdr": float(row.get("pdr", 0.0) or 0.0),
                "tpr": float(row.get("det_tpr", 0.0) or 0.0),
                "fpr": float(row.get("det_fpr", 0.0) or 0.0),
            }
            for row in data["e5"]
        ]
    if data["e5b"]:
        report["cmdp_floor"] = [
            {
                "floor": float(row.get("pdr_floor", 0.0) or 0.0),
                "achieved": float(row.get("pdr", 0.0) or 0.0),
                "violation_rate": float(row.get("cmdp_violation_rate", 0.0) or 0.0),
            }
            for row in data["e5b"]
        ]
    if data["e6"]:
        report["overhead"] = [
            {
                "label": row.get("label"),
                "control_per_round": float(row.get("control_per_round", 0.0) or 0.0),
                "wall_clock": float(row.get("wall_clock_seconds", 0.0) or 0.0),
            }
            for row in data["e6"] if row.get("control_per_round")
        ]
    return report


# --------------------------------------------------------------------------

def _table(headers, rows, widths=None):
    widths = widths or [max(12, len(h)) for h in headers]
    line = "  ".join(h.ljust(w) for h, w in zip(headers, widths))
    out = [line, "-" * len(line)]
    for row in rows:
        out.append("  ".join(str(c).ljust(w) for c, w in zip(row, widths)))
    return "\n".join(out)


def print_report(report):
    """Render the comparison to stdout."""
    print("=" * 78)
    print("A vs B vs C -- RESULTS")
    print("=" * 78)

    # ---- Track 1 ----
    if report.get("track1"):
        print()
        print("TRACK 1 -- each implementation in its native form")
        print()
        rows = []
        for model in ("Existing", "EAURP", "ATEAURP", "PSE-EAURP", "DRL-EAURP"):
            a = report["track1"].get("A", {}).get(model, {})
            b = report["track1"].get("B", {}).get(model, {})
            rows.append([
                model,
                "{:.3f}".format(a.get("pdr", float("nan"))),
                "{:.3f}".format(b.get("pdr", float("nan"))),
                "{:.1f}".format(a.get("delay", float("nan"))),
                "{:.1f}".format(b.get("delay", float("nan"))),
                "{:.0f}".format(a.get("lifetime", float("nan"))),
                "{:.0f}".format(b.get("lifetime", float("nan"))),
            ])
        print(_table(
            ["model", "A PDR", "B PDR", "A delay", "B delay", "A life", "B life"],
            rows, [12, 8, 8, 9, 9, 8, 8]))
        print()
        print("  'Existing' is Lekha.pdf Eq. (14)-(18): EAURP's own output times")
        print("  fixed constants. It is arithmetic, not a protocol, in both.")

    if report.get("drain_band"):
        band = report["drain_band"]
        print()
        print("  Where B's lifetime advantage comes from:")
        print("    Eq. (8) band [0.05,0.15]:  lifetime {:7.1f}  PDR {:.4f}".format(
            band["paper_lifetime"], band["paper_pdr"]))
        print("    code band   [0.04,0.12]:  lifetime {:7.1f}  PDR {:.4f}".format(
            band["code_lifetime"], band["code_pdr"]))
        print("    => {:+.1f} rounds from the undocumented constant alone.".format(
            band["delta"]))

    # ---- Track 2 clean ----
    if report.get("track2_clean"):
        print()
        print("TRACK 2 -- clean network, identical seeds and worlds")
        print()
        rows = []
        for policy, stats in report["track2_clean"].items():
            rows.append([
                policy[:28],
                "{:.3f}".format(stats["pdr"]),
                "{:.3f}".format(stats["pdr_routable"]),
                "{:.1f}".format(stats["delay"]),
                "{:.3f}".format(stats["throughput"]),
            ])
        print(_table(["implementation", "PDR", "PDR routable", "delay ms",
                      "throughput"], rows, [30, 8, 13, 9, 11]))

    # ---- Track 2 attack ----
    attacks = report.get("track2_attack") or {}
    for attack, per_policy in attacks.items():
        if not per_policy:
            continue
        print()
        print("TRACK 2 -- {0}: PDR / detection recall at each adversary "
              "fraction".format(attack))
        print()
        percentages = [0.0, 10.0, 20.0, 30.0, 40.0, 50.0]
        header = ["implementation"] + ["{:.0f}%".format(p) for p in percentages]
        rows = []
        for policy, points in per_policy.items():
            rows.append([policy[:28]] + [
                "{:.2f}/{:.2f}".format(points[p]["pdr"], points[p]["tpr"])
                if p in points else "-"
                for p in percentages
            ])
        print(_table(header, rows, [30] + [10] * len(percentages)))

    grayhole = attacks.get("grayhole", {})
    if grayhole:
        print()
        print("  The gray-hole column is the point of the advancement.")
        for policy, points in grayhole.items():
            recalls = [points[p]["tpr"] for p in points if p > 0]
            if recalls:
                print("    {:30s} mean detection recall under attack: {:.2f}".format(
                    policy[:30], float(np.mean(recalls))))

    # ---- ablation ----
    if report.get("ablation"):
        print()
        print("ABLATION of C (gray-hole @ 30%)")
        print()
        print(_table(["variant", "PDR", "TPR", "FPR"],
                     [[r["label"][:24], "{:.3f}".format(r["pdr"]),
                       "{:.3f}".format(r["tpr"]), "{:.3f}".format(r["fpr"])]
                      for r in report["ablation"]], [26, 8, 8, 8]))

    if report.get("cmdp_floor"):
        print()
        print("CMDP FEASIBILITY -- is the doc's 90% PDR floor reachable?")
        print()
        print(_table(["requested floor", "achieved PDR", "violation rate"],
                     [["{:.2f}".format(r["floor"]),
                       "{:.3f}".format(r["achieved"]),
                       "{:.3f}".format(r["violation_rate"])]
                      for r in report["cmdp_floor"]], [16, 14, 15]))

    if report.get("overhead"):
        print()
        print("OVERHEAD")
        print()
        print(_table(["implementation", "control pkts/round", "wall clock s"],
                     [[r["label"][:30],
                       "{:.1f}".format(r["control_per_round"]),
                       "{:.1f}".format(r["wall_clock"])]
                      for r in report["overhead"]], [32, 19, 13]))

    print()
    print("=" * 78)
    print("WHAT IS AND ISN'T COMPARABLE -- read before quoting any number")
    print("=" * 78)
    print(CAVEATS.strip())


CAVEATS = """
1. TRACK 1 AND TRACK 2 ARE NOT THE SAME EXPERIMENT.
   Track 1 numbers come from the papers' own probabilistic model: no routes, no
   relays, delivery decided by one coin flip against a global average. Track 2
   numbers come from a mechanistic simulator that walks every packet hop by hop
   through a real topology. A Track 1 PDR of 0.96 and a Track 2 PDR of 0.79 are
   not in disagreement; they are measuring different things. Compare within a
   track, never across.

2. B'S HEADLINE NUMBERS ARE PARTLY LITERALS.
   Cell 13 of senior_code.ipynb prints random.uniform values formatted as
   measured metrics (PDR 89.04, delay 498.64 ms, throughput 17.46 kbps,
   lifetime 99.28%). Those exact figures appear in Lekha.pdf Sec. IV-A-6 as
   "observed behaviour". Re-running the cell produces different values, because
   nothing is being measured. The simulated curves (the five models' sweeps)
   are genuinely computed; the summary block is not.

3. "EXISTING" IS NOT A PROTOCOL.
   Lekha.pdf Eq. (14)-(18) derives it by multiplying EAURP's own output by
   fixed constants. Every "improvement over Existing" is therefore arithmetic
   by construction. Track 2 replaces it with a real AODV implementation.

4. B'S LIFETIME ADVANTAGE IS A CONSTANT, NOT A PROTOCOL.
   Eq. (8) specifies one drain band. The delivered code uses a gentler one for
   the three proposed models. Running the same model under both bands isolates
   the effect; see the drain-band ablation above.

5. C IS EXPECTED TO LOOK WORSE THAN B ON RAW PDR IN TRACK 1.
   B computes delivery from a formula; C routes real packets through real
   adversaries across a topology in which about 9% of node pairs are not even
   connected. The comparison that means something is the Track 2 attack sweep,
   where both face the same world.

6. THE CHANNEL MODEL IS OUR ASSUMPTION.
   Neither paper contains a PHY or link model, so the per-hop reliability in
   Track 2 is a modelling choice, calibrated so a clean network lands in the
   same PDR regime the papers report. Conclusions rest on relative comparisons
   under identical channel settings, never on absolute values.

7. THE 90% PDR FLOOR IN THE ADVANCEMENT DOC MAY BE INFEASIBLE.
   The doc asks for PDR >= 0.90 under 30% adversarial nodes. On a topology
   where ~9% of pairs are unreachable before any attack, that ceiling cannot be
   reached, and the Lagrangian multiplier rises without ever satisfying the
   constraint. The CMDP feasibility table above measures where the real ceiling
   is instead of assuming one. This confirms the risk flagged in the Review 1
   critique.

8. A'S TRACK 1 LIFETIME IS CENSORED UNDER THE QUICK PROFILE.
   B always runs the senior's hardcoded SIM_ROUNDS = 2000. A runs 2000 only
   under --profile full; the quick profile stops it at 600, before any node has
   drained, so A's lifetime column reads as the run length rather than a first
   death. Compare the A and B lifetime columns only from a full-profile run --
   or use the drain-band ablation above, which controls for this directly.

9. DETECTION RATES ARE MEASURED AGAINST GROUND TRUTH.
   TPR and FPR compare each protocol's own verdicts with the known roles
   assigned by the threat model. B reports 0.00 everywhere because the
   delivered code sets node.malicious and never reads it - that is the honest
   consequence of the code, not a failure to measure it.
"""


def write_markdown(report, path):
    """Persist the comparison as a Markdown file."""
    import io

    buffer = io.StringIO()
    import contextlib

    with contextlib.redirect_stdout(buffer):
        print_report(report)

    directory = os.path.dirname(os.path.abspath(path))
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        handle.write("# AR-EAURP Review 2 - Results\n\n")
        handle.write("```\n")
        handle.write(buffer.getvalue())
        handle.write("\n```\n")
    return path


In [ ]:
%%writefile experiments/run_all.py
"""Run the whole Review-2 evaluation.

    python experiments/run_all.py --profile quick
    python experiments/run_all.py --profile full
    python experiments/run_all.py --only e3 --profile full

Track 1 (each implementation in its native form) and Track 2 (all three on one
mechanistic harness) are both produced. Sweeps checkpoint to CSV as they finish
and are skipped on a re-run unless ``--fresh`` is given, so a dropped Colab
session resumes instead of restarting.
"""

import argparse
import os
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from experiments import sweeps  # noqa: E402
from experiments.harness import csv_path, pretrained_c, write_rows  # noqa: E402
from src.common import config  # noqa: E402


def run_track1(profile, out_dir=None, skip_b=False, verbose=True):
    """A from the paper's equations, B from the delivered code."""
    print("=" * 78)
    print("TRACK 1 -- each implementation in its native form")
    print("=" * 78)

    from src.a_drl_eaurp import run_track1 as a_track1

    rounds = 2000 if profile.name == "full" else 600
    runs = 3 if profile.name == "full" else 2

    print("\nA: DRL-EAURP re-implemented from Lekha.pdf Eq. (1)-(38)")
    speed_results, policy_notes = a_track1.run_speed_sweep(rounds=rounds, runs=runs)
    density_results = a_track1.run_density_sweep(rounds=rounds, runs=runs,
                                                 verbose=False)
    rows = a_track1.to_rows(speed_results, density_results)
    a_track1.write_csv(rows, csv_path("a_paper_track1.csv", out_dir=out_dir))

    print("\nA: drain-band ablation -- isolating Eq. (8) from the delivered code")
    ablation = a_track1.drain_band_ablation(rounds=1400, runs=runs)
    print("   Eq. (8) band [0.05,0.15]: lifetime {0:7.1f}  PDR {1:.4f}".format(
        ablation["paper_Eq8_0.05-0.15"]["network_lifetime"],
        ablation["paper_Eq8_0.05-0.15"]["pdr"]))
    print("   code band  [0.04,0.12]: lifetime {0:7.1f}  PDR {1:.4f}".format(
        ablation["code_0.04-0.12"]["network_lifetime"],
        ablation["code_0.04-0.12"]["pdr"]))
    print("   => {0:+.1f} rounds of 'lifetime improvement' from the constant "
          "alone".format(ablation["lifetime_delta"]))
    write_rows(
        [
            dict(band=key, **value)
            for key, value in ablation.items()
            if isinstance(value, dict)
        ],
        csv_path("a_drain_band_ablation.csv", out_dir=out_dir),
    )

    if not skip_b:
        print("\nB: senior_code.ipynb, logic verbatim")
        from src.b_senior import senior_asis

        senior_results = senior_asis.run_all(runs=5 if profile.name == "full" else 2)
        senior_asis.write_csv(
            senior_asis.to_rows(senior_results),
            csv_path("b_senior_as_is.csv", out_dir=out_dir),
        )
        for model in ("Existing", "EAURP", "ATEAURP", "PSE-EAURP", "DRL-EAURP"):
            block = senior_results[model]
            print("   B/{0:11s} PDR {1:.3f}-{2:.3f}".format(
                model, min(block["pdr"]), max(block["pdr"])))
    return True


def run_track2(profile, only=None, out_dir=None, fresh=False, verbose=True):
    """All policies on the shared mechanistic harness."""
    print()
    print("=" * 78)
    print("TRACK 2 -- A, B and C as routing policies on one harness")
    print("=" * 78)

    print("\nPre-training C (clean commissioning run, no adversaries present)")
    shared_c, pretrain_summary = pretrained_c(profile, verbose=verbose)
    write_rows([pretrain_summary], csv_path("c_pretrain.csv", out_dir=out_dir))

    wanted = list(sweeps.EXPERIMENTS) if not only else [key.lower() for key in only]
    kwargs = dict(out_dir=out_dir, resume=not fresh, verbose=verbose)

    for key in wanted:
        runner = sweeps.EXPERIMENTS.get(key)
        if runner is None:
            print("  [warn] unknown experiment {0!r}".format(key))
            continue
        print()
        if key == "e5":
            runner(profile, **kwargs)
        else:
            runner(profile, shared_c=shared_c, **kwargs)
    return shared_c


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--profile", default="quick", choices=("quick", "full"))
    parser.add_argument("--only", nargs="*", default=None,
                        help="subset of e1..e6 to run")
    parser.add_argument("--out-dir", default=None,
                        help="results root (use a Drive path on Colab)")
    parser.add_argument("--fresh", action="store_true",
                        help="ignore existing CSVs and recompute everything")
    parser.add_argument("--skip-track1", action="store_true")
    parser.add_argument("--skip-track2", action="store_true")
    parser.add_argument("--skip-b", action="store_true",
                        help="skip B's slow verbatim run (it takes ~200 s)")
    args = parser.parse_args(argv)

    profile = config.get_profile(args.profile)
    started = time.perf_counter()

    print("AR-EAURP Review 2 -- profile={0} rounds={1} runs={2}".format(
        profile.name, profile.rounds, profile.runs))

    if not args.skip_track1:
        run_track1(profile, out_dir=args.out_dir, skip_b=args.skip_b)
    if not args.skip_track2:
        run_track2(profile, only=args.only, out_dir=args.out_dir, fresh=args.fresh)

    print()
    print("=" * 78)
    print("done in {0:.1f} s".format(time.perf_counter() - started))
    print("=" * 78)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile tests/test_harness.py
"""Unit tests for the mechanistic harness.

These run with numpy alone -- no torch, no matplotlib -- so they can be
executed on any machine before the notebook is ever opened in Colab.

Run with:  python -m pytest tests/ -q
"""

import os
import sys

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from src.common import (  # noqa: E402
    adversary, config, energy, metrics, mobility, node as node_mod, seeding,
    topology, traffic,
)
from src.common.simulator import SimParams, Simulation  # noqa: E402
from src.routing import aodv, ecc, trust  # noqa: E402
from src.routing.eaurp import AodvPolicy, EaurpPolicy  # noqa: E402


def build_nodes(n=100, seed=11, speed=20000):
    bundle = seeding.make_seed_bundle(seed)
    nodes = node_mod.NodeState(n, bundle.topology)
    mobility.assign_speeds(nodes, speed)
    topology.rebuild(nodes)
    return nodes, bundle


# --------------------------------------------------------------------------
# Topology and mobility
# --------------------------------------------------------------------------

def test_adjacency_is_symmetric_and_loopless():
    nodes, _ = build_nodes()
    assert (nodes.adjacency == nodes.adjacency.T).all()
    assert not np.diagonal(nodes.adjacency).any()


def test_links_respect_communication_range():
    nodes, _ = build_nodes()
    distances = topology.pairwise_distances(nodes)
    linked = nodes.adjacency
    assert distances[linked].max() <= config.COMM_RANGE + 1e-9
    unlinked = (~linked) & ~np.eye(nodes.n, dtype=bool)
    assert distances[unlinked].min() > config.COMM_RANGE - 1e-9


def test_mobility_keeps_nodes_inside_the_area():
    nodes, bundle = build_nodes()
    for _ in range(50):
        mobility.step(nodes, bundle.mobility)
    assert nodes.x.min() >= 0.0 and nodes.x.max() <= config.AREA_SIZE
    assert nodes.y.min() >= 0.0 and nodes.y.max() <= config.AREA_SIZE


def test_dead_nodes_leave_the_topology():
    nodes, _ = build_nodes()
    nodes.alive[5] = False
    nodes.revoked[6] = True
    topology.rebuild(nodes)
    assert not nodes.adjacency[5].any() and not nodes.adjacency[:, 5].any()
    assert not nodes.adjacency[6].any() and not nodes.adjacency[:, 6].any()


# --------------------------------------------------------------------------
# Routing
# --------------------------------------------------------------------------

def test_routes_are_loop_free_and_valid():
    nodes, _ = build_nodes()
    scores = aodv.route_scores(nodes)
    rng = np.random.default_rng(3)
    found = 0
    for _ in range(150):
        src, dst = (int(v) for v in rng.integers(0, nodes.n, 2))
        if src == dst:
            continue
        path = aodv.find_route(nodes, src, dst, scores)
        if path is None:
            continue
        found += 1
        assert len(set(path)) == len(path), "route contains a loop"
        assert topology.path_is_valid(nodes, path)
        assert path[0] == src and path[-1] == dst
    assert found > 20, "expected to find routes in a connected network"


def test_energy_threshold_excludes_relays():
    """base.pdf 3.5 -- nodes below 20% of initial energy must not be relays."""
    nodes, _ = build_nodes()
    scores = aodv.route_scores(nodes)
    drained = 7
    nodes.energy[drained] = 0.05 * config.INITIAL_ENERGY
    rng = np.random.default_rng(5)
    for _ in range(200):
        src, dst = (int(v) for v in rng.integers(0, nodes.n, 2))
        if src == dst or drained in (src, dst):
            continue
        path = aodv.find_route(nodes, src, dst, scores)
        if path:
            assert drained not in path[1:-1]


def test_revoked_nodes_are_never_relays():
    nodes, _ = build_nodes()
    nodes.revoked[9] = True
    topology.rebuild(nodes)
    scores = aodv.route_scores(nodes)
    rng = np.random.default_rng(7)
    for _ in range(200):
        src, dst = (int(v) for v in rng.integers(0, nodes.n, 2))
        if src == dst:
            continue
        path = aodv.find_route(nodes, src, dst, scores)
        if path:
            assert 9 not in path


def test_score_coefficients_sum_to_one():
    """base.pdf 3.3 -- a = 1 - avg/max, b = avg/max."""
    nodes, _ = build_nodes()
    a, b = aodv.score_coefficients(nodes)
    assert abs((a + b) - 1.0) < 1e-9
    assert 0.0 <= a <= 1.0 and 0.0 <= b <= 1.0


def test_route_cache_invalidates_on_broken_link():
    nodes, _ = build_nodes()
    scores = aodv.route_scores(nodes)
    cache = aodv.RouteCache()
    path = None
    rng = np.random.default_rng(13)
    while path is None or len(path) < 3:
        src, dst = (int(v) for v in rng.integers(0, nodes.n, 2))
        if src == dst:
            continue
        path = aodv.find_route(nodes, src, dst, scores)
    cache.put(path[0], path[-1], path)
    assert cache.get(nodes, path[0], path[-1]) == path
    nodes.alive[path[1]] = False
    assert cache.get(nodes, path[0], path[-1]) is None


# --------------------------------------------------------------------------
# Trust and the threat model
# --------------------------------------------------------------------------

def test_pfr_arithmetic():
    nodes, _ = build_nodes(n=10)

    class Pkt(object):
        size = 800

        @property
        def is_large(self):
            return True

    for index in range(10):
        trust.observe_forward(nodes, 0, 1, Pkt(), index < 6)
    assert abs(nodes.observed_pfr()[0, 1] - 0.6) < 1e-9
    # Unobserved pairs stay fully trusted (base.pdf only penalises what it saw).
    assert nodes.observed_pfr()[0, 2] == 1.0


def test_grayhole_evades_the_trust_threshold():
    """The core premise of the advancement.

    A gray-hole that drops only large packets must keep its *observed* Packet
    Forwarding Ratio comfortably above base.pdf's 0.6 revocation threshold --
    otherwise there would be nothing for the GAN detector to add.
    """
    nodes, bundle = build_nodes(n=20)
    nodes.role[3] = node_mod.FLAG_GRAYHOLE

    class Pkt(object):
        def __init__(self, size):
            self.size = size

        @property
        def is_large(self):
            return self.size >= config.LARGE_PACKET_THRESHOLD

    rng = np.random.default_rng(21)
    for _ in range(4000):
        size = int(rng.integers(config.PACKET_SIZE_MIN, config.PACKET_SIZE_MAX + 1))
        packet = Pkt(size)
        relayed = adversary.forwards_data(nodes, 3, packet, bundle.adversary)
        trust.observe_forward(nodes, 0, 3, packet, relayed)

    pfr = nodes.observed_pfr()[0, 3]
    assert 0.75 < pfr < 0.92, "observed PFR was {0:.3f}".format(pfr)
    assert pfr > config.TRUST_THRESHOLD, "gray-hole must evade the 0.6 threshold"

    # ... yet the size-conditioned view separates cleanly, which is exactly the
    # signal the GAN detector is given.
    large_pfr = nodes.fwd_seen_large[0, 3] / nodes.sent_large[0, 3]
    small_pfr = nodes.fwd_seen_small[0, 3] / nodes.sent_small[0, 3]
    assert small_pfr == 1.0
    assert large_pfr < 0.85
    assert (small_pfr - large_pfr) > 0.15


def test_blackhole_drops_everything():
    nodes, bundle = build_nodes(n=10)
    nodes.role[4] = node_mod.FLAG_BLACKHOLE

    class Pkt(object):
        size = 600
        is_large = False

    assert not any(
        adversary.forwards_data(nodes, 4, Pkt(), bundle.adversary) for _ in range(100)
    )


def test_slanderer_inverts_its_reports():
    nodes, _ = build_nodes(n=10)
    nodes.role[1] = node_mod.FLAG_SLANDERER
    nodes.role[2] = node_mod.FLAG_GRAYHOLE
    # Vouches for a fellow attacker...
    assert adversary.reported_trust(nodes, 1, 2, 0.1) == config.SLANDER_HIGH
    # ...and accuses an honest node.
    assert adversary.reported_trust(nodes, 1, 3, 0.99) == config.SLANDER_LOW
    # An honest observer reports what it measured.
    assert adversary.reported_trust(nodes, 0, 3, 0.77) == 0.77


def test_energy_excuse_protects_a_flat_battery():
    """base.pdf 3.7 -- do not brand a node malicious for being out of power."""
    nodes, _ = build_nodes(n=12)
    victim = 5
    nodes.energy[victim] = 0.05 * config.INITIAL_ENERGY
    nodes.sent_to[:, victim] = 20.0
    nodes.fwd_seen[:, victim] = 0.0
    trust.refresh_trust(nodes)
    nodes.trust[:, victim] = 0.1

    trust.pt_gid_round(nodes, enable_energy_excuse=True, round_idx=30)
    assert not nodes.revoked[victim]

    trust.pt_gid_round(nodes, enable_energy_excuse=False, round_idx=60)
    assert nodes.revoked[victim]


def test_malicious_fraction_is_respected():
    nodes, bundle = build_nodes(n=100)
    for fraction in (0.1, 0.3, 0.5):
        ids = adversary.assign_roles(nodes, "grayhole", fraction, bundle.adversary)
        assert ids.size == int(round(fraction * 100))
        assert int(nodes.malicious_mask().sum()) == ids.size


# --------------------------------------------------------------------------
# Energy
# --------------------------------------------------------------------------

def test_linear_depletion_is_monotone():
    nodes, bundle = build_nodes(n=30)
    model = energy.LinearDepletion()
    model.reset(nodes, bundle.channel)
    previous = nodes.energy.copy()
    for round_idx in range(30):
        model.step(nodes, round_idx, bundle.channel)
        assert (nodes.energy <= previous + 1e-9).all()
        previous = nodes.energy.copy()


def test_solar_harvesting_can_recharge_and_respects_the_cap():
    nodes, bundle = build_nodes(n=30)
    model = energy.SolarHarvesting()
    model.reset(nodes, bundle.channel)
    nodes.energy[:] = 50.0
    recharged = False
    for round_idx in range(300):
        before = nodes.energy.copy()
        model.step(nodes, round_idx, bundle.channel)
        if (nodes.energy > before + 1e-9).any():
            recharged = True
        assert nodes.energy.max() <= config.MAX_ENERGY_CAP + 1e-9
    assert recharged, "a solar model that never recharges anything is not a solar model"


def test_nodes_die_exactly_once():
    nodes, bundle = build_nodes(n=20)
    model = energy.LinearDepletion()
    # Below the minimum drain of 0.04, so a single step is certain to kill.
    nodes.energy[:] = 0.03
    model.step(nodes, 5, bundle.channel)
    assert (~nodes.alive).all()
    assert (nodes.death_round == 5).all()
    model.step(nodes, 6, bundle.channel)
    assert (nodes.death_round == 5).all(), "death round must not be overwritten"


# --------------------------------------------------------------------------
# Determinism -- the property the whole comparison rests on
# --------------------------------------------------------------------------

def test_same_seed_gives_identical_worlds():
    first, _ = build_nodes(seed=99)
    second, _ = build_nodes(seed=99)
    assert np.array_equal(first.x, second.x)
    assert np.array_equal(first.adjacency, second.adjacency)


def test_policy_randomness_cannot_shift_the_world():
    """A policy that burns extra randomness must not change the environment."""

    class GreedyPolicy(EaurpPolicy):
        name = "greedy"

        def on_round_start(self, sim, round_idx):
            super(GreedyPolicy, self).on_round_start(sim, round_idx)
            sim.seeds.policy.random(25)  # consume policy randomness

    params = dict(n_nodes=60, speed=20000, rounds=40, seed=7)
    plain = Simulation(EaurpPolicy(), SimParams(**params))
    plain.run()
    greedy = Simulation(GreedyPolicy(), SimParams(**params))
    greedy.run()
    assert np.allclose(plain.nodes.x, greedy.nodes.x)
    assert np.array_equal(plain.nodes.role, greedy.nodes.role)


def test_identical_runs_reproduce_exactly():
    params = SimParams(n_nodes=60, speed=20000, rounds=60, seed=4242)
    first = Simulation(EaurpPolicy(), params).run()
    second = Simulation(EaurpPolicy(), params).run()
    for key in ("pdr", "avg_delay_ms", "packets_sent", "packet_loss_bytes"):
        assert first[key] == second[key], "run is not reproducible on {0}".format(key)


# --------------------------------------------------------------------------
# ECC
# --------------------------------------------------------------------------

def test_ecc_roundtrip():
    assert ecc.verify_roundtrip()


def test_ecc_actually_changes_the_payload():
    keyring = ecc.ECCKeyring(4, enabled=True)
    message = b"A" * 128
    blob = keyring.encrypt(0, 1, message)
    assert blob[8:] != message


# --------------------------------------------------------------------------
# End-to-end
# --------------------------------------------------------------------------

def test_simulation_produces_a_complete_result_row():
    params = SimParams(n_nodes=60, speed=20000, rounds=80, seed=1)
    row = Simulation(EaurpPolicy(), params).run()
    for key in ("pdr", "pdr_routable", "avg_delay_ms", "throughput",
                "network_lifetime", "det_tpr", "det_fpr", "policy"):
        assert key in row
    assert 0.0 <= row["pdr"] <= 1.0
    assert row["packets_sent"] > 0


def test_loss_reasons_account_for_every_lost_packet():
    params = SimParams(n_nodes=80, speed=25000, rounds=100, seed=2,
                       attack="blackhole", malicious_fraction=0.3)
    row = Simulation(EaurpPolicy(), params).run()
    accounted = sum(row["loss_" + reason] for reason in metrics.LOSS_REASONS)
    assert accounted == row["packets_lost"]


def test_blackhole_hurts_delivery():
    clean = Simulation(
        EaurpPolicy(), SimParams(n_nodes=100, rounds=150, seed=3)
    ).run()
    attacked = Simulation(
        EaurpPolicy(),
        SimParams(n_nodes=100, rounds=150, seed=3, attack="blackhole",
                  malicious_fraction=0.3),
    ).run()
    assert attacked["pdr"] < clean["pdr"]
    assert attacked["loss_malicious_drop"] > 0


In [ ]:
import sysif "" not in sys.path:    sys.path.insert(0, "")# Fail loudly here rather than three cells later.from src.common import configfrom src.common.simulator import SimParams, Simulationfrom src.routing.eaurp import AodvPolicy, EaurpPolicyfrom src.a_drl_eaurp.policy_common import TabularDrlPolicyfrom src.b_senior.policy_common import SeniorDrlPolicyfrom src.c_ar_eaurp.protocol import ArEaurpPolicyprint("imports OK")print("profile FULL: rounds={0}, runs={1}".format(    config.FULL.rounds, config.FULL.runs))

### 0.2 Self-check26 unit tests over the harness. The one that matters most is `test_grayhole_evades_the_trust_threshold`: it asserts that the gray-hole's *observed* Packet Forwarding Ratio stays above base.pdf's 0.6 revocation threshold. If that failed, the advancement would have nothing to detect.

In [ ]:
!python -m pytest tests/ -q 2>&1 | tail -5

In [ ]:
# The premise of the whole advancement, measured rather than asserted.import numpy as npfrom src.common import seeding, node as nm, config, adversaryfrom src.routing import trust as trust_modbundle = seeding.make_seed_bundle(11)nodes = nm.NodeState(20, bundle.topology)nodes.role[3] = nm.FLAG_GRAYHOLEclass _Pkt:    def __init__(self, size): self.size = size    @property    def is_large(self): return self.size >= config.LARGE_PACKET_THRESHOLDrng = np.random.default_rng(21)for _ in range(20000):    pkt = _Pkt(int(rng.integers(512, 1025)))    trust_mod.observe_forward(nodes, 0, 3, pkt,                              adversary.forwards_data(nodes, 3, pkt, bundle.adversary))pfr = nodes.observed_pfr()[0, 3]large = nodes.fwd_seen_large[0, 3] / nodes.sent_large[0, 3]small = nodes.fwd_seen_small[0, 3] / nodes.sent_small[0, 3]print("observed scalar PFR   = {:.4f}   (revocation threshold {})".format(    pfr, config.TRUST_THRESHOLD))print("  -> evades it by       {:+.4f}".format(pfr - config.TRUST_THRESHOLD))print("large-packet PFR      = {:.4f}".format(large))print("small-packet PFR      = {:.4f}".format(small))print("size-conditioned gap  = {:.4f}   <- the signal the GAN detector is given"      .format(small - large))

---## 1. Track 1 - each implementation in its native form### 1A. A - DRL-EAURP from `Lekha.pdf`Written from the published equations alone. All five of the paper's models:Existing (Eq. 14-18), EAURP (Eq. 4-13), ATEAURP (Eq. 19-24), PSE-EAURP(Eq. 25-28) and DRL-EAURP (Eq. 3, 29-38).Where the paper is under-specified, the assumption is recorded in`paper_model.py` and listed in `DEVIATIONS`.

In [ ]:
from src.a_drl_eaurp import run_track1 as a_track1from src.a_drl_eaurp import paper_model as pma_speed, a_policy_notes = a_track1.run_speed_sweep(rounds=2000, runs=3)a_density = a_track1.run_density_sweep(rounds=2000, runs=3, verbose=False)a_track1.write_csv(a_track1.to_rows(a_speed, a_density),                   os.path.join(RESULTS_DIR, "csv", "a_paper_track1.csv"))

#### The DRL agent's action space is degenerateEq. (36) gives action 0 a `+0.08` bonus and action 1 a `+0.02` bonus, unconditionally. Action 0 therefore dominates in every state, the optimal policy is the constant "always exploit", and the learned state `<T,E,M>` has no influence on anything. Measured below rather than asserted.

In [ ]:
for speed, note in sorted(a_policy_notes.items())[:3]:    print("v={:6d}  states visited={:3d}  action-0 share={:.3f}  "          "states preferring action 0: {}/{}".format(              speed, note["q_states_visited"], note["action0_share"],              note["states_preferring_action0"], note["states_total"]))

#### Where does the 'network lifetime improvement' come from?Eq. (8) specifies one energy drain band, `dE in [0.05, 0.15]`, and Sec. IV-D-6 says the DRL model depletes energy the same way. The delivered code uses `[0.04, 0.12]` for ATEAURP, PSE-EAURP and DRL-EAURP, and `[0.05, 0.15]` only for the base. Running the *same* model under both bands isolates the effect.

In [ ]:
ablation = a_track1.drain_band_ablation(rounds=1400, runs=3)for band in ("paper_Eq8_0.05-0.15", "code_0.04-0.12"):    print("{:22s} lifetime={:8.1f}   PDR={:.4f}".format(        band, ablation[band]["network_lifetime"], ablation[band]["pdr"]))print()print("=> {:+.1f} rounds of 'lifetime improvement' come from the drain "      "constant alone,".format(ablation["lifetime_delta"]))print("   with the delivery ratio essentially unchanged.")

In [ ]:
import textwrapprint("DEVIATIONS between Lekha.pdf and the delivered code")print("=" * 74)for item in pm.DEVIATIONS:    print()    print(item["topic"].upper())    for field in ("paper", "code", "impact"):        print("  {:7s}: {}".format(            field, textwrap.fill(item[field], 66, subsequent_indent=" " * 11)))

### 1B. B - the senior's code, verbatimLogic copied character-for-character; the only additions are a `main()` guard,a headless matplotlib backend and CSV output. `random.seed(42)` is set in thesame place the notebook sets it, so the published numbers reproduce exactly.This cell takes ~3 minutes: the original does an O(N^2) neighbour rebuild inpure Python on every one of 2000 rounds.

In [ ]:
from src.b_senior import senior_asisb_results = senior_asis.run_all(runs=5)senior_asis.write_csv(senior_asis.to_rows(b_results),                      os.path.join(RESULTS_DIR, "csv", "b_senior_as_is.csv"))print()print("Published in Lekha.pdf:      PDR up to 0.96, delay ~80 ms, "      "lifetime ~1230 rounds")print("Reproduced here:             PDR {:.4f},  delay {:.2f} ms,  "      "lifetime {:.0f}".format(          max(b_results["DRL-EAURP"]["pdr"]),          min(b_results["DRL-EAURP"]["delay"]),          sum(b_results["DRL-EAURP"]["lifetime"]) / 7.0))

#### Which of B's numbers are measured?Cell 13 of the original notebook prints `random.uniform` literals formatted as measured metrics. Those exact values appear in Lekha.pdf Sec. IV-A-6 as "observed behaviour". Re-running it produces different numbers every time, because nothing is being measured.

In [ ]:
print("Cell 13 of senior_code.ipynb, run three times:")for attempt in range(3):    block = senior_asis.hardcoded_metrics_block()    print("  PDR={PacketDeliveryRatio:6.2f}  delay={AverageDelaynsec:7.2f}  "          "throughput={Throughput_kbps:5.2f}  lifetime={NetworkLifetime_percentage:5.2f}%"          .format(**block))print()for key, value in senior_asis.PROVENANCE.items():    print("  {:20s} {}".format(key, value))

---## 2. Track 2 - one harness, identical worldsEvery policy below sees byte-identical topology, mobility, traffic, channel andadversary streams for a given seed (six independent generators spawned from one`SeedSequence`), so any difference in the numbers is attributable to the policyand nothing else.### 2.1 Pre-training CThe GAN is trained on a **clean commissioning run with no adversaries present**,then frozen. It never sees an attack before being evaluated on one. The LSTM istrained on generated harvest traces and scored against two naive baselines.

In [ ]:
from experiments.harness import pretrained_cfrom src.common import configPROFILE = config.FULL          # switch to config.QUICK for a fast smoke runshared_c, pretrain_summary = pretrained_c(PROFILE, verbose=True)for key in sorted(pretrain_summary):    print("  {:34s} {}".format(key, pretrain_summary[key]))

In [ ]:
from experiments import plotsplots.plot_gan_training(shared_c.detector, out_dir=RESULTS_DIR)plots.plot_lstm_accuracy(shared_c.predictor, out_dir=RESULTS_DIR)evaluation = shared_c.predictor.evaluationprint()print("LSTM vs baselines (MAE, lower is better)")print("  LSTM            {:.6f}".format(evaluation.get("lstm_mae", float("nan"))))print("  persistence     {:.6f}".format(evaluation.get("persistence_mae", float("nan"))))print("  seasonal naive  {:.6f}".format(evaluation.get("seasonal_naive_mae", float("nan"))))print()print("  beats persistence:    ", evaluation.get("beats_persistence"))print("  beats seasonal naive: ", evaluation.get("beats_seasonal_naive"))if not evaluation.get("beats_seasonal_naive"):    print()    print("  NOTE: the harvest signal is a diurnal sinusoid, so a")    print("  'same time yesterday' lookup is a strong baseline. Reported as")    print("  measured - this is exactly the risk the Review 1 deck flagged.")

### 2.2 E1-E2 - speed and densityThe paper's own sweeps, run on the mechanistic harness.

In [ ]:
from experiments import sweepssweeps.e1_speed(PROFILE, shared_c=shared_c, out_dir=RESULTS_DIR)sweeps.e2_density(PROFILE, shared_c=shared_c, out_dir=RESULTS_DIR)

### 2.3 E3 - the attack sweep**The headline experiment, and the one neither base paper can run.**Three attacks x six adversary fractions (0-50%) x every implementation:* **black-hole** - advertises a perfect route, drops 100% of the data* **gray-hole** - relays control and small packets, drops ~25% of packets over  700 bytes. Observed PFR ~0.84, comfortably above the 0.6 revocation threshold* **trust poisoning** - a gray-hole that also publishes false PT_GID reports:  trust 1.0 for its colluders, 0.15 for the most trusted honest nodes

In [ ]:
sweeps.e3_attack(PROFILE, shared_c=shared_c, out_dir=RESULTS_DIR)

### 2.4 E4-E6 - energy, ablation, overhead

In [ ]:
sweeps.e4_energy(PROFILE, shared_c=shared_c, out_dir=RESULTS_DIR)sweeps.e5_ablation(PROFILE, out_dir=RESULTS_DIR)sweeps.e6_overhead(PROFILE, shared_c=shared_c, out_dir=RESULTS_DIR)

---## 3. Figures

In [ ]:
from experiments import plotsimport globfrom IPython.display import Image, displayplots.plot_all(out_dir=RESULTS_DIR)for path in sorted(glob.glob(os.path.join(RESULTS_DIR, "figures", "*.png"))):    print(os.path.basename(path))    display(Image(filename=path))

---## 4. A vs B vs C - the comparison

In [ ]:
from experiments.report import build_comparison, print_reportreport = build_comparison(out_dir=RESULTS_DIR)print_report(report)

In [ ]:
# Bundle everything for download.import shutilarchive = shutil.make_archive("ar_eaurp_review2_results", "zip", RESULTS_DIR)print("wrote", archive)try:    from google.colab import files    files.download(archive)except Exception:    print("(not on Colab, or download unavailable - the zip is on disk)")